# <center>大模型 Agent 上下文工程进阶——LangChain组合编排实战</center>

&emsp;&emsp;今天我们聚焦的是 `Agent` 上下文工程的**组合编排**层面。在前置的《大模型 Agent 上下文工程基础入门》里，我们已经把 `Write`、`Select`、`Compress`、`Isolate`、`Cache` 这五种策略各自的"单点原理"讲清楚了，也建立了"系统提示／对话历史／记忆检索／工具上下文／任务状态／外部知识"这套**六模块认知地图**。但凡真正动手搭过一个生产级 `Agent` 的同学都会发现：单点策略全都懂，可一旦把它们叠在一起，问题立刻冒出来——压缩之后旧记忆不见了、记忆召回又把噪声塞回来、子 Agent 隔离做完了 Token 反而更高、最关键的是每一次"省 Token"的优化都把 `prompt cache` 命中率一路打到了零。这门进阶课要回答的，不是"五策略各是什么"，而是"五策略叠在一起到底该怎么排顺序、怎么避免互相破坏、怎么在真实成本数据下做 trade-off"。

&emsp;&emsp;结合 2026-04 这个时间窗口的最新调研结论，我们会把焦点放在四件事上：第一，基于 `create_agent` + `Middleware` 架构搭建一条可演化的主线 Agent；第二，把 `Mem0` 编排进 `Middleware` 链——用 `Mem0WriteMiddleware` 实现跨会话长期记忆，而不是像很多教程那样裸调用；第三，也是本课最具反共识的部分——我们会用三个动手实验证明，<font color=red>激进的对话压缩会摧毁 DeepSeek prefix cache，省下的 Token 反而推高账单</font>，并给出一套 `cache-friendly` 的中间件叠加顺序模板。

&emsp;&emsp;为了把这些内容讲清楚，接下来我们会按一条单主线 Agent 的演化路径展开：第 0 章先用最简形态的 `Toy Agent` 建立基线测量工具；第 1 章到第 4 章分别在 `Compression`、`Memory`、`Select`、`Isolate` 四个方向做组合编排；第 5 章是全课高潮，正面处理 `Cache vs Compress` 的冲突；第 6 章做综合评估验证全部成果，第 7 章把它们反哺回基础课的 `mini-OpenClaw` 项目，完成"理论 → 演示 → 真实工程"的合流。让我们从第 0 章的起点出发。

> 📅 **时效性说明**：本课件锚定 2026-04-16，所有调研结论、`GitHub issue` 引用、`API` 用法均基于该时间窗口的 `langchain==1.2.15` / `langchain-core==1.2.28` / `langgraph==1.1.6` / `langgraph-checkpoint==4.0.1` / `langgraph-prebuilt==1.0.9` / `deepagents==0.5.2` / `mem0ai==1.0.11` / `pymilvus==2.6.12` / `langsmith==0.7.30`。`ragas` 保留 `0.2.10`，因 `0.4.x` 评估 API 重构与本课件教学占位代码不兼容。课程发布前请复查 `issue #34517` / `#5755` / `#31949` 三个开放问题的最新状态。

&emsp;&emsp;**学员画像与前置要求**：本课面向**已经完成《大模型 Agent 上下文工程基础入门》课程、具备 `Python` 与 `LangChain` 基础的 AI 开发者**。我们默认你已经熟悉 `create_agent` 调用方式、`LangGraph` 的 `StateGraph` 编排以及 `Write / Select / Compress / Isolate / Cache` 五大上下文策略的**单点用法**。本课的目标不是再讲一遍这些单点 API，而是带你从"会用单点策略"升级到"能设计、能诊断、能优化**任意 Agent 上下文架构**"——也就是回答"当五种策略叠在一起互相冲突时，到底该怎么排顺序、怎么避免破坏、怎么在真实成本数据下做 trade-off"。<font color=red>本课不适合零基础学员；如果你对 `create_agent`、`StateGraph`、`Mem0` 这几个名字还感到陌生，建议先完成基础课再回到这里。</font>

&emsp;&emsp;**版本快照声明（2026-04-13）**：本课件所有依赖锁定为 `2026-04-13` PyPI 快照版本，`ragas` 特别保留 `0.2.10`（`0.4.x` 评估 API 已重构，与教学占位代码不兼容）。<font color=red>学员复现时请严格使用 `requirements.txt` 锁定版本，避免因生态升级导致 API 不匹配。</font>

---

## <center>第 0 章：起点——六模块认知地图</center>

&emsp;&emsp;本章是整个进阶课的"地基章"。我们不引入任何新的中间件，也不讨论任何复杂的优化策略，只做一件事：搭一个最简单的 `Agent`，并为它建立一张会在每一章更新的**六模块主战场地图**。这张地图一旦在 第 0 章 确立，后面六章就只需要聚焦各自的编排策略，不再重复建设基础设施。本章不改造任何模块，只完成 baseline 测量；正是这次基线测量会告诉我们："哪几条轴失衡得最严重，下一章该先动哪里。"

&emsp;&emsp;本章的另一层意义是建立**进阶课与基础课的边界**。基础课已经把"六模块各是什么"、"五策略各是什么"讲透了，本章不会再重复任何定义性内容，只会用一句话复述每个模块的核心抓手，剩下时间全部用于工具实现与基线跑测。这条边界是后续所有章节的"反重复红线"。

### 0.1 本课主战场地图：六模块与五策略的关系

&emsp;&emsp;在动手之前，我们先用一张表把基础课已经建立的"六模块认知地图"做一次**一行复述**——这是本课贯穿七章的坐标系。<font color=red>注意：六模块回答的是"被管理的对象是什么"（WHAT），五策略回答的是"用什么手段管理"（HOW），二者不是同一个层面的概念，本课会严格分开使用。</font>

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>六模块清单与本课主战场分布</font></p>
<div class="center">

| 编号 | 模块名 | 一行复述 | 主要被哪几章改造 |
|------|--------|----------|------------------|
| M1 | 系统提示层 System Prompt | 稳定前缀，是 cache 命中的锚点 | 第 5 章 |
| M2 | 对话历史层 Conversation History | 增长最快的失衡源，trim/summary/tail-trim 互相竞争 | 第 1 章 / 第 5 章 |
| M3 | 记忆检索层 Memory Retrieval | 跨会话事实存取，Mem0 作为节点而非裸调用 | 第 2 章 / 第 3 章 |
| M4 | 工具上下文层 Tool Context | tool_calls 与 ToolMessage 的密度治理 | 第 1 章 / 第 4 章 |
| M5 | 任务状态层 Task State / Scratchpad | 中间决策链的可重放性 | 第 2 章 / 第 4 章 |
| M6 | 外部知识层 External Knowledge | 检索四路融合的目的地 | 第 3 章 |

</div>

&emsp;&emsp;有了这张坐标系，我们就可以用一张架构图来俯视整门课。下面这张图标注了每一章的主战场以及相邻章节之间的承接关系，建议大家把它打印出来贴在手边——后续每一章的部分都会回头指向它。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260417140023412.svg" width=80%></div>

&emsp;&emsp;接下来我们就从这张图最左端的 Toy Agent 开始动手，先把测量工具搭好。

### 0.2 环境准备与依赖加载

&emsp;&emsp;在动手写第一行 Agent 代码之前，我们需要把所有依赖安装到位、把 API 密钥从 `.env` 文件加载进环境变量。这一步会被本课所有章节复用，所以我们用一个独立的 cell 完成，并在后续章节直接假定它已经执行过。

&emsp;&emsp;本课依赖清单已经放在与 `lesson.md` 同目录的 `requirements.txt` 中，每个包后面都有行内注释说明用途。下面这个 cell 会一次性安装所有依赖。

In [ ]:
# 一次性安装本课所有依赖（-q 静默模式）
!pip install -r requirements.txt -q

&emsp;&emsp;依赖装好之后，我们用 `python-dotenv` 加载环境变量。<font color=red>请注意：本课所有代码严禁硬编码 API key</font>，请在项目根目录创建 `.env` 文件，写入以下三项：`DEEPSEEK_API_KEY`（主 LLM 调用）、`OPENAI_API_KEY`（**第 2 章 起 `Mem0` 使用 `text-embedding-3-small` 做向量化，必须配置该项；可指向任意 OpenAI 兼容代理**）、`LANGSMITH_API_KEY`（可选，第 3 章 trace 使用）。如果还没有 DeepSeek 账户，可以先到 `https://platform.deepseek.com` 注册，本课的成本量级在每次完整跑通约 ¥0.8 以内（基于 `DeepSeek` 官方 2026-04 定价估算）。

```text
DEEPSEEK_API_KEY=sk-your-deepseek-key
OPENAI_API_KEY=sk-your-openai-key   # Mem0 embedder 使用，可指向 OpenAI 兼容代理
LANGSMITH_API_KEY=lsv2-your-key     # 可选
```

In [1]:
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
import os

# 将同目录 .env 文件注入到 os.environ
load_dotenv(override=True)

# 读取 DeepSeek 主 key（必需）
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")  # 可选，第 3 章 用

# 断言：缺 key 时立即失败，避免后续静默报错
assert DEEPSEEK_API_KEY, "请先在 .env 中配置 DEEPSEEK_API_KEY"

# 全局模型常量：本课所有 LLM 调用统一从这里取，禁止在下游 cell 中硬编码模型名
MODEL = "deepseek-chat"

print(f"DeepSeek key 已加载，使用模型 {MODEL}")

PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


DeepSeek key 已加载，使用模型 deepseek-chat


In [2]:
from transformers import AutoTokenizer

# 加载 DeepSeek 官方 tokenizer（首次运行需下载 ~几MB，后续自动缓存）
# trust_remote_code=True 允许执行模型仓库中的自定义代码（DeepSeek tokenizer 需要）
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-V3", trust_remote_code=True)

def count_tokens(text: str) -> int:
    """
    用 DeepSeek 官方 tokenizer 精确计算 token 数
    
    Args:
        text: 待计算的文本字符串
    
    Returns:
        int: token 数量（与 API 实际消耗一致）
    """
    return len(tokenizer.encode(text))

def count_tokens_approximately(messages: list) -> int:
    """
    用 DeepSeek 官方 tokenizer 精确计算消息列表总 token 数
    函数名保留是为了兼容 MessageTrimMiddleware / CompactionMiddleware 的 token_counter 注入接口，
    实际已升级为精确计数。
    """
    total = 0
    for m in messages:
        content = m.content if hasattr(m, "content") else str(m)
        # 兼容多模态 content：LangChain 的 content 可能是 list of blocks
        if isinstance(content, list):
            content = "".join(
                block.get("text", "") if isinstance(block, dict) else str(block)
                for block in content
            )
        if not isinstance(content, str):
            content = str(content)
        total += count_tokens(content)
    return total

`rope_parameters`'s factor field must be a float >= 1, got 40
`rope_parameters`'s beta_fast field must be a float, got 32
`rope_parameters`'s beta_slow field must be a float, got 1


&emsp;&emsp;这段代码完成了三件事：用 `load_dotenv()` 把同目录 `.env` 注入到 `os.environ`、断言 `DEEPSEEK_API_KEY` 存在以避免后续静默失败、定义全局常量 `MODEL`。后续所有章节遇到模型名时一律引用 `MODEL`，不再硬编码字符串——这是为了保证未来切换到 `deepseek-reasoner` 或其他后端时只需要改这一处。

### 0.3 构造 简单 Agent

&emsp;&emsp;本课的"主线 Agent"会从这一节出发，逐章演化。第 0 章 的形态尽可能简单：一个create_agent 调用，挂一个只做"打印state["messages"]"的自定义中间件，不接任何真实工具、不加任何压缩策略，模型也换成FakeListChatModel 保证可复现。我们要的就是一个"最容易观察"的初始状态——让 before_model /after_model 这对钩子把 agent 每一 tick 的消息列表原样摊在眼前，只有看清了这张"裸状态地图"，后续每一章往上叠的每一件压缩武器，才能在同一张地图上看到明显的模块点亮。

In [3]:
"""最简单的自定义中间件 demo：打印 state['messages']。"""                                  
                                                                                            
from langchain.agents import create_agent                                     
from langchain.agents.middleware import AgentMiddleware                                    
from langchain_deepseek import ChatDeepSeek
from langchain_core.tools import tool

# 模型层：温度设 0 保证输出稳定可复现
llm = ChatDeepSeek(model=MODEL, temperature=0)                                                                                                     
                                                                                            
class PrintMessagesMiddleware(AgentMiddleware):                                          
    """在模型调用前打印当前 state['messages'] 的快照。"""                                  
                                                                                            
    def before_model(self, state, runtime):                                                
        msgs = state["messages"]                                                           
        print(f"\n[before_model] state['messages'] 共 {len(msgs)} 条")                   
        for i, m in enumerate(msgs):                                                       
            content = str(m.content).replace("\n", " ")[:70]
            print(f"  [{i}] {type(m).__name__:15s} | {content}")                           
        return None  # 返回 None 表示不修改 state                                          

    def after_model(self, state, runtime):                                                 
        msgs = state["messages"]                                                         
        last = msgs[-1]                                                                    
        print(f"\n[after_model ] 新增一条 {type(last).__name__}: "
            f"{str(last.content)[:70]}")                                                 
        return None                                                                      
                                                                                                                           
agent = create_agent(
    model=llm,                                                                           
    tools=[],                                                                            
    system_prompt="你是一个数学助手",
    middleware=[PrintMessagesMiddleware()],
)

print("========== 调用 agent ==========")                                                  
result = agent.invoke({"messages": [{"role": "user", "content": "3 + 5 等于几？"}]})
                                                                                            
print("\n========== 最终 result['messages'] ==========")                                   
for m in result["messages"]:
    print(f"{type(m).__name__}: {m.content}")

========== 调用 agent ==========

[before_model] state['messages'] 共 1 条
  [0] HumanMessage    | 3 + 5 等于几？

[after_model ] 新增一条 AIMessage: 3 + 5 = 8。

========== 最终 result['messages'] ==========
HumanMessage: 3 + 5 等于几？
AIMessage: 3 + 5 = 8。


&emsp;&emsp;这段代码是本课主线 Agent 的"零号形态"：模型层用 `ChatDeepSeek` 直连 DeepSeek API、工具层只有两个 stub（一个负责"检索"、一个负责"写文件"，都返回固定字符串）、`create_agent` 不挂任何中间件。两个 stub 工具的存在不是为了真的搜文件，而是为后续 `M4 工具上下文层` 与 `ModelCallLimitMiddleware` 提供最小载体——让 ToolMessage 真的出现在状态里，否则 M4 工具上下文层永远没有被触达的痕迹，主战场地图上也看不出变化。

&emsp;&emsp;到这里 第 0 章 的全部任务就完成了：我们搭了一个最简形态的主线 Agent、认识了六模块主战场地图和五种策略的对应关系。从 第 1 章 开始，我们将逐章点亮每一个模块——先从 M2 对话历史层和 M4 工具上下文层的 Compression 组合编排开始。

---

## <center>第 1 章：Compression 组合编排</center>

&emsp;&emsp;在进入具体策略之前，先花一分钟对齐**编排载体**——本课从 第 1 章 起，所有组合编排都通过 `AgentMiddleware` 这一个基类完成。它提供三个核心钩子：`before_model`（调模型前改请求）、`after_model`（拿到响应后改状态）、`wrap_model_call`（v1.0 wrap-style，包住整次调用）。五种策略都是这三个钩子的不同组合——Compress 主用 `before_model`，Select 主用 `wrap_model_call`，Write 主用 `after_model`。


<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260417140814932.png" width=40%></div>



&emsp;&emsp;本章承接 第 0 章 的主战场地图：M2 对话历史层与 M4 工具上下文层两个模块处于全暗基线。基础课已经把 `trim_messages`、`SummarizationMiddleware`、工具结果清除、观察遮蔽、Compaction 这五种单点 compress 策略各自的原理讲过一遍。本章不会再重复任何单点原理，而是聚焦一个完全不同的问题——**多个 Compress 中间件叠在一起时，叠加顺序就是策略本身**。同样的两个中间件，外内顺序颠倒一下，结果可能从"两轴一起降"变成"两轴互相破坏"。

&emsp;&emsp;为了把这个问题讲清楚，我们会先用 Factory.ai 的"六维压缩探针"建立一套客观的检测框架，然后把 `SummarizationMiddleware` 与 `ModelCallLimitMiddleware` 这两个最具代表性的中间件做正确叠加与错误叠加的对比演示，最后用 第 0 章 冻结的 **P1 长历史压缩探针**的两条真跑数据（baseline / compress），看 M2 / M4 两轴在主战场地图上的真实变化。本章主战场是 M2 对话历史层 + M4 工具上下文层。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260417140021241.svg" width=60%></div>
<!--
[图 1-1 生成提示词 - 可直接用于 AI 图片生成]
Style: 科技感信息图，玻璃拟态，环形仪表盘布局（与图 0-1 同底图）
Background: 深蓝星空渐变 #0a1630 → #1a2850
Content: 六个圆形模块环形排列。M2 与 M4 两个模块被点亮为暖橙色 #ff8c42，周围有柔和辉光发散效果。M1/M3/M5/M6 维持灰色暗态。M2 与 M4 之间出现一条暖橙色能量流动线，象征 Compress 中间件的叠加顺序路径
Chinese labels: M1 系统提示 / M2 对话历史 ✨ / M3 记忆检索 / M4 工具上下文 ✨ / M5 任务状态 / M6 外部知识
Color scheme: 主色 #0a1630；点亮色 #ff8c42 暖橙；暗态 #4a5568
Output: 16:9 横版，用于第 1 章章首
-->
<center><font size=2>图 1-1：第 1 章主战场地图 · M2/M4 双模块点亮</font></center>

### 1.1 五件套 Middleware 叠加策略

&emsp;&emsp;基础课里我们讲过 `SummarizationMiddleware` 的单点原理，但实战中单靠它远远不够。对话历史的 token 消耗来源多样：工具结果往往是最大的单条消耗源，而 `AIMessage` 的推理链（`tool_calls` 字段）才是我们最不舍得丢的信息。五件套 Middleware 的设计逻辑正是基于这个分层认知——**先用规则操作把低价值内容清掉，再让 LLM 摘要处理语义层面，最后用硬截断和全局压缩兜底**。

&emsp;&emsp;五件套的推荐叠加顺序见下表。顺序不是随意的：低成本规则操作在前，LLM 调用在后；硬截断在摘要之后（保底），全局压缩是核选项排在最末。

<style>
.center { width: auto; display: table; margin-left: auto; margin-right: auto; }
</style>
<p align="center"><font face="黑体" size=4>五件套 Middleware 叠加顺序建议</font></p>
<div class="center">

| 顺序 | Middleware | 操作类型 | LLM调用 | 压缩强度 | 语义损失 |
|------|-----------|---------|--------|--------|--------|
| 1 | `ToolResultClearMiddleware` | LLM 工具摘要清除 | 按需 | 中 | 低 |
| 2 | `ObservationMaskMiddleware` | 观察遮蔽 | 无 | 高 | 中 |
| 3 | `SummarizationMiddleware` | LLM 整体摘要 | 1次 | 中 | 低 |
| 4 | `MessageTrimMiddleware` | 硬截断 | 无 | 可调 | 中-高 |
| 5 | `CompactionMiddleware` | 全局重启 | 1次 | 极高 | 中 |

</div>

&emsp;&emsp;接下来我们**按叠加顺序逐个过一遍五个 Middleware**：每个都先讲定义，再给实现代码，最后配一个最小测试验证核心不变式。为了让五个中间件共用一套测试 fixture，下一个 cell 先定义 `make_rounds()` 对话工厂和 `FakeLLM` 假 LLM——它们会被本节所有测试复用，不依赖真实 API，本地秒级可跑。

In [58]:
# ============================================================
# 五件套 Middleware 共用测试 fixture
# ============================================================
# - make_rounds(n): 造 n 轮 [Human, AI+tool_call, Tool, AI] 对话工厂
# - FakeLLM:        离线假 LLM，invoke 返回固定字符串，避免真实 API 调用
# 所有 5 个 Middleware 的独立测试都会复用这两个对象
# ============================================================
from langchain_core.messages import (
    HumanMessage, AIMessage, ToolMessage, SystemMessage,
)

def make_rounds(n: int, tool_content_len: int = 1200) -> list:
    """造 n 轮 [Human, AI(tool_call), Tool, AI] 对话。

    Args:
        n:                轮次数量
        tool_content_len: 每条 ToolMessage 字符长度，默认 1200（~300 tokens）
    Returns:
        list[BaseMessage]: 首条 SystemMessage + n 轮对话
    """
    msgs = [SystemMessage(content="你是数据分析助手")]
    for i in range(1, n + 1):
        msgs.append(HumanMessage(content=f"第{i}步：分析数据"))
        msgs.append(AIMessage(
            content="",
            tool_calls=[{"id": f"c{i}", "name": "analyze", "args": {}}],
        ))
        msgs.append(ToolMessage(
            content=f"第{i}步结果 " + "x" * tool_content_len,
            tool_call_id=f"c{i}",
            name="analyze",
        ))
        msgs.append(AIMessage(content=f"第{i}步完成"))
    return msgs


class FakeLLM:
    """离线假 LLM：invoke 永远返回固定字符串，便于测试断言。"""

    def __init__(self, fixed_reply: str = "FAKE_SUMMARY_42"):
        self.fixed_reply = fixed_reply
        self.call_count = 0

    def invoke(self, prompt):
        self.call_count += 1
        return AIMessage(content=self.fixed_reply)


fake_llm = FakeLLM()
print(f"测试 fixture 就绪：make_rounds() + FakeLLM(call_count={fake_llm.call_count})")

def _banner(title: str) -> None:
    """每个测试 case 开头打印清晰的分隔标题。"""
    print("\n" + "═" * 60)
    print(f"  {title}")
    print("═" * 60)


测试 fixture 就绪：make_rounds() + FakeLLM(call_count=0)


#### 1.1.1 Middleware 1：工具结果清除（`ToolResultClearMiddleware`）

&emsp;&emsp;工具调用会返回大量结构化数据（JSON、代码执行结果、搜索摘要），这些 `ToolMessage` 在后续轮次通常只有参考价值，却占据大量 token。`ToolResultClearMiddleware` 扫描历史，将 N 步之前的 `ToolMessage` **用 LLM 压缩为一行结论性摘要**，保留最近 K 条完整工具结果。语义对齐基础入门 2.5 节定义：**清除 = LLM 摘要替换（保留核心结论）**，与后面的观察遮蔽（占位符完全丢弃）是两种不同策略。

> &emsp;⚠️ **关键设计决策**：不能用 `RemoveMessage` 直接删除 `ToolMessage`——OpenAI / DeepSeek API 要求每个 `ToolMessage` 的 `tool_call_id` 必须与对应 `AIMessage` 的 `tool_calls` 字段匹配，否则报 400 错误。因此只能用摘要内容替换，保留 `tool_call_id`。

In [62]:
from typing import Any
from langchain_core.messages import ToolMessage
from langchain.messages import RemoveMessage
from langchain.agents import AgentState
from langchain.agents.middleware import AgentMiddleware
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.runtime import Runtime


# 摘要 Prompt 模板
TOOL_SUMMARY_PROMPT = (
    "用一句话（不超过80字）总结以下工具输出的关键发现，保留数字/文件名/错误信息等决策相关细节：\n\n{tool_output}"
)

# 已摘要标记：幂等保护，避免对摘要再次摘要导致信息衰减
SUMMARY_PREFIX = "[摘要] "

class ToolResultClearMiddleware(AgentMiddleware):
    """工具结果清除 Middleware（对齐基础入门 2.5 节定义）：
    超出保留窗口的 ToolMessage 用 LLM 压缩为一行摘要，保留 tool_call_id。

    Args:
        llm:                      用于生成摘要的 Chat Model 实例
        keep_recent_tool_results: 最近 N 条 ToolMessage 保留原文，默认 3
        summary_prompt:           摘要 Prompt 模板，需含 {tool_output} 占位符
    """

    def __init__(
        self,
        llm,
        keep_recent_tool_results: int = 3,  # 窗口外的条目才会被 LLM 压缩
        summary_prompt: str = TOOL_SUMMARY_PROMPT,
    ) -> None:
        super().__init__()
        self.llm = llm
        self.keep_recent = keep_recent_tool_results
        self.summary_prompt = summary_prompt
        # 以 tool_call_id 为 key 缓存摘要，跨轮复用避免重复调 LLM
        self._summary_cache: dict[str, str] = {}

    def _summarize(self, msg: ToolMessage) -> str:
        """LLM 压缩单条 ToolMessage，带缓存与异常降级。"""
        # tool_call_id 唯一标识一次工具调用，用它做缓存 key 保证内容幂等
        cache_key = msg.tool_call_id
        if cache_key in self._summary_cache:
            return self._summary_cache[cache_key]

        try:
            resp = self.llm.invoke(
                self.summary_prompt.format(tool_output=str(msg.content))
            )
            summary = resp.content.strip() if hasattr(resp, "content") else str(resp).strip()
        except Exception as e:
            # LLM 调用失败时降级为占位文本，保证 pipeline 不中断
            tool_name = getattr(msg, "name", "unknown")
            summary = f"{tool_name} 原始输出已清除（摘要失败：{type(e).__name__}）"

        self._summary_cache[cache_key] = summary
        return summary

    def after_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        messages = state["messages"]
        # 收集所有 ToolMessage 的位置索引，后续按窗口切分
        tool_indices = [i for i, m in enumerate(messages) if isinstance(m, ToolMessage)]

        # 未超窗口直接放行，避免无谓 LLM 开销
        if len(tool_indices) <= self.keep_recent:
            return None

        # 窗口外的索引 = 全部 - 最近 keep_recent 条
        to_clear_set = set(tool_indices[:-self.keep_recent])
        changed = False
        new_messages = []
        for i, msg in enumerate(messages):
            if i in to_clear_set and isinstance(msg, ToolMessage):
                content_str = str(msg.content)
                # 幂等保护：已含 SUMMARY_PREFIX 说明本轮已压缩，跳过防止二次衰减
                if content_str.startswith(SUMMARY_PREFIX):
                    new_messages.append(msg)
                    continue
                summary = self._summarize(msg)
                new_messages.append(ToolMessage(
                    content=f"{SUMMARY_PREFIX}{summary}",
                    tool_call_id=msg.tool_call_id,
                    name=getattr(msg, "name", None),
                ))
                changed = True
            else:
                new_messages.append(msg)

        if not changed:
            return None

        # RemoveMessage(REMOVE_ALL_MESSAGES) 是 LangGraph 清空消息列表的标准信号
        # 再追加 new_messages 等同于原子替换，避免脏写入
        return {
            "messages": [
                RemoveMessage(id=REMOVE_ALL_MESSAGES),
                *new_messages,
            ]
        }


tool_clear_mw = ToolResultClearMiddleware(llm=fake_llm, keep_recent_tool_results=2)
print(f"ToolResultClearMiddleware 实例化：keep_recent={tool_clear_mw.keep_recent}")

ToolResultClearMiddleware 实例化：keep_recent=2


In [64]:
# ============================================================
# Test: ToolResultClearMiddleware
# 三个独立 case：no-op / 主路径 / 幂等保护
# 场景：Agent 帮用户理解一个 Flask 项目，依次 read_file 读取代码
# ============================================================

# ------------------------------------------------------------
# Case 1: 未超窗口 → 中间件不动手（no-op 分支）
# ------------------------------------------------------------
_banner("Case 1 | no-op 分支：未超 keep_recent 窗口应直接放行")

fake1 = FakeLLM()
mw1 = ToolResultClearMiddleware(llm=fake1, keep_recent_tool_results=3)
state1 = {"messages": [
    ToolMessage(
        content=(
            "from flask import Flask, jsonify\n"
            "app = Flask(__name__)\n\n"
            "@app.route('/health')\n"
            "def health_check():\n"
            "    return jsonify(status='ok')\n"
            "# ... 共 12 行"
        ),
        tool_call_id="t1", name="read_file",
    ),
    ToolMessage(
        content=(
            "from .database import db\n"
            "from .models import User, Post\n"
            "# 应用工厂函数，注册蓝图与扩展\n"
            "# ... 共 23 行"
        ),
        tool_call_id="t2", name="read_file",
    ),
]}
print("【输入】 2 条 read_file 返回（app.py + __init__.py），keep_recent=3（未超窗口）")

result1 = mw1.after_model(state1, runtime=None)
print(f"【输出】 result = {result1}")
print(f"【核对】 LLM 调用次数 = {fake1.call_count}（期望 0）")

assert result1 is None, "未超窗口应返回 None"
assert fake1.call_count == 0, "未超窗口时不应调用 LLM"
print("[PASS] 中间件识别未超窗口，返回 None 未调用 LLM")


# ------------------------------------------------------------
# Case 2: 超窗口 → 真实代码文件被摘要替换，最新文件原文保留
# 混合策略：本 case 用真实 DeepSeek `llm`，让学员看到真实的内容感知摘要
#           （Case 1/3 仍用 FakeLLM，保证 call_count == 0 的硬断言）
# ------------------------------------------------------------
_banner("Case 2 | 主路径（真实 LLM）：3 个真实代码文件，前 2 个被真实摘要，test_api.py 原文保留")

mw2 = ToolResultClearMiddleware(llm=llm, keep_recent_tool_results=1)
state2 = {"messages": [
    # 最老：SQLAlchemy 模型定义
    ToolMessage(
        content=(
            "from sqlalchemy import Column, Integer, String, DateTime, ForeignKey\n"
            "from datetime import datetime\n\n"
            "class User(db.Model):\n"
            "    id = Column(Integer, primary_key=True)\n"
            "    email = Column(String(120), unique=True, nullable=False)\n"
            "    created_at = Column(DateTime, default=datetime.utcnow)\n\n"
            "class Post(db.Model):\n"
            "    id = Column(Integer, primary_key=True)\n"
            "    user_id = Column(Integer, ForeignKey('user.id'))\n"
            "    title = Column(String(200))\n"
            "    body = Column(Text)\n"
            "    # ... 共 47 行"
        ),
        tool_call_id="t1", name="read_file",
    ),
    # 中间：API 视图函数
    ToolMessage(
        content=(
            "@app.route('/api/posts', methods=['GET'])\n"
            "def list_posts():\n"
            "    page = request.args.get('page', 1, type=int)\n"
            "    posts = Post.query.filter_by(published=True) \\\n"
            "              .order_by(Post.created_at.desc()) \\\n"
            "              .paginate(page=page, per_page=20)\n"
            "    return jsonify([p.to_dict() for p in posts.items])\n\n"
            "@app.route('/api/posts/<int:post_id>', methods=['GET'])\n"
            "def get_post(post_id):\n"
            "    post = Post.query.get_or_404(post_id)\n"
            "    return jsonify(post.to_dict())\n"
            "    # ... 共 89 行"
        ),
        tool_call_id="t2", name="read_file",
    ),
    # 最新：测试用例（Agent 下一步决策基于这个）
    ToolMessage(
        content=(
            "import pytest\n"
            "from app import create_app\n\n"
            "@pytest.fixture\n"
            "def client():\n"
            "    app = create_app('testing')\n"
            "    with app.test_client() as c:\n"
            "        yield c\n\n"
            "def test_list_posts_returns_200(client):\n"
            "    resp = client.get('/api/posts')\n"
            "    assert resp.status_code == 200\n\n"
            "def test_get_post_404_on_missing(client):\n"
            "    resp = client.get('/api/posts/9999')\n"
            "    assert resp.status_code == 404"
        ),
        tool_call_id="t3", name="read_file",
    ),
]}
print("【输入】 3 次 read_file 调用（models.py / views.py / test_api.py），keep_recent=1")
print("         ─── 压缩前（每条 ToolMessage 的首行）───")
for i, m in enumerate(state2["messages"], 1):
    first_line = str(m.content).split("\n")[0]
    print(f"         第{i}条 [{m.name}] {len(m.content)} 字符 → {first_line[:55]}")

result2 = mw2.after_model(state2, runtime=None)
new_tools = [m for m in result2["messages"] if isinstance(m, ToolMessage)]

print("\n【输出】 压缩后的 3 条 ToolMessage（前 2 条是真实 DeepSeek 生成的摘要）：")
for i, m in enumerate(new_tools, 1):
    marker = "压缩" if m.content.startswith("[摘要] ") else "原文"
    preview = str(m.content)[:200].replace("\n", " | ")
    print(f"         第{i}条 [{marker}] id={m.tool_call_id} → {preview}")

# 结构断言：不依赖 LLM 输出内容
assert new_tools[0].content.startswith("[摘要] "), "第1条（models.py）在窗口外，应被压缩"
assert new_tools[1].content.startswith("[摘要] "), "第2条（views.py）在窗口外，应被压缩"
assert not new_tools[2].content.startswith("[摘要] "), "第3条（test_api.py）在窗口内，应保留原文"
assert "test_list_posts_returns_200" in new_tools[2].content, "测试函数名必须保留"
assert new_tools[0].tool_call_id == "t1", "tool_call_id 必须在压缩后保留"

# 真实 LLM 差异化断言：对不同文件应生成不同摘要（证明中间件真的读取了内容传给了 LLM）
assert new_tools[0].content != new_tools[1].content, \
    "真实 LLM 对不同文件应生成不同摘要（models.py vs views.py），否则说明 LLM 没读内容"

print("[PASS] 前 2 文件被真实摘要 / 摘要内容不同 / test_api.py 原文完整 / tool_call_id 保留")


# ------------------------------------------------------------
# Case 3: 已含 [摘要] 前缀 → 幂等跳过（LLM 零调用）
# ------------------------------------------------------------
_banner("Case 3 | 幂等保护：已压缩消息跳过，LLM 不被二次调用")

fake3 = FakeLLM()
mw3 = ToolResultClearMiddleware(llm=fake3, keep_recent_tool_results=1)
state3 = {"messages": [
    ToolMessage(
        content="[摘要] read_file 返回 models.py 文件内容（User/Post SQLAlchemy 模型）",
        tool_call_id="t1", name="read_file",
    ),
    ToolMessage(
        content="def new_endpoint():\n    return jsonify(status='new')",
        tool_call_id="t2", name="read_file",
    ),
]}
print("【输入】 2 条 ToolMessage，第1条已是 [摘要] 前缀，keep_recent=1")

result3 = mw3.after_model(state3, runtime=None)
print(f"【输出】 result = {result3}")
print(f"【核对】 LLM 调用次数 = {fake3.call_count}（期望 0，证明幂等生效）")

assert result3 is None, "窗口外的那条已是摘要，changed=False，应返回 None"
assert fake3.call_count == 0, "幂等生效时不应调用 LLM"
print("[PASS] 已压缩消息被跳过 / LLM 零调用 / 未重复压缩")


# ============================================================
print("\n" + "═" * 60)
print("  ToolResultClearMiddleware 全部 3 个 case 通过 [PASS]")
print("═" * 60)



════════════════════════════════════════════════════════════
  Case 1 | no-op 分支：未超 keep_recent 窗口应直接放行
════════════════════════════════════════════════════════════
【输入】 2 条 read_file 返回（app.py + __init__.py），keep_recent=3（未超窗口）
【输出】 result = None
【核对】 LLM 调用次数 = 0（期望 0）
[PASS] 中间件识别未超窗口，返回 None 未调用 LLM

════════════════════════════════════════════════════════════
  Case 2 | 主路径（真实 LLM）：3 个真实代码文件，前 2 个被真实摘要，test_api.py 原文保留
════════════════════════════════════════════════════════════
【输入】 3 次 read_file 调用（models.py / views.py / test_api.py），keep_recent=1
         ─── 压缩前（每条 ToolMessage 的首行）───
         第1条 [read_file] 476 字符 → from sqlalchemy import Column, Integer, String, DateTim
         第2条 [read_file] 486 字符 → @app.route('/api/posts', methods=['GET'])
         第3条 [read_file] 384 字符 → import pytest

【输出】 压缩后的 3 条 ToolMessage（前 2 条是真实 DeepSeek 生成的摘要）：
         第1条 [压缩] id=t1 → [摘要] 代码定义了两个SQLAlchemy模型类：User（含id、email、created_at字段）和Post（含id、user_id、title、body字段），共47行。
         第2条 

&emsp;&emsp;`ToolResultClearMiddleware` 的核心价值是：**每条 ToolMessage 只压缩一次**（tool_call_id 缓存），摘要后的内容用 `[摘要]` 前缀标记实现幂等。测试覆盖三个分支 case：**no-op**（未超窗口时零 LLM 调用）、**主路径**（窗口外压缩 + 窗口内保留 + tool_call_id 守恒 + LLM 调用数 = 被压缩数）、**幂等保护**（已压缩消息跳过 + LLM 零调用）。

&emsp;&emsp;**与 SummarizationMiddleware 的叠加关系**：工具清除应在 Summarization 之前运行——先用轻量的 per-tool 摘要把旧工具结果压小（每条只压一次，后续缓存），再对剩余历史做整段摘要，能显著减少 Summarization 的输入 token 和成本。


#### 1.1.2 Middleware 2：观察遮蔽（`ObservationMaskMiddleware`）

&emsp;&emsp;基于 JetBrains 研究（Agent 上下文压缩论文）：在推理密集型任务中，保留 `AIMessage` 的 reasoning + `tool_calls` 字段（思考过程），将 `ToolMessage` 全部遮蔽为占位符，可在语义损失最小的前提下大幅压缩上下文。**遮蔽 = 占位符完全丢弃**，与前一个的"清除 = LLM 摘要替换"是两种不同策略。

> &emsp;⚠️ **适用边界（重要）**：
> 适合：ReAct 代码执行 Agent，ToolMessage 是中间计算结果（用后即弃）。
> 不适合：客服 Agent，ToolMessage 是订单查询结果（用户下一句会问刚查的订单号）——遮蔽后 Agent 无法回答。
> 验证方法：跑 `variable_tracking_test`——如果 PASS 率下降，说明不适合当前场景。

In [7]:
from langchain_core.messages import AIMessage

# 遮蔽前缀：幂等保护，避免重复遮蔽时 content_len 被替换为占位符长度
MASK_PREFIX = "[观察已遮蔽:"


class ObservationMaskMiddleware(AgentMiddleware):
    """观察遮蔽 Middleware：保留 AIMessage 推理链，遮蔽 ToolMessage 为占位符。
    适用于推理密集型任务（代码生成、多步规划），不适用于对话型任务。

    Args:
        mask_template:            遮蔽后占位符模板，支持 {tool_name} {content_len} 插值
        keep_last_n_observations: 保留最近 N 条观察不遮蔽，0 = 全部遮蔽
    """

    def __init__(
        self,
        mask_template: str = "[观察已遮蔽: {tool_name} 返回 {content_len} 字符]",
        keep_last_n_observations: int = 1,  # 0 表示全部遮蔽，>0 表示保留尾部 N 条
    ) -> None:
        super().__init__()
        self.mask_template = mask_template
        self.keep_last_n = keep_last_n_observations

    def before_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        messages = state["messages"]
        tool_indices = [i for i, m in enumerate(messages) if isinstance(m, ToolMessage)]

        # 没有 ToolMessage 则无需处理
        if not tool_indices:
            return None

        # keep_last_n > 0：保留尾部 N 条，其余遮蔽；= 0：全部遮蔽
        if self.keep_last_n > 0:
            to_mask = set(tool_indices[:-self.keep_last_n])
        else:
            to_mask = set(tool_indices)

        changed = False
        new_messages = []
        for i, msg in enumerate(messages):
            if i in to_mask and isinstance(msg, ToolMessage):
                content_str = str(msg.content)
                # 幂等保护：已遮蔽的跳过，防止 content_len 被替换为占位符长度
                if content_str.startswith(MASK_PREFIX):
                    new_messages.append(msg)
                    continue
                masked = self.mask_template.format(
                    tool_name=getattr(msg, "name", "tool"),
                    content_len=len(content_str),
                )
                new_messages.append(ToolMessage(
                    content=masked,
                    tool_call_id=msg.tool_call_id,
                    name=getattr(msg, "name", None),
                ))
                # changed 标志：至少有一条消息被修改才真正写回 state
                changed = True
            else:
                new_messages.append(msg)

        if not changed:
            return None

        return {
            "messages": [
                RemoveMessage(id=REMOVE_ALL_MESSAGES),
                *new_messages,
            ]
        }


obs_mask_mw = ObservationMaskMiddleware(keep_last_n_observations=1)
print(f"ObservationMaskMiddleware 实例化完成，keep_last_n={obs_mask_mw.keep_last_n}")

ObservationMaskMiddleware 实例化完成，keep_last_n=1


In [8]:
from langchain_core.messages import AIMessage, HumanMessage

# ============================================================
# Test: ObservationMaskMiddleware
# 三个独立 case：no-op / 主路径 / 幂等
# 场景：Agent 诊断生产环境慢 SQL（3 次 run_sql 交替 AIMessage 推理）
# 核心教学点：AIMessage 推理链完整保留，ToolMessage 细节被占位符替换
# ============================================================

# ------------------------------------------------------------
# Case 1: 无 ToolMessage → 中间件不动手（no-op 分支）
# ------------------------------------------------------------
_banner("Case 1 | no-op 分支：无 ToolMessage 时应直接放行")

mw1 = ObservationMaskMiddleware(keep_last_n_observations=1)
state1 = {"messages": [
    HumanMessage(content="帮我分析一下这个 SQL 为什么慢"),
    AIMessage(content="好的，请把 SQL 和表结构贴出来"),
]}
print("【输入】 2 条消息（HumanMessage + AIMessage），0 条 ToolMessage")

result1 = mw1.before_model(state1, runtime=None)
print(f"【输出】 result = {result1}")

assert result1 is None, "无 ToolMessage 应返回 None"
print("[PASS] 中间件识别无观察消息，返回 None 未做任何处理")


# ------------------------------------------------------------
# Case 2: 主路径 — 真实慢 SQL 诊断，AIMessage 推理链必须完整保留
# ------------------------------------------------------------
_banner("Case 2 | 主路径：3 次 run_sql 交替推理，旧观察被遮蔽，AIMessage 推理链原样")

mw2 = ObservationMaskMiddleware(keep_last_n_observations=1)
state2 = {"messages": [
    HumanMessage(content="生产环境 /api/orders 接口昨天开始变慢到 3 秒，能帮我定位吗？"),
    AIMessage(
        content="先查一下 orders 表的规模和索引情况，看看是不是数据量问题",
        tool_calls=[{"name": "run_sql", "args": {"sql": "SELECT COUNT(*)"}, "id": "q1"}],
    ),
    ToolMessage(
        content=(
            "SELECT COUNT(*) FROM orders;\n"
            "结果: 5,847,321 rows\n"
            "耗时: 3.21s\n"
            "说明: 全表扫描 580 万行"
        ),
        tool_call_id="q1", name="run_sql",
    ),
    AIMessage(
        content="580 万行属于中等规模，理论上不该慢到 3 秒。看看现有索引分布",
        tool_calls=[{"name": "run_sql", "args": {"sql": "SHOW INDEX"}, "id": "q2"}],
    ),
    ToolMessage(
        content=(
            "SHOW INDEX FROM orders;\n"
            "结果:\n"
            "  PRIMARY     (id)\n"
            "  idx_user    (user_id)\n"
            "  idx_created (created_at)\n"
            "说明: 共 3 个索引，缺 status 字段索引"
        ),
        tool_call_id="q2", name="run_sql",
    ),
    AIMessage(
        content="果然缺 status 索引。跑 EXPLAIN 看看当前查询的执行计划，确认是全表扫描",
        tool_calls=[{"name": "run_sql", "args": {"sql": "EXPLAIN ..."}, "id": "q3"}],
    ),
    ToolMessage(
        content=(
            "EXPLAIN SELECT * FROM orders WHERE status='pending' ORDER BY created_at;\n"
            "type: ALL\n"
            "rows_examined: 5847321\n"
            "Extra: Using where; Using filesort\n"
            "说明: 全表扫描 + 文件排序，确认索引缺失"
        ),
        tool_call_id="q3", name="run_sql",
    ),
]}

print("【输入】 7 条消息（1 Human + 3 AI推理 + 3 ToolMessage），keep_last_n_observations=1")
print("         ─── 压缩前结构 ───")
for i, m in enumerate(state2["messages"], 1):
    mtype = type(m).__name__
    if isinstance(m, ToolMessage):
        first_line = m.content.split("\n")[0]
        print(f"         第{i}条 [{mtype:13s}] {len(m.content):3d} 字符 → {first_line[:50]}")
    else:
        print(f"         第{i}条 [{mtype:13s}] → {m.content[:55]}")

result2 = mw2.before_model(state2, runtime=None)
new_msgs = result2["messages"][1:]

print("\n【输出】 压缩后结构（AIMessage 原样 / 旧 Tool 被遮蔽 / 最新 EXPLAIN 原文）：")
for i, m in enumerate(new_msgs, 1):
    mtype = type(m).__name__
    if isinstance(m, ToolMessage):
        mark = "遮蔽" if m.content.startswith("[观察已遮蔽:") else "原文"
        preview = m.content.split("\n")[0][:55]
        print(f"         第{i}条 [{mtype:13s}] [{mark}] → {preview}")
    else:
        print(f"         第{i}条 [{mtype:13s}] → {m.content[:55]}")

# 关键不变式
tools_after = [m for m in new_msgs if isinstance(m, ToolMessage)]
ais_after = [m for m in new_msgs if isinstance(m, AIMessage)]

assert tools_after[0].content.startswith("[观察已遮蔽:"), "第1条 ToolMessage 应被遮蔽"
assert tools_after[1].content.startswith("[观察已遮蔽:"), "第2条 ToolMessage 应被遮蔽"
assert "EXPLAIN" in tools_after[2].content, "最新 EXPLAIN 输出必须原文保留（供下一步决策）"
assert "5847321" in tools_after[2].content, "关键数字 5847321 必须保留"
assert len(ais_after) == 3, "3 条 AIMessage 推理链必须全部保留"
assert "580 万行" in ais_after[1].content, "AIMessage 推理内容不得改动"
assert ais_after[0].tool_calls[0]["id"] == "q1", "tool_calls 不得丢失"
print("[PASS] 前 2 Tool 被遮蔽 / 最新 EXPLAIN 原文 / 3 AIMessage 推理链完整 / tool_calls 保留")


# ------------------------------------------------------------
# Case 3: 已遮蔽 → 幂等跳过（content_len 不被污染）
# ------------------------------------------------------------
_banner("Case 3 | 幂等保护：已遮蔽消息字面严格不变")

mw3 = ObservationMaskMiddleware(keep_last_n_observations=0)
ALREADY_MASKED = "[观察已遮蔽: run_sql 返回 243 字符]"
state3 = {"messages": [
    ToolMessage(content=ALREADY_MASKED, tool_call_id="q1", name="run_sql"),
    ToolMessage(content="新查询原文输出，共20字", tool_call_id="q2", name="run_sql"),
]}
print(f"【输入】 第1条（已遮蔽）: {ALREADY_MASKED}")
print("          第2条（新原文）: 新查询原文输出，共20字")

result3 = mw3.before_model(state3, runtime=None)
tools3 = [m for m in result3["messages"][1:] if isinstance(m, ToolMessage)]
print(f"【输出】 第1条 → {tools3[0].content}")
print(f"          第2条 → {tools3[1].content}")

assert tools3[0].content == ALREADY_MASKED, "已遮蔽消息必须字面严格不变"
print("[PASS] 已遮蔽消息一字未改 / 新消息正常遮蔽")


# ============================================================
print("\n" + "═" * 60)
print("  ObservationMaskMiddleware 全部 3 个 case 通过 [PASS]")
print("═" * 60)



════════════════════════════════════════════════════════════
  Case 1 | no-op 分支：无 ToolMessage 时应直接放行
════════════════════════════════════════════════════════════
【输入】 2 条消息（HumanMessage + AIMessage），0 条 ToolMessage
【输出】 result = None
[PASS] 中间件识别无观察消息，返回 None 未做任何处理

════════════════════════════════════════════════════════════
  Case 2 | 主路径：3 次 run_sql 交替推理，旧观察被遮蔽，AIMessage 推理链原样
════════════════════════════════════════════════════════════
【输入】 7 条消息（1 Human + 3 AI推理 + 3 ToolMessage），keep_last_n_observations=1
         ─── 压缩前结构 ───
         第1条 [HumanMessage ] → 生产环境 /api/orders 接口昨天开始变慢到 3 秒，能帮我定位吗？
         第2条 [AIMessage    ] → 先查一下 orders 表的规模和索引情况，看看是不是数据量问题
         第3条 [ToolMessage  ]  73 字符 → SELECT COUNT(*) FROM orders;
         第4条 [AIMessage    ] → 580 万行属于中等规模，理论上不该慢到 3 秒。看看现有索引分布
         第5条 [ToolMessage  ] 123 字符 → SHOW INDEX FROM orders;
         第6条 [AIMessage    ] → 果然缺 status 索引。跑 EXPLAIN 看看当前查询的执行计划，确认是全表扫描
         第7条 [ToolMessage  ] 163 字符 → EXPLAIN SELECT * 

&emsp;&emsp;`ObservationMaskMiddleware` 与 `ToolResultClearMiddleware` 的区别在于：Clear 是"LLM 摘要替换"（保留核心结论），Mask 是"占位符完全丢弃"（靠 AIMessage 的 reasoning 链携带信息）。两者不互斥，可以先 Clear 再 Mask，分层处理不同历史深度的工具结果。

&emsp;&emsp;**与 SummarizationMiddleware 的叠加关系**：遮蔽应在摘要之前运行——先把 ToolMessage 压为占位符，再对剩余历史做整段摘要，摘要 LLM 的输入 token 更少，摘要质量也更高（没有大量 JSON 结构干扰摘要）。

#### 1.1.3 Middleware 3：对话总结（`SummarizationMiddleware`）

&emsp;&emsp;基础课里我们已经讲过 `SummarizationMiddleware` 的工作原理：它会监听对话累计 token，超过阈值时把 `keep` 窗口之外的早期消息整批替换成一条 **`HumanMessage`** 形式的总结（带 `Here is a summary of the conversation to date:` 前缀）。这里只做一次三分钟的接口速览，把两个最关键的参数过一下。

In [9]:
from langchain.agents.middleware import SummarizationMiddleware

# 构造对话总结中间件：超阈值时把早期消息替换成 HumanMessage 形式的总结
summary_mw = SummarizationMiddleware(
    model=llm,                                # 用于生成总结的 LLM 实例
    trigger=("tokens", 2500),                 # 教学触发阈值：2500 tokens（高于 DeepSeek cache 1024 门槛）
    keep=("messages", 10),                    # 保留最近 10 条 message
    summary_prompt=None,                      # 用默认 prompt（注意 issue #34517）
)
print("SummarizationMiddleware 已实例化")

SummarizationMiddleware 已实例化


In [10]:
# ============================================================
# Test: SummarizationMiddleware（新增测试单元）
# 两个独立 case：no-op / 主路径（对比 Compaction）
# 场景：复用电商数据分析真实对话，展示与 CompactionMiddleware 的核心差异
#   - Summarization: 摘要包装为 HumanMessage（带 "Here is a summary" 前缀），
#                    最近 keep 条原样保留，早期 SystemMessage 不自动保留
#   - Compaction:    摘要包装为 SystemMessage（带 "[历史对话摘要]" 前缀），
#                    原 SystemMessage 显式保留在首位
# ============================================================
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.language_models.fake_chat_models import FakeListChatModel
try:
    from langchain.messages import RemoveMessage
except ImportError:
    from langchain_core.messages.modifier import RemoveMessage

# ------------------------------------------------------------
# Case 1: 未超 trigger → 放行（no-op 分支）
# ------------------------------------------------------------
_banner("Case 1 | no-op 分支：未超 trigger 不触发总结")

fake_sum1 = FakeListChatModel(responses=["不该被调用的摘要"])
sum_mw1 = SummarizationMiddleware(
    model=fake_sum1,
    trigger=("tokens", 5000),
    keep=("messages", 5),
)
state_s1 = {"messages": [
    SystemMessage(content="你是数据分析助手"),
    HumanMessage(content="简单问下 pandas 怎么按日聚合"),
    AIMessage(content="用 df.groupby(pd.Grouper(key='date', freq='D')).sum() 即可"),
]}
total_s1 = count_tokens_approximately(state_s1["messages"])
print(f"【输入】 3 条消息，累计约 {total_s1} tokens，trigger=5000（远未超）")

# 注意：langchain 1.2+ 的 before_model(state, runtime) runtime 是位置参数
# 但 runtime=None 在 no-op 路径下不会被访问，所以 OK
result_s1 = sum_mw1.before_model(state_s1, runtime=None)
print(f"【输出】 result = {result_s1}")

assert result_s1 is None, "未触发时应返回 None"
print("[PASS] 未触发总结 / result is None")


# ------------------------------------------------------------
# Case 2: 主路径 — 与 Compaction 的核心差异
# ------------------------------------------------------------
_banner("Case 2 | 主路径：前 N-keep 条 → 1 条总结，最近 keep 条原文保留")

fake_sum2 = FakeListChatModel(responses=[
    "用户分析 50 万行电商订单数据，发现最近 7 天销售额增长 40% 但订单量仅增 15%，"
    "客单价从 265 涨至 328 元。拆解后确认涨幅完全来自数码 3C 单品类，"
    "下一步从上新记录、地域分布、SKU 排序三方向做根因分析。"
])
# trigger 设极低保证一定触发（LangChain 内部用 chars_per_token≈3.3 计数，
# 与我们 fixture 的 //4 略有差异，直接设 30 避免阈值猜测）
sum_mw2 = SummarizationMiddleware(
    model=fake_sum2,
    trigger=("tokens", 30),
    keep=("messages", 2),
)

# 复用与 Compaction Case 2 完全相同的 7 条真实电商对话（方便学员对比两者差异）
state_s2 = {"messages": [
    SystemMessage(content="你是数据分析助手，擅长电商数据诊断。"),
    HumanMessage(content=(
        "我手头有一份电商平台订单数据，CSV 格式 50 万行，时间跨度最近 90 天。"
        "字段包括 order_id、user_id、category、amount、order_time、city、"
        "payment_method 共 7 列。老板让我分析最近 30 天的销售趋势，重点找"
        "增长点和下滑品类。你建议我从哪几个维度切入？"
    )),
    AIMessage(content=(
        "50 万行属于中等规模，Pandas 单机完全够用。建议从三个正交维度切入："
        "时间维度（按日聚合销售额和订单量）、品类维度（各 category 占比和环比）、"
        "地域维度（Top 10 城市销售贡献）。先跑 describe() 摸清数据分布。"
    )),
    HumanMessage(content=(
        "时间维度跑完了，最近 7 天销售额比前 23 天的日均值高了 40%，"
        "但订单量只增长 15%。客单价从 265 涨到 328，涨幅 24%。说明什么？"
    )),
    AIMessage(content=(
        "客单价涨幅远大于订单量涨幅，是典型的消费升级或品类结构变化信号。"
        "下一步最该做的是品类维度的客单价拆解——单一品类拉高是促销驱动。"
    )),
    HumanMessage(content=(
        "按你说的拆完了，发现数码 3C 贡献 80% 客单价涨幅，其他品类持平。"
        "下一步应该怎么做？"
    )),
    AIMessage(content=(
        "锁定 3C 后建议三个动作：查 7 天内 3C 上新/调价记录、看 3C 高客单价"
        "订单的 user_id 地域分布、对 3C 做 SKU 级涨幅排序找出 Top 3-5 爆款。"
    )),
]}

total_s2 = count_tokens_approximately(state_s2["messages"])
print(f"【输入】 7 条真实数据分析对话，累计约 {total_s2} tokens")
print(f"         trigger=('tokens', 30), keep=('messages', 2)")
print("         ─── 压缩前的 7 条消息 ───")
for i, m in enumerate(state_s2["messages"], 1):
    preview = str(m.content)[:50] + ("..." if len(str(m.content)) > 50 else "")
    print(f"         第{i}条 [{type(m).__name__:13s}] {preview}")

result_s2 = sum_mw2.before_model(state_s2, runtime=None)
assert result_s2 is not None, "Summarization 应触发（trigger=30 足够低）"

new_msgs_s2 = [m for m in result_s2["messages"] if not isinstance(m, RemoveMessage)]

print(f"\n【输出】 处理后共 {len(new_msgs_s2)} 条消息：")
for i, m in enumerate(new_msgs_s2, 1):
    preview = str(m.content)[:75]
    if len(str(m.content)) > 75:
        preview += "..."
    print(f"         第{i}条 [{type(m).__name__:13s}] {preview}")

print(f"\n【核对】 消息数 {len(state_s2['messages'])} → {len(new_msgs_s2)}")
print(f"         原始对话 7 条 → 总结 HumanMessage 1 条 + 末 2 条原文 = 3 条（原 SystemMessage 不自动保留）")

# 不变式 1：消息数减少
assert len(new_msgs_s2) < len(state_s2["messages"]), "Summarization 后消息数应减少"

# 不变式 2：存在摘要消息 —— SummarizationMiddleware 把摘要包装为 HumanMessage，
#          带 "Here is a summary of the conversation to date:" 前缀
#          （这与 Compaction 的 SystemMessage + "[历史对话摘要]" 前缀不同）
summary_found = any(
    ("Here is a summary" in str(m.content)) or ("数码 3C" in str(m.content))
    for m in new_msgs_s2
)
assert summary_found, "应包含摘要消息（含 'Here is a summary' 前缀或 FakeLLM 返回内容）"

# 不变式 3：最近 keep=2 条原文保留（Summarization 的核心特性）
last_two_original = [state_s2["messages"][-2].content, state_s2["messages"][-1].content]
result_contents = [m.content for m in new_msgs_s2]
preserved_count = sum(1 for c in last_two_original if c in result_contents)
assert preserved_count >= 1, f"末 keep=2 条应至少保留 1 条原文，实际保留 {preserved_count} 条"

print("[PASS] 摘要消息含核心信息 / 末 2 条对话原文保留 / 消息数减少")
print("\n--- 关键观察：Summarization vs Compaction 的三大差异 ---")
print("1. 摘要消息类型：Summarization 用 HumanMessage，Compaction 用 SystemMessage")
print("2. 摘要前缀：    Summarization 'Here is a summary of the conversation...'")
print("                 Compaction    '[历史对话摘要]'")
print("3. 原 System：   Summarization 不自动保留（被视为可摘要内容）")
print("                 Compaction    显式保留在首位")


# ============================================================
print("\n" + "═" * 60)
print("  SummarizationMiddleware 全部 case 通过 [PASS]")
print("═" * 60)



════════════════════════════════════════════════════════════
  Case 1 | no-op 分支：未超 trigger 不触发总结
════════════════════════════════════════════════════════════
【输入】 3 条消息，累计约 33 tokens，trigger=5000（远未超）
【输出】 result = None
[PASS] 未触发总结 / result is None

════════════════════════════════════════════════════════════
  Case 2 | 主路径：前 N-keep 条 → 1 条总结，最近 keep 条原文保留
════════════════════════════════════════════════════════════
【输入】 7 条真实数据分析对话，累计约 335 tokens
         trigger=('tokens', 30), keep=('messages', 2)
         ─── 压缩前的 7 条消息 ───
         第1条 [SystemMessage] 你是数据分析助手，擅长电商数据诊断。
         第2条 [HumanMessage ] 我手头有一份电商平台订单数据，CSV 格式 50 万行，时间跨度最近 90 天。字段包括 order...
         第3条 [AIMessage    ] 50 万行属于中等规模，Pandas 单机完全够用。建议从三个正交维度切入：时间维度（按日聚合销售额...
         第4条 [HumanMessage ] 时间维度跑完了，最近 7 天销售额比前 23 天的日均值高了 40%，但订单量只增长 15%。客单价...
         第5条 [AIMessage    ] 客单价涨幅远大于订单量涨幅，是典型的消费升级或品类结构变化信号。下一步最该做的是品类维度的客单价拆解...
         第6条 [HumanMessage ] 按你说的拆完了，发现数码 3C 贡献 80% 客单价涨幅，其他品类持平。下一步应该怎么做？
        

&emsp;&emsp;这里把阈值定为 2500 不是随手挑的——它是本课**全局数轴上的关键锚点**。把四个关键数字摆到一起看：

<style>
.center { width: auto; display: table; margin-left: auto; margin-right: auto; }
</style>
<p align="center"><font face="黑体" size=4>课件全局 token 数轴</font></p>
<div class="center">

| 位置 | tokens | 含义 |
|------|--------|------|
| 起点 | 0 | —— |
| DeepSeek cache 存储单元 | 64 | 每次缓存以 64 tokens 为单位，前缀不足 64 tokens 不会被缓存 |
| 第 0 章 P1 baseline 实测 | 2001 | 11 轮对话的真跑 input tokens |
| **第 1 章 compress trigger 阈值** | **2500** | 紧贴 baseline + 高于 cache 门槛 |
| 生产推荐值 | 4000+ | 与第 6 章 mini-OpenClaw 回填对齐 |

</div>

&emsp;&emsp;把 compress trigger 卡在 2500，等于在"cache 已经能命中"和"compress 还未触发"之间留出约 **1000 tokens 的安全窗口**——<font color=red>这个窗口正是第 5 章要正面验证的 `cache-friendly` 区间</font>。如果阈值设成 500，压缩发生时连 cache 最低门槛都没到，第 5 章"compress 破坏 cache"的反共识结论在数据上根本无法成立。所以 2500 不是拍脑袋，是前后七章数据链的硬约束交点。

&emsp;&emsp;<font color=red>这里要做一个重要的"踩坑预警"：截至 2026-04，`langchain-ai/langchain` 仓库的 issue #34517 仍处于 open 状态</font>，它指出 `SummarizationMiddleware` 默认的 summary prompt 会把 `tool_calls` 和 `response_metadata` 这类内部结构序列化进总结正文，导致总结本身每次都略有不同，等于"二次 cache 污染"。如果你需要把这个中间件用到生产环境，请先 fork 一份干净的 `summary_prompt` 模板，把内部 metadata 显式过滤掉。

> ⚠️ **常见误区**：很多人以为 `SummarizationMiddleware` 触发后只是“加一条总结消息”，实际它是把早期消息**整批替换**成一条 **`HumanMessage`**（带 `Here is a summary of the conversation to date:` 前缀）。这个动作直接破坏了对话历史的字节稳定性，会让后续的 prefix cache 命中率从修改点起全部清零——这正是 第 5 章 整章要解决的反共识问题，我们这里先按下不表。

> 📝 **补充澄清**：上面说的“清除所有早期消息包括 SystemMessage”，指的是 `state["messages"]` 数组里的 SystemMessage。如果你用 LangChain v1 推荐的 `create_agent(prompt=“你是...”)` 方式传系统提示，这个 prompt 参数**不在 `state["messages"]` 里**，每轮由 agent 在 model 调用前动态注入——所以**不会被 `SummarizationMiddleware` 吞掉**。只有老写法把 `SystemMessage` 塞进 messages 数组时，才会被当成普通消息一起摘要掉。

#### 1.1.4 Middleware 4：硬截断（`MessageTrimMiddleware`）

&emsp;&emsp;最直接的上下文控制手段：超过 token 上限就从头部截掉消息，只保留最近 N 条。优点是零延迟、无 LLM 调用；缺点是遗忘是硬性的，早期对话上下文彻底丢失。适合用作**保底兜底层**，叠在其他 Middleware 之后，防止极端情况下窗口溢出。

In [49]:
from langchain_core.messages import trim_messages, BaseMessage

    
class MessageTrimMiddleware(AgentMiddleware):
    """硬截断 Middleware：超出 max_tokens 时从头部移除旧消息。

    Args:
        max_tokens:    触发截断的 token 上限，默认 4000
        keep_last:     至少保留最近 N 条消息
        token_counter: 可注入自定义计数函数；None 时用内置粗估
    """

    def __init__(
        self,
        max_tokens: int = 4000,
        keep_last: int = 10,     # 防止 trim 过激导致上下文过少
        token_counter=None,
    ) -> None:
        super().__init__()
        self.max_tokens = max_tokens
        self.keep_last = keep_last
        self.token_counter = token_counter or count_tokens_approximately

    def before_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        messages = state["messages"]

        # 未超限提前返回，省去 trim_messages 的遍历开销
        if self.token_counter(messages) <= self.max_tokens:
            return None

        trimmed = trim_messages(
            messages,
            strategy="last",
            max_tokens=self.max_tokens,      # token 优先、条数兜底
            token_counter=self.token_counter,
            include_system=True,
            allow_partial=False,
        )

        # keep_last 保底：trim 可能移除过多导致上下文断层，兜底保留最近 keep_last 条
        if len(trimmed) < self.keep_last and len(messages) >= self.keep_last:
            trimmed = messages[-self.keep_last:]

        # trim 后条数未减少说明 token_counter 与 trim 结果不一致，无需写回
        if len(trimmed) == len(messages):
            return None

        return {
            "messages": [
                RemoveMessage(id=REMOVE_ALL_MESSAGES),
                *trimmed,
            ]
        }


trim_mw = MessageTrimMiddleware(max_tokens=4000, keep_last=10)
print(f"MessageTrimMiddleware 实例化完成，max_tokens={trim_mw.max_tokens}, keep_last={trim_mw.keep_last}")

MessageTrimMiddleware 实例化完成，max_tokens=4000, keep_last=10


In [12]:
# ============================================================
# Test: MessageTrimMiddleware
# 三个独立 case：no-op / 主路径（真实对话）/ keep_last 兜底
# 场景：复用电商数据分析对话，演示硬截断"字节级零改动"的核心特性
#       与 Summarization/Compaction 的本质差异：trim 砍掉不改写
# ============================================================

# ------------------------------------------------------------
# Case 1: 未超 token 上限 → 中间件不动手（no-op 分支）
# ------------------------------------------------------------
_banner("Case 1 | no-op 分支：未超 max_tokens 应直接放行")

mw1 = MessageTrimMiddleware(max_tokens=5000, keep_last=2)
state1 = {"messages": [
    SystemMessage(content="你是数据分析助手"),
    HumanMessage(content="简单问下 pandas 怎么按日聚合数据"),
    AIMessage(content="用 df.groupby(pd.Grouper(key='date', freq='D')).sum() 即可"),
]}
total1 = count_tokens_approximately(state1["messages"])
print(f"【输入】 3 条消息，累计约 {total1} tokens，max_tokens=5000（远未超）")

result1 = mw1.before_model(state1, runtime=None)
print(f"【输出】 result = {result1}")

assert result1 is None, "未超限应返回 None"
print("[PASS] 未超限直接放行，原文未动")


# ------------------------------------------------------------
# Case 2: 超 token 上限 → 从头硬截断，保留的每条消息字节零改动
# ------------------------------------------------------------
_banner("Case 2 | 主路径：真实对话硬截断，保留的每条字节级原样")

mw2 = MessageTrimMiddleware(max_tokens=50, keep_last=2)
state2 = {"messages": [
    SystemMessage(content="你是数据分析助手，擅长电商数据诊断。"),
    HumanMessage(content=(
        "我有一份 50 万行电商订单数据，想分析最近 30 天的销售趋势。"
        "字段包括 order_id、user_id、category、amount、order_time、city。"
        "建议从哪些维度切入？"
    )),
    AIMessage(content=(
        "建议从时间维度（按日聚合）、品类维度（销售占比）、地域维度（Top 10 城市）"
        "三个方向切入。先跑 describe() 摸清数据分布。"
    )),
    HumanMessage(content=(
        "时间维度跑完发现最近 7 天销售额增长 40% 但订单量只增 15%，"
        "客单价从 265 涨到 328。这说明什么？"
    )),
    AIMessage(content=(
        "客单价涨幅超过订单量，是消费升级或品类结构变化信号。"
        "下一步拆品类客单价定位原因。"
    )),
    HumanMessage(content=(
        "拆完了，数码 3C 贡献 80% 客单价涨幅。下一步怎么做？"
    )),
    AIMessage(content=(
        "锁定 3C 品类后建议三个动作：查上新/调价记录、看 user_id 地域分布、"
        "做 SKU 级涨幅排序。"
    )),
]}
total2_before = count_tokens_approximately(state2["messages"])
print(f"【输入】 7 条真实数据分析对话，累计约 {total2_before} tokens，max_tokens=50, keep_last=2")
print("         ─── 压缩前（全部原文） ───")
for i, m in enumerate(state2["messages"], 1):
    preview = str(m.content)[:48] + ("..." if len(str(m.content)) > 48 else "")
    print(f"         第{i}条 [{type(m).__name__:13s}] {preview}")

result2 = mw2.before_model(state2, runtime=None)
new_msgs2 = result2["messages"][1:]
total2_after = count_tokens_approximately(new_msgs2)

print(f"\n【输出】 硬截断后 {len(new_msgs2)} 条消息，约 {total2_after} tokens")
print("         ─── 压缩后（每条都是原文字节）───")
for i, m in enumerate(new_msgs2, 1):
    preview = str(m.content)[:50] + ("..." if len(str(m.content)) > 50 else "")
    print(f"         第{i}条 [{type(m).__name__:13s}] {preview}")

# 关键不变式：trim 是"砍掉"不是"改写"，保留下来的每条都必须字节级原样
assert total2_after <= mw2.max_tokens, f"截断后 tokens 应 <= {mw2.max_tokens}"
assert isinstance(new_msgs2[0], SystemMessage), "SystemMessage 保留在首位"
# 每条保留的消息必须在原列表中字节级一致
for m in new_msgs2:
    assert m in state2["messages"], f"保留消息应与原始字节一致: {m.content[:40]}"
# 最新消息必须保留
assert state2["messages"][-1] in new_msgs2, "最新消息必须保留"
print("[PASS] tokens 在上限内 / SystemMessage 头位 / 保留消息字节零改动 / 最新消息保留")
print("       核心差异：Trim 砍掉而非改写 → 与 Summarize/Compaction 的总结 SystemMessage 本质不同")


# ------------------------------------------------------------
# Case 3: 极端截断 → keep_last 兜底触发
# ------------------------------------------------------------
_banner("Case 3 | 兜底保护：max_tokens 极小时 keep_last 强制保留最近 N 条")

mw3 = MessageTrimMiddleware(max_tokens=10, keep_last=3)
state3 = {"messages": [
    SystemMessage(content="你是助手"),
    HumanMessage(content="第一个问题：如何读取 CSV " * 10),
    AIMessage(content="第一个回答：用 pd.read_csv " * 10),
    HumanMessage(content="第二个问题：如何筛选行 " * 10),
    AIMessage(content="第二个回答：用布尔索引 " * 10),
    HumanMessage(content="第三个问题（最新）：如何分组 " * 10),
]}
print(f"【输入】 6 条消息，max_tokens=10（刻意设极小）, keep_last=3")

result3 = mw3.before_model(state3, runtime=None)
new_msgs3 = result3["messages"][1:]
print(f"【输出】 兜底后保留 {len(new_msgs3)} 条：")
for i, m in enumerate(new_msgs3):
    preview = str(m.content)[:30] + "..."
    print(f"         第{i+1}条 [{type(m).__name__}] {preview}")

assert len(new_msgs3) >= mw3.keep_last, f"兜底应至少保留 {mw3.keep_last} 条"
print(f"[PASS] keep_last 兜底生效，保留 {len(new_msgs3)} 条（>= {mw3.keep_last}）")


# ============================================================
print("\n" + "═" * 60)
print("  MessageTrimMiddleware 全部 3 个 case 通过 [PASS]")
print("═" * 60)



════════════════════════════════════════════════════════════
  Case 1 | no-op 分支：未超 max_tokens 应直接放行
════════════════════════════════════════════════════════════
【输入】 3 条消息，累计约 34 tokens，max_tokens=5000（远未超）
【输出】 result = None
[PASS] 未超限直接放行，原文未动

════════════════════════════════════════════════════════════
  Case 2 | 主路径：真实对话硬截断，保留的每条字节级原样
════════════════════════════════════════════════════════════
【输入】 7 条真实数据分析对话，累计约 207 tokens，max_tokens=50, keep_last=2
         ─── 压缩前（全部原文） ───
         第1条 [SystemMessage] 你是数据分析助手，擅长电商数据诊断。
         第2条 [HumanMessage ] 我有一份 50 万行电商订单数据，想分析最近 30 天的销售趋势。字段包括 order_id、u...
         第3条 [AIMessage    ] 建议从时间维度（按日聚合）、品类维度（销售占比）、地域维度（Top 10 城市）三个方向切入。先...
         第4条 [HumanMessage ] 时间维度跑完发现最近 7 天销售额增长 40% 但订单量只增 15%，客单价从 265 涨到 3...
         第5条 [AIMessage    ] 客单价涨幅超过订单量，是消费升级或品类结构变化信号。下一步拆品类客单价定位原因。
         第6条 [HumanMessage ] 拆完了，数码 3C 贡献 80% 客单价涨幅。下一步怎么做？
         第7条 [AIMessage    ] 锁定 3C 品类后建议三个动作：查上新/调价记录、看 user_id 地域分布、做 SKU 级涨...

【输出】 硬

&emsp;&emsp;`trim_messages(strategy='last')` 的含义是：从消息列表**头部**删除旧消息，直到剩余 token 数 <= `max_tokens`。`include_system=True` 确保 `SystemMessage` 始终不被删除。`keep_last` 是一个保底机制——如果 token 限制过低导致 trim 删得太激进，我们至少保留最近 N 条，防止 Agent 完全失忆。

&emsp;&emsp;**与 SummarizationMiddleware 的叠加关系**：推荐顺序为 `SummarizationMiddleware -> MessageTrimMiddleware`（Trim 在后作为保底）。SummarizationMiddleware 先摘要中段，再由 Trim 确保最终不超硬上限。反向叠加（Trim 先）会让 Summarization 摘要一个已经被截断的历史，信息损失更大。

#### 1.1.5 Middleware 5：全局压缩重启（`CompactionMiddleware`）

&emsp;&emsp;最激进的压缩策略：当对话 token 总数超过阈值，将中段历史全部摘要为一条历史摘要 SystemMessage，只保留开头 SystemMessage + 历史摘要 + 最近 N 条。类似于清空记忆册，只留一句话总结。压缩比最高（可达 90%），但语义保真度取决于摘要质量，是**其他四种策略都失效时的最后手段**。

&emsp;&emsp;`CompactionMiddleware` 通常排在所有其他 Middleware 之后（`trigger_tokens=8000`），前四层已经大幅压缩后，Compaction 在极端情况才实际触发——这正是不要过早触发核选项的工程实践。

In [13]:
# 全局压缩提示词
COMPACTION_SUMMARY_PROMPT = """
请将以下对话历史压缩为一段简洁的上下文摘要，保留所有关键事实、决策和结论。
不要遗漏任何用户明确告知的信息。

对话历史：
{history}

输出格式：
[对话摘要]
（直接输出摘要内容，不要加前缀）
"""


class CompactionMiddleware(AgentMiddleware):
    """全局压缩重启 Middleware：超过 trigger_tokens 时执行完整压缩。

    Args:
        model:          LLM 实例，用于生成历史摘要
        trigger_tokens: token 总数超过此值才触发，默认 3500
        keep_recent:    压缩后保留的最近消息条数，默认 0
        token_counter:  token 计数函数；None 时用粗估
        summary_prompt: 摘要提示词模板，含 {history} 插槽
    """

    def __init__(
        self,
        model,
        trigger_tokens: int = 3500,  # 比上游 Trim/Mask 阈值高，作为最后保险丝
        keep_recent: int = 0,
        token_counter=None,
        summary_prompt: str = COMPACTION_SUMMARY_PROMPT,
    ) -> None:
        super().__init__()
        self.model = model
        self.trigger_tokens = trigger_tokens
        self.keep_recent = keep_recent
        self.token_counter = token_counter or count_tokens_approximately
        self.summary_prompt = summary_prompt

    def before_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        messages = state["messages"]
        total_tokens = self.token_counter(messages)

        # 未超阈值直接放行，不做任何压缩
        if total_tokens <= self.trigger_tokens:
            return None

        # 分离首条 SystemMessage，压缩完成后需放回列表最前
        if messages and isinstance(messages[0], SystemMessage):
            system_msg = messages[0]
            rest = messages[1:]
        else:
            system_msg = None
            rest = messages

        # 中段历史（去掉 keep_recent 尾部）才是压缩目标                                                                                                                                                           
        if self.keep_recent == 0:                                             
            # 全局重启语义：保留 system + 摘要，所有对话消息全部压缩                                    
            to_compact = rest                                                                                                                                                                                     
            recent = []
        elif len(rest) > self.keep_recent:                                                                                                                                                                        
            to_compact = rest[:-self.keep_recent]                                                                                                                                                                 
            recent = rest[-self.keep_recent:]                                                           
        else:                                                                                                                                                                                                     
            # 剩余消息不足 keep_recent，压缩无意义直接跳过                    
            return None 

        if not to_compact:
            return None

        # 截断每条消息至 500 字符，防止历史文本撑爆 LLM 上下文
        history_text = "\n".join(
            f"[{m.type.upper()}]: {str(m.content)[:500]}" for m in to_compact
        )
        prompt = self.summary_prompt.format(history=history_text)
        summary_text = self.model.invoke(prompt).content
        compaction_msg = SystemMessage(content=f"[历史对话摘要]\n{summary_text}")

        new_messages = []
        if system_msg:
            new_messages.append(system_msg)
        new_messages.append(compaction_msg)
        new_messages.extend(recent)

        return {
            "messages": [
                RemoveMessage(id=REMOVE_ALL_MESSAGES),
                *new_messages,
            ]
        }


# 实例化：5000 token 阈值（前四层压缩后通常达不到，属于保险丝角色）
# 注意：复用前面已经定义的全局 llm，不再重新创建 ChatDeepSeek 实例（避免全局污染）
compaction_mw = CompactionMiddleware(
    model=llm,
    trigger_tokens=5000,
    keep_recent=5,
)
print(f"CompactionMiddleware 实例化完成，trigger_tokens={compaction_mw.trigger_tokens}")

CompactionMiddleware 实例化完成，trigger_tokens=5000


In [14]:
# ============================================================
# Test: CompactionMiddleware
# 三个独立 case：no-op（真实短对话）/ 主压缩（真实对话）/ 边界保护（真实对话）
# ============================================================

# ------------------------------------------------------------
# Case 1: 未超 trigger_tokens → 不触发（真实短对话）
# ------------------------------------------------------------
_banner("Case 1 | no-op 分支：真实短对话未超 trigger，零压缩零 LLM 调用")

fake_c1 = FakeLLM(fixed_reply="不该被调用的摘要")
mw_c1 = CompactionMiddleware(model=fake_c1, trigger_tokens=1000, keep_recent=0)
state_c1 = {"messages": [
    SystemMessage(content="你是数据分析助手"),
    HumanMessage(content="今天的订单数据跑完了，你帮我看看结果？"),
    AIMessage(content="好的，把你的 notebook 或结果截图发我，我来分析下"),
    HumanMessage(content="趋势正常，没什么异常值"),
    AIMessage(content="那就先归档吧，明天再跑对比"),
]}
total_c1 = count_tokens_approximately(state_c1["messages"])
print(f"【输入】 5 条真实短对话，累计约 {total_c1} tokens，trigger=1000（未超）")

result_c1 = mw_c1.before_model(state_c1, runtime=None)
print(f"【输出】 result = {result_c1}")
print(f"【核对】 FakeLLM 调用次数 = {fake_c1.call_count}（期望 0）")

assert result_c1 is None, "未超 trigger 应返回 None"
assert fake_c1.call_count == 0, "未触发压缩时不应调用 LLM"
print("[PASS] 未触发压缩 / LLM 零调用")


# ------------------------------------------------------------
# Case 2: 全局重启（keep_recent=0）→ 真实多轮对话压缩为一段摘要
# 混合策略：本 case 用真实 DeepSeek `llm`，让学员看到真实 Compaction 效果
#           （Case 1/3 仍用 FakeLLM，保证 call_count == 0 的硬断言）
# ------------------------------------------------------------
_banner("Case 2 | 主路径 · 全局重启（真实 LLM）：真实多轮对话 → 一段真实摘要")

mw_c2 = CompactionMiddleware(model=llm, trigger_tokens=200, keep_recent=0)

state_c2 = {"messages": [
    SystemMessage(content="你是数据分析助手，擅长电商数据诊断。"),
    HumanMessage(content=(
        "我手头有一份电商平台订单数据，CSV 格式 50 万行，时间跨度最近 90 天。"
        "字段包括 order_id、user_id、category、amount、order_time、city、"
        "payment_method 共 7 列。老板让我分析最近 30 天的销售趋势，重点找"
        "增长点和下滑品类。你建议我从哪几个维度切入？"
    )),
    AIMessage(content=(
        "50 万行属于中等规模，Pandas 单机完全够用。建议从三个正交维度切入："
        "第一是时间维度，按日聚合销售额和订单量，看是否存在节假日峰值或工作日"
        "规律；第二是品类维度，计算各 category 的销售占比和环比变化，找出"
        "增长和下滑最快的 Top 3；第三是地域维度，按 city 分组看 Top 10 城市"
        "的销售贡献。建议先跑一次 describe() 摸清数据分布再定方法。"
    )),
    HumanMessage(content=(
        "时间维度跑完了，结果挺有意思：最近 7 天销售额比前 23 天的日均值高了"
        " 40%，但订单量只增长 15%。算了一下客单价，最近 7 天是 328 元，前 "
        "23 天是 265 元，涨幅 24%。这组数据说明什么？下一步该看什么？"
    )),
    AIMessage(content=(
        "客单价涨幅远大于订单量涨幅，是典型的消费升级或品类结构变化信号，而不"
        "是单纯的流量拉动。通常有两种可能：一是最近 7 天有促销活动把高单价"
        "商品推上去了；二是用户结构变化吸引了更高消费能力的客群。下一步最该"
        "做的是品类维度的客单价拆解——单一品类拉高是促销驱动，多品类普涨是"
        "用户结构变化。"
    )),
    HumanMessage(content=(
        "按你说的拆完了，发现数码 3C 品类贡献了 80% 的客单价涨幅，服装、食品、"
        "家居这三个品类的客单价基本持平，只有 3C 单品类在拉高整体数据。这种"
        "情况下下一步应该怎么做？想定位到更具体的原因。"
    )),
    AIMessage(content=(
        "数码 3C 单品类拉高整体客单价，原因可以锁定了。建议交叉三个动作做根因"
        "分析：第一，查最近 7 天 3C 类目的上新记录和调价记录；第二，看 3C 高"
        "客单价订单的 user_id 分布，如果集中在少数城市可能是地域定向活动；"
        "第三，对 3C 类目内部做 SKU 维度的客单价涨幅排序，找出 Top 3-5 爆款"
        "SKU。三步跑完能把现象落到具体 SKU 级原因。"
    )),
]}
total_c2 = count_tokens_approximately(state_c2["messages"])

print(f"【输入】 7 条真实对话消息，累计约 {total_c2} tokens")
print(f"         trigger=200, keep_recent=0（全局重启，不保留任何原始对话）")
print("         ─── 压缩前的 7 条消息 ───")
for i, m in enumerate(state_c2["messages"], 1):
    mtype = type(m).__name__
    preview = str(m.content)[:55] + ("..." if len(str(m.content)) > 55 else "")
    print(f"         第{i}条 [{mtype:13s}] {preview}")

result_c2 = mw_c2.before_model(state_c2, runtime=None)
new_msgs_c2 = result_c2["messages"][1:]

print(f"\n【输出】 处理后只剩 {len(new_msgs_c2)} 条消息（期望 2：原 System + 真实摘要 System）")
print("         ─── 压缩后的消息（完整内容，第2条是真实 DeepSeek 生成的摘要）───")
for i, m in enumerate(new_msgs_c2, 1):
    print(f"         第{i}条 [{type(m).__name__}]")
    for line in str(m.content).split("\n"):
        print(f"                 {line}")
    print()
print(f"【核对】 消息数 {len(state_c2['messages'])} → {len(new_msgs_c2)}（压缩比 {len(state_c2['messages'])/len(new_msgs_c2):.1f}x）")

# 结构不变式
assert len(new_msgs_c2) == 2, f"全局重启后应只剩 2 条，实际 {len(new_msgs_c2)}"
assert isinstance(new_msgs_c2[0], SystemMessage)
assert new_msgs_c2[0].content == "你是数据分析助手，擅长电商数据诊断。"
assert isinstance(new_msgs_c2[1], SystemMessage)

# 真实 LLM 内容断言：放宽匹配，命中核心业务关键词中的任意 2 个即可
summary_text = new_msgs_c2[1].content
assert len(summary_text) > 30, "摘要应有实质内容（至少 30 字符）"
core_keywords = ["50 万行", "数码 3C", "销售额", "客单价", "订单", "电商", "3C", "品类"]
hit_keywords = [kw for kw in core_keywords if kw in summary_text]
assert len(hit_keywords) >= 2, \
    f"真实 LLM 摘要应至少包含 2 个核心业务关键词，实际命中 {len(hit_keywords)}：{hit_keywords}"

# 消极证据：6 条原始对话全部消失（不依赖 LLM 输出内容）
for orig in state_c2["messages"][1:]:
    assert orig.content not in [m.content for m in new_msgs_c2], \
        f"原始对话 '{orig.content[:20]}...' 不应出现在结果"

print(f"[PASS] 全局重启生效 / 只剩 System+真实摘要 / 6 条原始对话消失 / 命中关键词: {hit_keywords}")


# ------------------------------------------------------------
# Case 3: 超阈值但对话数不足 keep_recent → 边界保护（真实短对话）
# ------------------------------------------------------------
_banner("Case 3 | 边界保护：可压缩消息不足 keep_recent 时放弃压缩")

fake_c3 = FakeLLM(fixed_reply="不该被调用的摘要")
mw_c3 = CompactionMiddleware(model=fake_c3, trigger_tokens=100, keep_recent=5)
state_c3 = {"messages": [
    SystemMessage(content="你是数据分析助手"),
    HumanMessage(content=(
        "我刚拿到 2024 年全年电商订单数据，共 180 万行，想做年度趋势分析，"
        "你建议用什么工具和切入点？"
    )),
    AIMessage(content=(
        "180 万行建议直接用 Pandas，先做月度聚合看同比环比，再看品类和地域"
        "维度，最后做季节性分解。"
    )),
]}
total_c3 = count_tokens_approximately(state_c3["messages"])
print(f"【输入】 3 条真实对话（1 System + 2 对话），约 {total_c3} tokens")
print(f"         trigger=100, keep_recent=5（虽超 trigger，但对话数 2 < keep_recent）")

result_c3 = mw_c3.before_model(state_c3, runtime=None)
print(f"【输出】 result = {result_c3}")
print(f"【核对】 FakeLLM 调用次数 = {fake_c3.call_count}（期望 0）")

assert result_c3 is None, "中段可压缩消息不足时应返回 None"
assert fake_c3.call_count == 0, "无可压缩内容时不应调用 LLM"
print("[PASS] 识别中段不足 keep_recent / 放弃压缩 / LLM 零调用")


# ============================================================
print("\n" + "═" * 60)
print("  CompactionMiddleware 全部 3 个 case 通过 [PASS]")
print("═" * 60)



════════════════════════════════════════════════════════════
  Case 1 | no-op 分支：真实短对话未超 trigger，零压缩零 LLM 调用
════════════════════════════════════════════════════════════
【输入】 5 条真实短对话，累计约 43 tokens，trigger=1000（未超）
【输出】 result = None
【核对】 FakeLLM 调用次数 = 0（期望 0）
[PASS] 未触发压缩 / LLM 零调用

════════════════════════════════════════════════════════════
  Case 2 | 主路径 · 全局重启（真实 LLM）：真实多轮对话 → 一段真实摘要
════════════════════════════════════════════════════════════
【输入】 7 条真实对话消息，累计约 536 tokens
         trigger=200, keep_recent=0（全局重启，不保留任何原始对话）
         ─── 压缩前的 7 条消息 ───
         第1条 [SystemMessage] 你是数据分析助手，擅长电商数据诊断。
         第2条 [HumanMessage ] 我手头有一份电商平台订单数据，CSV 格式 50 万行，时间跨度最近 90 天。字段包括 order_id、u...
         第3条 [AIMessage    ] 50 万行属于中等规模，Pandas 单机完全够用。建议从三个正交维度切入：第一是时间维度，按日聚合销售额和订...
         第4条 [HumanMessage ] 时间维度跑完了，结果挺有意思：最近 7 天销售额比前 23 天的日均值高了 40%，但订单量只增长 15%。算...
         第5条 [AIMessage    ] 客单价涨幅远大于订单量涨幅，是典型的消费升级或品类结构变化信号，而不是单纯的流量拉动。通常有两种可能：一是最近...
         第6条 [HumanMessage ] 按你说的拆完

&emsp;&emsp;`CompactionMiddleware` 与 `SummarizationMiddleware` 的最大区别在于触发策略和输出结构：Summarization 是保留最近 K 条原始消息 + 一条摘要，Compaction 是重置历史，只留一条摘要 SystemMessage + 最近 N 条。Compaction 之后消息数可以从数十条压缩到个位数，但这也意味着摘要失真的代价更大——因此 `trigger_tokens=8000` 应当设为比其他 Middleware 的阈值高一档，作为真正的最后防线。

&emsp;&emsp;**五件套的完整叠加顺序**：`ToolResultClearMiddleware -> ObservationMaskMiddleware -> SummarizationMiddleware -> MessageTrimMiddleware -> CompactionMiddleware`。前四层逐步清理，CompactionMiddleware 只在极端情况才实际触发，两者之间的成本差距正是加法优先设计原则的体现。

#### 1.1.6 五件套叠加对照实验

&emsp;&emsp;五件套设计好了，接下来跑一个对照实验，直观看到从无压缩到五件全开的压缩率和一致性变化。实验以相同的测试对话为输入，依次叠加 Middleware，用 `compression_report` 输出每一档的量化指标。五个中间件的 `before_model` 签名已全部统一为 `(state, runtime)`，对照实验代码不再需要 try/except 猜签名。

In [15]:
from compression_metrics import compression_report
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain.agents.middleware import SummarizationMiddleware
try:
    from langchain.messages import RemoveMessage
except ImportError:
    from langchain_core.messages.modifier import RemoveMessage

# ============================================================
# 1.2.6 五件套叠加对照实验（真实生产 Bug 排查场景）
# ------------------------------------------------------------
# 场景：Agent 帮用户诊断 /api/orders POST 接口的 500 错误
#       依次调用 read_logs / grep / read_file / run_sql / run_test /
#       git_blame 六类工具，累计 ~2000 tokens，覆盖所有压缩场景
# ============================================================

# ---------- 实验专用 middleware 实例（阈值针对本实验调优）----------
exp_trim = MessageTrimMiddleware(max_tokens=800, keep_last=4)
exp_clear = ToolResultClearMiddleware(
    llm=FakeLLM(fixed_reply="[Clear 摘要] 工具返回的日志/代码/SQL 输出片段（原文已被压缩）"),
    keep_recent_tool_results=1,
)
exp_mask = ObservationMaskMiddleware(keep_last_n_observations=1)
exp_sum = SummarizationMiddleware(
    model=FakeListChatModel(responses=[
        "[早期对话总结] Agent 查询了 nginx 日志（18:30 起 32% 请求 500）、"
        "Python traceback（Lock wait timeout）、orders.py 代码（with_for_update 行锁）、"
        "InnoDB 状态（热门商品行锁争用），初步定位到并发创建订单的竞态条件。"
    ]),
    trigger=("tokens", 400),
    keep=("messages", 4),
)
exp_compaction = CompactionMiddleware(
    model=FakeLLM(fixed_reply=(
        "[全局摘要] Agent 通过 6 次工具调用（日志/grep/代码/SQL/测试/git）"
        "定位生产 500 错误根因：commit abc1234 引入的 with_for_update 行锁在"
        "热门商品并发场景下触发 Lock wait timeout。建议回滚换乐观锁或引入"
        "Redis 预扣库存。"
    )),
    trigger_tokens=1200,
    keep_recent=2,
)

# ---------- 真实生产 Bug 排查对话（~2000 tokens）----------
test_messages = [
    SystemMessage(content="你是生产环境 Bug 排查助手，擅长从日志、代码、数据库多源定位问题。"),
    HumanMessage(content=(
        "生产环境 /api/orders POST 接口从昨天 18:30 开始出现 500 错误，"
        "大概 30% 的请求受影响，订单创建失败。error rate 从 0.1% 飙到 32%。"
        "帮我定位原因。"
    )),
    # Turn 1: read_logs
    AIMessage(
        content="先看看 nginx access log 和 app error log 的时间点，找到第一波 500 的请求特征。",
        tool_calls=[{"id": "t1", "name": "read_logs", "args": {"file": "nginx/access.log"}}],
    ),
    ToolMessage(
        content=(
            "2026-04-14 18:28:55 POST /api/orders 200 145b 45ms\n"
            "2026-04-14 18:29:12 POST /api/orders 200 145b 52ms\n"
            "2026-04-14 18:30:03 POST /api/orders 500 82b 3245ms\n"
            "2026-04-14 18:30:07 POST /api/orders 200 145b 48ms\n"
            "2026-04-14 18:30:14 POST /api/orders 500 82b 3812ms\n"
            "2026-04-14 18:30:18 POST /api/orders 500 82b 3901ms\n"
            "2026-04-14 18:30:22 POST /api/orders 200 145b 51ms\n"
            "... (共 1847 次 POST，其中 592 次返回 500 = 32.1%)\n"
            "500 响应平均耗时 3500ms，远高于正常 50ms"
        ),
        tool_call_id="t1", name="read_logs",
    ),
    # Turn 2: grep Traceback
    AIMessage(
        content="500 都是 3+ 秒才返回，像超时。grep app.log 看 Python Traceback",
        tool_calls=[{"id": "t2", "name": "grep", "args": {"pattern": "Traceback"}}],
    ),
    ToolMessage(
        content=(
            "File \"/app/api/orders.py\", line 47, in create_order\n"
            "    db.session.commit()\n"
            "File \"sqlalchemy/orm/session.py\", line 1468, in commit\n"
            "    self._transaction.commit(_to_root=self.future)\n"
            "File \"sqlalchemy/engine/base.py\", line 2591, in commit\n"
            "sqlalchemy.exc.OperationalError: (pymysql.err.OperationalError)\n"
            "(1205, 'Lock wait timeout exceeded; try restarting transaction')\n"
            "共匹配 589 次"
        ),
        tool_call_id="t2", name="grep",
    ),
    # Turn 3: read_file
    AIMessage(
        content="确认是 MySQL 锁等待超时。读 orders.py 的 create_order 函数",
        tool_calls=[{"id": "t3", "name": "read_file", "args": {"path": "app/api/orders.py"}}],
    ),
    ToolMessage(
        content=(
            "@app.route('/api/orders', methods=['POST'])\n"
            "def create_order():\n"
            "    data = request.get_json()\n"
            "    order = Order(user_id=data['user_id'], status='pending', total=0)\n"
            "    db.session.add(order)\n"
            "    db.session.flush()\n"
            "    total = 0\n"
            "    for item in data['items']:\n"
            "        # 这里用了 with_for_update() 加行锁 <-- 可疑\n"
            "        product = Product.query.filter_by(id=item['product_id']) \\\n"
            "                         .with_for_update().first()\n"
            "        if product.stock < item['quantity']:\n"
            "            raise BusinessError('库存不足')\n"
            "        product.stock -= item['quantity']\n"
            "        db.session.add(OrderItem(order_id=order.id, ...))\n"
            "        total += product.price * item['quantity']\n"
            "    order.total = total\n"
            "    db.session.commit()  # 这里挂了\n"
            "    return jsonify(order_id=order.id), 201"
        ),
        tool_call_id="t3", name="read_file",
    ),
    # Turn 4: run_sql
    AIMessage(
        content="看到 with_for_update() 加 X 锁。查 MySQL SHOW ENGINE INNODB STATUS 看锁等待详情",
        tool_calls=[{"id": "t4", "name": "run_sql", "args": {"sql": "SHOW ENGINE INNODB STATUS"}}],
    ),
    ToolMessage(
        content=(
            "---TRANSACTION 2847291847, ACTIVE 3 sec starting index read\n"
            "mysql tables in use 1, locked 1\n"
            "LOCK WAIT 3 lock struct(s), 2 row lock(s)\n"
            "Waiting for lock on: `shop`.`product` lock_mode X waiting\n"
            "Record lock, heap no 47 PHYSICAL RECORD: n_fields 12\n"
            "\n"
            "统计: 共 23 个事务在等待 product 表行锁\n"
            "最长等待 3.8 秒\n"
            "涉及 product_id: 12, 34, 89（销售热门款）"
        ),
        tool_call_id="t4", name="run_sql",
    ),
    # Turn 5: run_test
    AIMessage(
        content="锁争用集中在热门商品行锁。跑并发测试复现",
        tool_calls=[{"id": "t5", "name": "run_test", "args": {"test": "test_create_concurrent"}}],
    ),
    ToolMessage(
        content=(
            "tests/test_orders.py::test_create_concurrent_orders_hot_product FAILED\n"
            "\n"
            "def test_create_concurrent_orders_hot_product():\n"
            "    '''模拟 50 并发请求购买热门商品 id=12'''\n"
            "    with ThreadPoolExecutor(max_workers=50) as ex:\n"
            "        futures = [ex.submit(create_order_req, pid=12) for _ in range(50)]\n"
            "        results = [f.result() for f in futures]\n"
            ">   assert sum(r.status_code == 201 for r in results) == 50\n"
            "E   assert 21 == 50\n"
            "E   实际: 21 成功 / 29 超时 (500)\n"
            "E   超时请求平均耗时: 3912ms\n"
            "E   成功请求平均耗时: 287ms\n"
            "FAILED 1 test, 0 passed in 18.32s"
        ),
        tool_call_id="t5", name="run_test",
    ),
    # Turn 6: git_blame
    AIMessage(
        content="复现了。看看 orders.py 最近谁改的 with_for_update",
        tool_calls=[{"id": "t6", "name": "git_blame", "args": {"file": "app/api/orders.py"}}],
    ),
    ToolMessage(
        content=(
            "commit abc1234 2026-04-14 14:23:11 zhangwei\n"
            "    feat(orders): 添加库存并发保护，使用 with_for_update 加行锁\n"
            "    - 解决 issue #1823: 库存超卖问题\n"
            "    - 在 create_order 中对 Product.query 加 X 锁\n"
            "\n"
            "commit def5678 2026-04-10 09:12:45 lisi\n"
            "    fix(orders): 修复订单总价计算错误\n"
            "\n"
            "commit ghi9012 2026-04-08 16:34:22 wangwu\n"
            "    refactor(orders): 重构订单创建逻辑"
        ),
        tool_call_id="t6", name="git_blame",
    ),
    # 最终结论
    AIMessage(content=(
        "定位清楚了。commit abc1234 昨天 14:23 为解决库存超卖问题引入 "
        "with_for_update 行锁，但没考虑热门商品并发场景。多用户同时购买 "
        "product_id=12 这类热款时，行锁争用导致 3+ 秒等待触发 InnoDB lock "
        "wait timeout。建议：①短期回滚 abc1234 换乐观锁（version + CAS）"
        "解决超卖；②长期引入 Redis 预扣库存（lua 原子操作）解耦库存锁。"
    )),
]
total_tokens = count_tokens_approximately(test_messages)
print(f"测试对话构造完成：{len(test_messages)} 条消息，约 {total_tokens} tokens")
print(f"场景：Agent 通过 6 次工具调用诊断 /api/orders 500 错误并给出修复方案")
print()

# ============================================================
# 五件套叠加对照实验配置（按推荐叠加顺序）
# ============================================================
EXPERIMENT_CONFIGS = {
    "A: 无压缩 (baseline)":  [],
    "B: +硬截断":             [exp_trim],
    "C: +工具清除":           [exp_clear, exp_trim],
    "D: +观察遮蔽":           [exp_clear, exp_mask, exp_trim],
    "E: +摘要压缩":           [exp_clear, exp_mask, exp_sum, exp_trim],
    "F: 五件全开":            [exp_clear, exp_mask, exp_sum, exp_trim, exp_compaction],
}

# 表头
print(f"{'配置':<22} {'压缩率':>7} {'一致性':>7} {'before→after':>15}   {'最早保留内容':<42} {'最终结论':<34}")
print("─" * 138)

for name, mws in EXPERIMENT_CONFIGS.items():
    state = {"messages": list(test_messages)}
    for mw in mws:
        # 兼容 langchain Summarization 不传 runtime 的签名
        try:
            result = mw.before_model(state, None)
        except TypeError:
            result = mw.before_model(state)
        if result is None:
            continue
        if isinstance(result, dict) and "messages" in result:
            # 过滤掉 RemoveMessage 哨兵
            new_msgs = [m for m in result["messages"] if not isinstance(m, RemoveMessage)]
            state = {"messages": new_msgs}

    report = compression_report(
        test_messages,
        state["messages"],
        token_counter=count_tokens_approximately,
    )
    earliest = report["earliest_preview_after"][:40]
    latest = report["latest_ai_preview_after"][:32]
    print(
        f"{name:<22} "
        f"{report['compression_ratio']:>7.1%} "
        f"{report['consistency']:>7.3f} "
        f"{report['before_tokens']:>6}→{report['after_tokens']:<6}   "
        f"{earliest:<42} {latest:<34}"
    )

print()
print("─" * 138)
print("观察要点：")
print("  A→B  硬截断使 tokens 线性下降，但 consistency 接近 1（保留的都是原文片段）")
print("  B→C  工具清除是跳升最显著的一档（ToolMessage 是最大 token 来源）")
print("  C→D  遮蔽比摘要更激进（无 LLM 调用，直接占位符替换）")
print("  D→E  首次触动 AIMessage/HumanMessage（前 N-keep 条 → 1 条总结 SystemMessage）")
print("  E→F  全局重启 = 最后防线，把所有早期对话压成一段话")
print()
print("关键指标：所有配置的『最终结论』列都应包含『commit abc1234』等核心信息 → 结论不丢")


测试对话构造完成：15 条消息，约 1270 tokens
场景：Agent 通过 6 次工具调用诊断 /api/orders 500 错误并给出修复方案

配置                         压缩率     一致性    before→after   最早保留内容                                     最终结论                              
──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
A: 无压缩 (baseline)         0.0%   1.000   1270→1270     2026-04-14 18:28:55 POST /api/orders 200   定位清楚了。commit abc1234 昨天 14:23 为解  
B: +硬截断                  51.1%   0.995   1270→621      ---TRANSACTION 2847291847, ACTIVE 3 sec    定位清楚了。commit abc1234 昨天 14:23 为解  
C: +工具清除                 55.1%   0.789   1270→570      [摘要] [Clear 摘要] 工具返回的日志/代码/SQL 输出片段（原文已被   定位清楚了。commit abc1234 昨天 14:23 为解  
D: +观察遮蔽                 59.4%   0.789   1270→516      [观察已遮蔽: read_logs 返回 43 字符]                定位清楚了。commit abc1234 昨天 14:23 为解  
E: +摘要压缩                 70.7%   0.670   1270→372      [观察已遮蔽: run_test 返回 43 字符]                 定位清楚了。commit abc1

&emsp;&emsp;观察实验输出表格，可以发现几个清晰规律：**A→B**（加硬截断）tokens 线性下降但 `consistency` 接近 1——硬截断只砍不改，保留的每条都是原文字节；**B→C**（加工具清除）是跳升最显著的一档，从 4.8% 直接跳到 65.3%，因为 `ToolMessage` 往往是单条最大的 token 来源；**C→D**（加观察遮蔽）在此基础上进一步压缩到 68.0%，`Mask` 比 `Clear` 更激进的地方在于它无 LLM 调用、直接占位符替换；**D→E**（加摘要压缩）首次触动 `AIMessage` / `HumanMessage`，把早期对话整批替换为一条 `Here is a summary...` 的 `HumanMessage`；**E→F**（加 `Compaction`）压缩率和 `D→E` 完全相同（都是 75.7%），这不是 bug——**正是 `Compaction` 作为"最后防线"的设计哲学被验证**：前面四层已经把 tokens 压到 202，远低于 `Compaction` 的 `trigger_tokens=1200`，它自然不触发。如果把 `Compaction` 的阈值调到和其他层一样低，它就失去了"保底兜底"的角色意义。

&emsp;&emsp;<font color=red>三条必须记住的生产坑</font>：通过真跑实验我们暴露了 LangChain 1.x 中 `SummarizationMiddleware` 的几个反直觉行为，这些在官方文档里没有突出说明，生产里踩一次要查半天。

&emsp;&emsp;**坑 1：`SummarizationMiddleware` 的摘要是 `HumanMessage` 不是 `SystemMessage`**。很多人看到"对话总结"这个名字就以为摘要会以 system prompt 形式注入，实际上 LangChain 1.x 把摘要包装成一条 `HumanMessage`，内容前缀是硬编码的 `"Here is a summary of the conversation to date:"`——这个前缀无法通过参数关闭。下游 Agent 看到这条消息时，会把它当作"用户告诉我的上下文"而不是"系统设定"，在某些对话路径下会引起微妙的语义漂移。这也是为什么本课的 `CompactionMiddleware` 选择用 `SystemMessage + [历史对话摘要]` 前缀的原因——更接近"系统级上下文"的语义。

&emsp;&emsp;**坑 2：`SummarizationMiddleware` 不会自动保留首条 `SystemMessage`**。它把整段消息当成一条流水线，`keep` 窗口之外的部分会被整批替换，**包括原始的 system prompt**。如果你原本依赖 system prompt 约束 Agent 的角色边界和工具使用规则，Summarization 触发后这些约束就消失了——Agent 进入"裸奔"状态。`CompactionMiddleware` 显式把首条 `SystemMessage` 剥离出来再塞回压缩结果的第一位，正是为了堵这个坑。生产里如果必须用 `SummarizationMiddleware`，请手动 fork 它的 `before_model` 加上 system prompt 保护。



In [17]:
# ============================================================
# 1.2.7 把五件套挂到 create_agent 上做端到端验证
# ------------------------------------------------------------
# 和 1.2.6 的差别：不再手动调 before_model，而是让 create_agent
# 在真实模型调用前自动跑 middleware 链，让真实 DeepSeek 消费
# 压缩后的上下文，观察推理结论是否仍能命中关键证据 commit abc1234
#
# 关键约束（基于 PROBE 实测）：
#   1. langchain 的 before_model 按列表顺序触发（A→B→C，非反向）
#   2. 同一类 middleware 在一个 agent 内不能出现两次（重复实例报错）
#   3. trim/sum/compaction 用 1.2.6 实测的 ~1300 tokens 调阈值
#      确保每档都会触发，且 trim_messages 内部 pair-aware 不破坏配对
# ============================================================

# ---------- 1.2.7 专用 middleware 实例（与 1.2.6 隔离，避免互相覆盖）----------
mw_trim = MessageTrimMiddleware(max_tokens=600, keep_last=4)

mw_clear = ToolResultClearMiddleware(
    llm=FakeLLM(fixed_reply="[Clear 摘要] 工具返回的日志/代码/SQL 输出（原文已被压缩）"),
    keep_recent_tool_results=1,
)

mw_mask = ObservationMaskMiddleware(keep_last_n_observations=1)

mw_sum = SummarizationMiddleware(
    model=FakeListChatModel(responses=[
        "[早期对话总结] Agent 通过 6 次工具调用（日志/grep/代码/SQL/测试/git）"
        "定位 /api/orders 500 错误：commit abc1234 引入的 with_for_update 行锁"
        "在热门商品并发场景触发 Lock wait timeout。"
    ]),
    trigger=("tokens", 500),
    keep=("messages", 4),
)

mw_compaction = CompactionMiddleware(
    model=FakeLLM(fixed_reply=(
        "[全局摘要] 6 次工具调用定位生产 500 错误根因：commit abc1234 引入的 "
        "with_for_update 行锁在热门商品并发场景下触发 Lock wait timeout。"
        "建议回滚换乐观锁或引入 Redis 预扣库存。"
    )),
    trigger_tokens=600,
    keep_recent=2,
)

# 探针：强制 DeepSeek 在结论中引用 commit hash，而不是用"昨天14:23"等替代
probe = HumanMessage(content="请引用 commit hash，一句话总结这次 Bug 的根因和修复方案。")

# ---------- 六档配置（列表顺序 = 执行顺序，温和 → 激进）----------
EXPERIMENT_CONFIGS_AGENT = {
    "A: 无压缩 (baseline)":  [],
    "B: +硬截断":             [mw_trim],
    "C: +工具清除":           [mw_clear, mw_trim],
    "D: +观察遮蔽":           [mw_clear, mw_mask, mw_trim],
    "E: +摘要压缩":           [mw_clear, mw_mask, mw_sum, mw_trim],
    "F: 五件全开":            [mw_clear, mw_mask, mw_sum, mw_trim, mw_compaction],
}

print(f"{'配置':<22} {'压缩率':>7} {'before→after':>15}  {'命中':>6}  {'最终回复(前60字)':<60}")
print("─" * 130)

for name, mws in EXPERIMENT_CONFIGS_AGENT.items():
    # 每个配置一个新 agent，避免跨配置实例污染
    agent = create_agent(
        model=llm,
        tools=[],
        system_prompt="你是生产环境 Bug 排查助手，擅长从日志、代码、数据库多源定位问题。",
        middleware=mws,
    )

    try:
        result = agent.invoke({"messages": list(test_messages) + [probe]})
        final_msgs = result["messages"]
        new_reply = final_msgs[-1]
        # 末尾是新生成的 AIMessage，倒数第二条是 probe，再往前是压缩后的历史
        compressed_history = final_msgs[:-2]

        report = compression_report(
            test_messages,
            compressed_history,
            token_counter=count_tokens_approximately,
        )
        hit = "abc1234" in (new_reply.content or "")
        reply_preview = new_reply.content.replace("\n", " ")[:60]
        print(
            f"{name:<22} "
            f"{report['compression_ratio']:>7.1%} "
            f"{report['before_tokens']:>6}→{report['after_tokens']:<6}  "
            f"{'[PASS]' if hit else '[FAIL]':>6}  "
            f"{reply_preview:<60}"
        )
    except Exception as e:
        print(f"{name:<22} EXCEPTION: {type(e).__name__}: {str(e)[:80]}")

print()
print("─" * 130)
print("观察要点：")
print("  · 1.2.6 用 FakeLLM 测纯压缩效果（仅 middleware 行为）")
print("  · 1.2.7 用真实 DeepSeek 测端到端：压缩后的上下文能否支撑正确推理")
print("  · 命中 abc1234 = 关键证据未丢；若某档 [FAIL]，说明该档压缩损伤了核心信息")


配置                         压缩率    before→after      命中  最终回复(前60字)                                                  
──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
A: 无压缩 (baseline)         0.0%   1270→1270    [PASS]  根因：commit abc1234 引入的 `with_for_update()` 行锁在高并发购买热门商品时导致 My
B: +硬截断                  62.6%   1270→475     [PASS]  根因：commit abc1234 引入的 `with_for_update` 行锁在高并发购买热门商品时引发锁争用，导
C: +工具清除                 55.5%   1270→565     [PASS]  根因：commit `abc1234` 引入的 `with_for_update()` 行锁在高并发下单场景下导致热门商
D: +观察遮蔽                 59.4%   1270→516     [PASS]  根因：commit abc1234 引入的 `with_for_update()` 行锁在高并发场景下导致热门商品锁等待
E: +摘要压缩                 73.8%   1270→333     [PASS]  根据 commit abc1234 引入的 `with_for_update` 行锁，根因是**在高并发场景下，对热门商
F: 五件全开                  73.8%   1270→333     [PASS]  根因：commit `abc1234` 引入的 `with_for_update` 行锁在高并发场景下触发 InnoDB

─────────────────────────────────────────────────────────────

### 1.3 本章小结与下一步

&emsp;&emsp;本章我们建立了"叠加顺序就是策略"这个核心认知，跑通了正确叠加版本，看到了 M2 / M4 两个模块在主战场地图上被点亮。但是压缩有一个无法回避的代价：被压掉的早期对话内容真的丢了，下一次用户问起前面的某个细节时，Agent 会"失忆"。

&emsp;&emsp;<font color=red>这意味着我们必须给 Agent 装一个真正的长期记忆层。</font>下一章不再重复基础课的"Mem0 三种 Write 是什么"，而是讲怎么把 Mem0 编排成 `Mem0WriteMiddleware`——在每次 LLM 响应后异步写入向量库，并配套 `SessionWriteMiddleware` 和 `TaskStateMiddleware`，构建完整的三层 Write 架构。

---

## <center>第 2 章：Write 写入策略</center>

&emsp;&emsp;第 1 章解决的是"单次对话消耗太多 token"的问题——通过 `SummarizationMiddleware` 和 `ModelCallLimitMiddleware` 的组合编排，我们把同一轮对话里冗余的历史压到最小。但压缩只是"瘦身"，会话一旦结束，所有内容随进程消亡，下次打开 Agent 一切从零开始。用户前天告诉 Agent"我们选定用 DeepSeek + LangChain"，今天重开会话，Agent 毫无记忆——这不是压缩能解决的，这是**写入缺失**的问题。

&emsp;&emsp;本章引入三层 Write 架构，把对话内容持久化到三种不同的存储介质，分别对应三种失忆场景：**Layer 1 — SessionWriteMiddleware** 在同一进程内的会话间传递短期上下文，**Layer 2 — Mem0WriteMiddleware** 把决策型对话写进向量库实现跨会话长期记忆，**Layer 3 — TaskStateMiddleware** 把任务指令追加到结构化的 `todo.md` 实现任务级持久化。三层各自独立、可按需叠加，通过 Middleware 链串联进同一个 Agent 流程。

&emsp;&emsp;实现机制上，三层共享一个核心设计理念：**写入是副作用，不阻断主链路**。每个 Middleware 的 `__call__` 方法读取 `context`、判断触发条件、执行写入，然后原样返回 `context`——主链路始终不被打断，写入失败只打印 WARN 而不抛异常。接下来 2.1 到 2.4，我们逐层搭建，最后在 2.4 合流做一次九轮对话的集成演示，用三层数据验证写入结果。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260417140100591.svg" width=60%></div>

### 2.1 SessionWrite：会话级短期记忆

&emsp;&emsp;第 1 章的 `Compression` 只是 in-memory 瘦身——把历史消息折叠成一段摘要，但这段摘要仍然活在当前进程的 `messages` 列表里，会话结束即丢失。Layer 1 要迈出下一步：**把摘要和最近消息真正持久化到磁盘**，让进程重启、或者切换 session_id 后依然能恢复上下文。

&emsp;&emsp;基础入门课 `SessionManager` 已经给出了一套经过验证的思路：消息数达到阈值时，把前半部分用 LLM 压缩成 `compressed_context`，只保留最近 `KEEP_RECENT` 条消息原文，整个 session 按 `session_id` 落盘到 `sessions/*.json`。这套逻辑经济实用——既能控制下一次调用时携带的 token 量，又通过 JSON 文件实现了零依赖的持久化，不需要引入任何外部存储。

&emsp;&emsp;接下来三个小节的路线图是：2.1.1 把上面这套逻辑写成可独立运行的过程式代码，讲清每一步的意义；2.1.2 将它最小封装成 `LangChain Middleware`，让它能接入 Agent 主链路；2.1.3 用一段独立测试验证触发压缩和读回 JSON 文件两件事。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Compression 压缩层 vs Write Layer 1 写入层职责对比</font></p>
<div class="center">

| 对比维度 | 压缩层（Ch1 Compression） | 写入层（Layer 1） |
|----------|--------------------------|------------------|
| **触发时机** | LLM 调用前（每轮） | 消息数达到阈值时 |
| **操作对象** | in-memory messages | `sessions/*.json` |
| **生命周期** | 单进程内有效 | 跨进程持久 |
| **存储介质** | 无（纯内存操作） | 文件系统 |
| **核心动作** | 替换前历史为摘要 | 把早期消息压缩并落盘 |

</div>

#### 2.1.1 核心逻辑：SessionManager 式文件持久化

&emsp;&emsp;参考基础入门课 `SessionManager` 的实现，Layer 1 的核心是这样一段过程式代码，整个流程分三步走：第一步，创建一个带 `compressed_context` 和 `messages` 字段的 session dict，写入 JSON 文件（此时 compressed_context 为空，messages 有 8 条）；第二步，检查消息数是否达到 `COMPRESS_TRIGGER`，若达到则把前半部分 `early_messages` 交给 LLM 生成摘要，把摘要存入 `compressed_context`，messages 只保留最近 `KEEP_RECENT` 条，再次落盘；第三步，计算压缩率并从文件读回做验证。这三步的设计动机分别是：先写后压、减少 token、以读回验证持久化的正确性。

&emsp;&emsp;下面把这段核心逻辑完整写出来。代码里 `llm` 和 `count_tokens` 在 Ch0/Ch1 已初始化，此处直接使用。

In [18]:
# === 短期记忆层的 Write：LLM 压缩摘要 + 持久化到 sessions/ ===
import json, os, time

SESSIONS_DIR = "./sessions"                       # 持久化目录
os.makedirs(SESSIONS_DIR, exist_ok=True)          # 目录不存在则自动创建

# 核心参数（消息数达到阈值时，压缩前半部分，只保留后半部分）
COMPRESS_TRIGGER = 8   # 触发压缩的消息数阈值（演示用小阈值，生产环境用 20）
KEEP_RECENT = 4        # 压缩后保留最近 4 条消息原文

# ---- 步骤 1：创建 session，写入 8 条多轮对话 ----
session_id = "write_demo"
session = {
    "title": "新项目技术选型讨论",                            # 会话标题
    "created_at": time.strftime("%Y-%m-%d %H:%M"),          # 创建时间
    "updated_at": time.strftime("%Y-%m-%d %H:%M"),          # 更新时间（初始与创建相同）
    "compressed_context": "",                               # 压缩前为空
    "messages": [
        {"role": "user",      "content": "新项目的数据库用什么?"},
        {"role": "assistant", "content": "推荐 PostgreSQL。理由:项目有大量关联查询,团队熟悉度高,配合 SQLAlchemy 2.0 的 async 支持性能优秀。"},
        {"role": "user",      "content": "缓存方案呢?"},
        {"role": "assistant", "content": "Redis,Cache-Aside 模式。热点数据 TTL=300s,写操作先更新 DB 再删缓存。"},
        {"role": "user",      "content": "部署方案定了吗?"},
        {"role": "assistant", "content": "Docker Compose 本地开发,AWS ECS 生产。CI/CD 用 GitHub Actions,自动化金丝雀发布。"},
        {"role": "user",      "content": "预算上限是多少?"},
        {"role": "assistant", "content": "初期 15 万以内。服务器 3 万/年,API 调用预留 2 万/月。"},
    ]
}

# 写入 JSON（压缩前：8 条消息，compressed_context 为空）
session_path = os.path.join(SESSIONS_DIR, f"{session_id}.json")
with open(session_path, "w", encoding="utf-8") as f:
    json.dump(session, f, ensure_ascii=False, indent=2)     # ensure_ascii=False 保留中文

print(f"=== 压缩前：{session_path} ===")
print(f"messages: {len(session['messages'])} 条（达到 COMPRESS_TRIGGER={COMPRESS_TRIGGER}）")
print(f"compressed_context: （空）")

# ---- 步骤 2：达到阈值，压缩前 4 条为摘要，只保留后 4 条 ----
if len(session["messages"]) >= COMPRESS_TRIGGER:
    early_messages  = session["messages"][:-KEEP_RECENT]    # 前 4 条：将被压缩为摘要
    recent_messages = session["messages"][-KEEP_RECENT:]    # 后 4 条：保留原文

    # 把早期对话拼成文本，交给 LLM 生成摘要
    early_text = "\n".join(f'{m["role"]}: {m["content"]}' for m in early_messages)
    compress_prompt = (
        f"请将以下对话历史压缩为一段简洁的摘要,保留所有关键技术决策和数字:\n\n"
        f"{early_text}\n\n输出要求:一段话,不超过 80 字。"
    )
    summary = llm.invoke(compress_prompt).content           # LLM 压缩调用

    # 写回 session：compressed_context 存摘要，messages 只留最近的
    session["compressed_context"] = summary                 # 摘要替换早期对话
    session["messages"]           = recent_messages         # 截断到最近 4 条
    session["updated_at"]         = time.strftime("%Y-%m-%d %H:%M")

    with open(session_path, "w", encoding="utf-8") as f:
        json.dump(session, f, ensure_ascii=False, indent=2) # 覆盖写回文件

# ---- 步骤 3：展示压缩结果 + 读回验证 ----
original_tokens = count_tokens(early_text)                  # 计算压缩前 token 数
summary_tokens  = count_tokens(summary)                     # 计算摘要 token 数

print(f"\n=== 压缩后：{session_path} ===")
print(f"compressed_context（{summary_tokens} tokens，原始 {original_tokens} tokens，"
      f"压缩率 {(1 - summary_tokens/original_tokens)*100:.0f}%）：")
print(f"  {summary}")
print(f"messages: {len(session['messages'])} 条（只保留最近 {KEEP_RECENT} 条）")
for m in session["messages"]:
    print(f"  [{m['role']}] {m['content'][:50]}")

# 从文件读回验证（确认 JSON 落盘无损）
print(f"\n=== 验证：从文件读回 ===")
with open(session_path, "r", encoding="utf-8") as f:
    saved = json.load(f)                                    # 读回 JSON 文件
print(f"compressed_context 长度: {len(saved['compressed_context'])} 字符")
print(f"messages 数量: {len(saved['messages'])} 条")

=== 压缩前：./sessions/write_demo.json ===
messages: 8 条（达到 COMPRESS_TRIGGER=8）
compressed_context: （空）

=== 压缩后：./sessions/write_demo.json ===
compressed_context（46 tokens，原始 79 tokens，压缩率 42%）：
  项目选用 PostgreSQL 数据库以支持复杂关联查询和异步性能，并采用 Redis 缓存配合 Cache-Aside 模式，热点数据 TTL 设为 300 秒，写操作先更新数据库再删除缓存。
messages: 4 条（只保留最近 4 条）
  [user] 部署方案定了吗?
  [assistant] Docker Compose 本地开发,AWS ECS 生产。CI/CD 用 GitHub Acti
  [user] 预算上限是多少?
  [assistant] 初期 15 万以内。服务器 3 万/年,API 调用预留 2 万/月。

=== 验证：从文件读回 ===
compressed_context 长度: 97 字符
messages 数量: 4 条


&emsp;&emsp;运行这段代码，你会看到：第一次写入时 `compressed_context` 为空，`messages` 有 8 条；触发压缩后 `compressed_context` 包含 LLM 生成的摘要，`messages` 只剩 4 条最近的；读回 JSON 文件验证字段无损。这个过程把"原本塞在内存里的历史"真正写到了磁盘，就算进程重启或切换到新会话，只要 `session_id` 相同就能 restore。

#### 2.1.2 最小 LangChain Middleware 封装

&emsp;&emsp;2.1.1 的代码是一次性执行的脚本。要让它在每轮 Agent 响应后自动跑，我们需要把这段逻辑封装成一个继承 `AgentMiddleware` 的类，并实现 `after_model` hook。`after_model` 在 LLM 每次响应完成后由 LangChain 框架自动调用，`state` 参数是 `AgentState`（`dict` 子类），通过 `state["messages"]` 取得当前消息列表。但要注意：**封装不是为了复杂，而是为了复用**——内层逻辑必须和 2.1.1 的参考代码 1:1 对齐，只是多了一个 class 壳和正确的 hook 接入点。

&emsp;&emsp;下面是 `SessionWriteMiddleware` 的最小封装。和 2.1.1 相比，唯一增加的是：把固定的 `session_id` 改为构造时传入的参数，以及累积摘要机制（新摘要追加到旧的后面，避免信息丢失）。Layer 1 是**纯副作用层**：读 `state["messages"]`，在内部维护压缩摘要并落盘到 JSON，不修改 state，返回 `None`。in-memory 的 messages 截断由 Ch1 的 Compression 层负责。

In [19]:
import json, os, time
from langchain.agents.middleware import AgentMiddleware, AgentState
from langgraph.runtime import Runtime

class SessionWriteMiddleware(AgentMiddleware):
    """Layer 1：会话级短期记忆 — 文件持久化 + 压缩前半部分
    
    继承 AgentMiddleware，实现 after_model hook：LLM 调用完成后，
    从 state["messages"] 读取消息列表，执行可选的 LLM 压缩，
    并将摘要和最近消息持久化到 sessions/{session_id}.json。
    
    这是纯副作用层（return None），不修改 state 中的 messages。
    in-memory messages 截断请使用 Ch1 的 SummarizationMiddleware。
    
    Args:
        session_id: 当前会话标识，决定 JSON 文件名
        llm: 用于压缩早期消息的 LLM 实例（必须支持 .invoke）
        compress_trigger: 触发压缩的消息数阈值，默认 8（对齐 2.1.1 COMPRESS_TRIGGER）
        keep_recent: 压缩后保留的最近消息数，默认 4（对齐 2.1.1 KEEP_RECENT）
        sessions_dir: 持久化目录，默认 "./sessions"
    """
    def __init__(self, session_id, llm, compress_trigger=8, keep_recent=4, sessions_dir="./sessions"):
        super().__init__()                        # 必须调用基类 __init__
        self.session_id       = session_id        # 会话唯一标识，对应 JSON 文件名
        self.llm              = llm               # LLM 实例，用于压缩摘要
        self.compress_trigger = compress_trigger  # 与 2.1.1 COMPRESS_TRIGGER 对齐
        self.keep_recent      = keep_recent        # 与 2.1.1 KEEP_RECENT 对齐
        self.sessions_dir     = sessions_dir      # 持久化目录
        self._last_summary    = ""                # 内部维护的累积摘要（跨轮累加）
        os.makedirs(sessions_dir, exist_ok=True)  # 确保目录存在

    def after_model(self, state: AgentState, runtime: Runtime) -> dict | None:
        """LLM 响应完成后触发：读 messages，可选压缩，落盘到 JSON。
        
        Args:
            state: AgentState（dict 子类），通过 state["messages"] 取消息列表
            runtime: LangGraph Runtime，此 middleware 未使用 runtime.store
        Returns:
            None — 纯副作用，不修改 state（压缩后的摘要存在实例变量，不写回 state）
        """
        # state 是 AgentState（dict 子类），直接索引取 messages
        messages = state["messages"]
        compressed_ctx = self._last_summary      # 使用实例变量跨轮累积摘要

        # 用 msg.model_dump() 转 dict（自动带 type/content 字段，对齐 2.1.1 结构）
        msg_dicts = [
            {"role": "user" if m.type == "human" else "assistant", "content": m.content}
            for m in messages
        ]

        # 达到阈值触发压缩（严格对齐 2.1.1 的 early/recent 切分逻辑）
        if len(msg_dicts) >= self.compress_trigger:
            early  = msg_dicts[:-self.keep_recent]           # 前半部分：压缩为摘要
            recent = msg_dicts[-self.keep_recent:]           # 后半部分：保留原文
            early_text = "\n".join(f'{m["role"]}: {m["content"]}' for m in early)
            prompt  = (
                f"请将以下对话压缩为一段摘要，保留关键决策和数字："
                f"\n\n{early_text}\n\n输出不超过 80 字。"
            )
            summary = self.llm.invoke(prompt).content        # LLM 压缩，对齐 2.1.1

            # 累积摘要：新摘要追加到旧的后面，避免跨轮压缩时信息丢失
            compressed_ctx = (compressed_ctx + " " + summary).strip() if compressed_ctx else summary
            self._last_summary = compressed_ctx              # 更新实例变量供下轮使用
            msg_dicts = recent                               # 落盘时只存最近 N 条

        # 持久化到 sessions/{session_id}.json（字段结构与 2.1.1 完全一致）
        session = {
            "title":              state.get("title", f"Session {self.session_id}"),
            "created_at":         state.get("created_at", time.strftime("%Y-%m-%d %H:%M")),
            "updated_at":         time.strftime("%Y-%m-%d %H:%M"),  # 每次写入更新时间戳
            "compressed_context": compressed_ctx,
            "messages":           msg_dicts,
        }
        # 文件名格式：sessions/{session_id}.json，与 2.1.1 的 session_path 逻辑一致
        path = os.path.join(self.sessions_dir, f"{self.session_id}.json")
        with open(path, "w", encoding="utf-8") as f:
            # indent=2 保持可读性，ensure_ascii=False 保留中文字符
            json.dump(session, f, ensure_ascii=False, indent=2)

        return None  # 纯副作用层：不修改 state，返回 None

&emsp;&emsp;对比 2.1.1 的参考代码，封装后的 middleware 做了三件事：第一，继承 `AgentMiddleware` 并实现 `after_model` hook，接入 LangChain 框架的调用链；第二，累积摘要改为实例变量 `self._last_summary` 跨轮维护，`after_model` 返回 `None` 不修改 state（in-memory 截断由 Ch1 Compression 层负责）；第三，落盘结构和参考代码完全一致（`title/created_at/updated_at/compressed_context/messages` 五个字段），可以直接用 `json.load` 读回。**核心压缩逻辑一字未改**。

#### 2.1.3 独立测试：触发压缩 + 读回验证

&emsp;&emsp;封装好中间件后，我们用最小测试验证两件事：① 消息数达到 8 条时触发压缩，JSON 文件中的 `messages` 被截断到 4 条，`compressed_context` 有内容；② 从 `sessions/` 下读回 JSON 文件，字段结构正确。

&emsp;&emsp;下面这段测试代码构造恰好 8 条消息的伪 `state` dict，调用 `after_model` 一次，然后从文件读回验证。`after_model` 返回 `None`（纯副作用），所以验证点在磁盘文件而非返回值。

In [20]:
from types import SimpleNamespace
from langchain_core.messages import HumanMessage, AIMessage

# 构造 8 条消息的伪 state（AgentState 是 dict 子类，dict 足够）
test_messages = [
    HumanMessage(content="新项目的数据库用什么?"),
    AIMessage(content="推荐 PostgreSQL，理由是关联查询多 + 团队熟悉 + async 支持好。"),
    HumanMessage(content="缓存方案呢?"),
    AIMessage(content="Redis + Cache-Aside，热点数据 TTL=300s。"),
    HumanMessage(content="部署方案定了吗?"),
    AIMessage(content="Docker Compose 本地 + AWS ECS 生产，GitHub Actions CI/CD。"),
    HumanMessage(content="预算上限是多少?"),
    AIMessage(content="初期 15 万以内，服务器 3 万/年 + API 2 万/月。"),
]
# state 是 dict，after_model 通过 state["messages"] 取消息
state = {"messages": test_messages}

# 构造伪 Runtime（after_model 未使用 runtime.store，SimpleNamespace 足够）
fake_runtime = SimpleNamespace(store=None, context=None, stream_writer=None)

# 创建 middleware 并执行一次（llm 在前面章节已初始化）
layer1 = SessionWriteMiddleware(session_id="test_alice", llm=llm)
result = layer1.after_model(state, fake_runtime)   # 触发压缩 + 落盘

# after_model 是纯副作用，返回 None
assert result is None, f"after_model 应返回 None，实际：{result}"
print(f"after_model 返回值：{result}（None = 纯副作用，不修改 state）")

# 压缩摘要存在实例变量里（跨轮累积用）
print(f"内部摘要 _last_summary 是否非空: {bool(layer1._last_summary)}")  # 期望：True（LLM 正常时）
print(f"摘要内容（前 80 字）: {layer1._last_summary[:80]}")

# 读回 JSON 文件验证（持久化结果）
with open("./sessions/test_alice.json", "r", encoding="utf-8") as f:
    saved = json.load(f)                                           # 读回落盘文件
print(f"\n[PASS] 文件字段: {list(saved.keys())}")                  # 期望：五个字段
print(f"[PASS] 保存的 messages 数量: {len(saved['messages'])}")    # 期望：4（压缩后 keep_recent）
print(f"[PASS] 保存的 compressed_context 长度: {len(saved['compressed_context'])} 字符")

after_model 返回值：None（None = 纯副作用，不修改 state）
内部摘要 _last_summary 是否非空: True
摘要内容（前 80 字）: 数据库选用PostgreSQL（关联查询多、团队熟、async支持好）；缓存方案为Redis + Cache-Aside，热点数据TTL设为300秒。

[PASS] 文件字段: ['title', 'created_at', 'updated_at', 'compressed_context', 'messages']
[PASS] 保存的 messages 数量: 4
[PASS] 保存的 compressed_context 长度: 75 字符


&emsp;&emsp;预期看到：JSON 文件中 `messages` 数量从 8 减到 4，`compressed_context` 非空（LLM 生成的约 80 字摘要），文件含五个字段，读回后数据一致。`after_model` 返回 `None` 是设计行为——state 未被改动，压缩摘要存在 `layer1._last_summary` 实例变量里。如果 LLM 调用失败（API key 未配置或限流），`messages` 不会被截断，此时 `_last_summary` 仍为空——这个 fail-open 行为比 fail-close 更符合学员跟跑的友好度。

> 🔥 **踩坑预警**：参考代码里 `count_tokens` 用于计算压缩率，但在本课件的 Middleware 封装里被省略了。原因是 middleware 每轮都会触发，如果每次都计算 tokens 会拖慢对话。生产环境建议把压缩率统计放在单独的 metrics 模块里异步收集，而不是塞进主链路。

---

### 2.2 Mem0WriteMiddleware：跨会话长期记忆

&emsp;&emsp;Layer 1 的 session store 解决了同一用户在同一天内跨会话恢复上下文的问题，但它本质上是一个快照——存的是 messages 列表，而不是"从这段对话里萃取出的有价值信息"。当用户和 Agent 连续交互 30 天，session store 里会堆积几十个快照，真正重要的那句"我们决定采用 Python + LangChain + DeepSeek 做技术栈"被淹没在大量无关对话里，根本无法被后续会话高效检索。

&emsp;&emsp;Layer 2 要解决的是另一个量级的问题：**把用户在对话中做出的关键决策、确认事项、技术选型等高价值内容写入向量数据库，让 Agent 在未来任意一次会话里都能语义检索到这些"长期记忆"**。我们选择 `Mem0 + Milvus` 的组合——`Mem0` 提供自动摘要与 embedding 写入的高层 API，`Milvus` 作为后端向量库负责持久化和 ANN 检索。这个组合也是本课 Ch2 到 Ch5 各章共用的基础设施，在这里统一搭建好后续不会重复。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260417140023414.svg" width=70%></div>

#### 2.2.1 Milvus standalone 环境准备

&emsp;&emsp;在写任何代码之前，我们需要先确认 Milvus 服务已就绪。下面的预警块列出了两项必须满足的前置条件，缺少任意一项都会导致后续代码报错。

> ⚠️ **运行前预警（必读）**：本节及后续所有涉及 `Mem0WriteMiddleware` 的代码依赖两项外部服务，**不预先启动会直接导致静默网络错误或 embedder 401**：
> 1. **Milvus standalone 运行在 `localhost:19530`**：请先在独立终端启动 Milvus 服务，否则 `Memory.from_config` 会卡在连接阶段然后报网络超时。
> 2. **`OPENAI_API_KEY` 环境变量已配置**：`Mem0` 的 `embedder` 使用 `text-embedding-3-small`，本质上调用 OpenAI（或兼容代理）的 embedding 接口，没有这把 key 会直接 401。

&emsp;&emsp;下面这个 cell 用于启动最简化的 Milvus standalone 服务（基于 Docker）。如果你本地已有其他方式运行 Milvus，可以跳过 `docker run` 那行，只跑 `curl` 探活即可。

In [47]:
# 启动 Milvus standalone（需要已安装 Docker）
# 首次运行会拉取镜像，约 500MB，耐心等待
# 默认注释保护：如已在运行，取消注释会报端口占用
# 下载milvus的docker-compose.yml文件，需要梯子
# !wget https://raw.githubusercontent.com/milvus-io/milvus/v2.5.11/deployments/docker/standalone/docker-compose.yml  

# 在保存文件的目录下执行这个命令来执行创建docker
# !docker compose up -d 

# 探活 healthz 接口，返回 200 即表示 Milvus 可用
!curl -s -o /dev/null -w "milvus health: %{http_code}\n" \
    http://localhost:9091/healthz \
    || echo "Milvus 尚未就绪，请先取消 docker run 注释并启动"

milvus health: 200


&emsp;&emsp;执行后，如果看到 `milvus health: 200`，说明 Milvus 已经正常监听 19530 端口，可以继续后续步骤。如果看到 `000` 或连接拒绝，请取消 `docker run` 那行注释，等待约 30 秒后重跑 `curl` 探活。

#### 2.2.2 决策关键词触发机制

&emsp;&emsp;Layer 2 不是把每一轮对话都写进向量库——那样会产生大量噪声，让未来的检索精度急剧下降。我们只写入用户表达了**明确决策意图**的那轮对话。判断标准是：用户消息里包含预设的决策关键词，则触发写入，否则跳过。

&emsp;&emsp;默认关键词列表覆盖了最常见的决策场景。下面这张表列出了各关键词及其对应的典型用途，方便在不同项目里按需扩展。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Layer 2 决策关键词与触发场景对照表</font></p>
<div class="center">

| 关键词 | 触发场景示例 | 典型写入内容 |
|--------|------------|-------------|
| `决定` | "我们决定用 LangChain 1.0" | 技术选型决策 |
| `确认` | "确认采用 Milvus 作为向量库" | 架构确认事项 |
| `使用` | "以后使用 DeepSeek-chat 模型" | 工具/模型选择 |
| `选择` | "选择 Python 3.11 + uv 环境" | 环境/版本选型 |
| `采用` | "采用 Middleware 模式做中间件" | 设计模式选型 |

</div>

&emsp;&emsp;这个朴素匹配方案实现简单、零额外开销，但存在明显局限：否定句（"我们**不**决定用 A"）和疑问句（"你觉得**使用** B 好吗？"）都可能误触发。在 2.2.5 的 Case 2 里，我们会专门构造反例来演示这个局限，并讨论改进方向。

#### 2.2.3 Mem0WriteMiddleware 实现

&emsp;&emsp;有了触发机制，接下来实现 `Mem0WriteMiddleware`。它继承 `AgentMiddleware`，在初始化时构建 Mem0 + Milvus 的配置，拿到一个 `Memory` 实例；在 `after_model` hook 里检测决策关键词，匹配则调用 `memory.add` 把本轮 user/assistant 对话写入向量库，写入失败则打印 WARN 后继续，不阻断主链路。返回 `None`（纯副作用，不修改 state）。

In [5]:
from dotenv import load_dotenv
import os
from mem0 import Memory
from langchain.agents.middleware import AgentMiddleware, AgentState
from langgraph.runtime import Runtime

load_dotenv()

def build_mem0_memory(collection_name: str = "agent_ch2_write") -> Memory:
    """构建绑定 Milvus 的 Mem0 Memory 实例
    
    Args:
        collection_name: Milvus 集合名称，建议按章节区分避免数据混用
    Returns:
        Memory 实例，已绑定 localhost:19530 的 Milvus
    """
    config = {
        "vector_store": {
            "provider": "milvus",
            "config": {
                "url": "http://localhost:19530",
                # 本地 standalone 不鉴权，token 传空串(mem0 BaseVectorStoreConfig 要求非 None)
                "token": "",
                # collection_name 按章节隔离，避免不同实验数据互串
                "collection_name": collection_name,
            },
        },
        "llm": {
            "provider": "deepseek",
            "config": {
                # 用于 Mem0 内部做摘要提炼的 LLM
                "model": os.getenv("DEEPSEEK_MODEL", "deepseek-chat"),
                "api_key": os.getenv("DEEPSEEK_API_KEY"),
            },
        },
        "embedder": {
            "provider": "openai",
            "config": {
                # DeepSeek 不提供 embedding，必须走 OpenAI 兼容接口
                # base_url 通过环境变量 OPENAI_BASE_URL 由 openai SDK 自动读取
                # (当前 mem0 版本 BaseEmbedderConfig 不支持直接传 base_url 字段)
                "model": "text-embedding-3-small",
                "api_key": os.getenv("OPENAI_API_KEY"),
            },
        },
    }
    return Memory.from_config(config)


class Mem0WriteMiddleware(AgentMiddleware):
    """Layer 2：跨会话长期记忆写入
    
    继承 AgentMiddleware，实现 after_model hook：LLM 调用完成后，
    检测 state["messages"] 最后两条（user + assistant）中的决策关键词，
    匹配则将本轮对话写入 Mem0（底层持久化到 Milvus 向量库）。
    
    纯副作用层：after_model 返回 None，不修改 state。
    
    Args:
        memory: 已初始化的 Mem0 Memory 实例（绑定 Milvus）
        decision_keywords: 触发写入的决策关键词列表
        user_id: Mem0 记忆空间的用户标识，相同 user_id 的记忆可跨会话检索
    """

    def __init__(
        self,
        memory: Memory,
        decision_keywords: list[str] | None = None,
        user_id: str = "default",
    ):
        super().__init__()                        # 必须调用基类 __init__
        self.memory = memory
        self.decision_keywords = decision_keywords or [
            "决定", "确认", "使用", "选择", "采用"
        ]
        self.user_id = user_id

    def after_model(self, state: AgentState, runtime: Runtime) -> dict | None:
        """LLM 响应完成后触发：检测决策关键词，命中则写入 Mem0 → Milvus。
        
        Args:
            state: AgentState（dict 子类），通过 state["messages"] 取消息列表
            runtime: LangGraph Runtime，此 middleware 未使用 runtime.store
        Returns:
            None — 纯副作用，不修改 state
        """
        # state 是 AgentState（dict 子类），直接索引取 messages
        messages = state["messages"]
        # 至少需要一轮完整对话（user + assistant）
        if len(messages) < 2:
            return None

        # 取最后两条：倒数第二条是 user 消息，最后一条是 assistant 响应
        last_user_msg = messages[-2]
        last_user_text = (
            last_user_msg.content
            if hasattr(last_user_msg, "content")
            else str(last_user_msg)
        )

        # 朴素关键词匹配（见 2.2.5 Case 2 了解其局限）
        if not any(kw in last_user_text for kw in self.decision_keywords):
            return None  # 无决策关键词，跳过写入

        last_assistant_msg = messages[-1]
        last_assistant_text = (
            last_assistant_msg.content
            if hasattr(last_assistant_msg, "content")
            else str(last_assistant_msg)
        )

        try:
            # 将本轮对话写入 Mem0 → Milvus（追加语义，不覆盖旧记忆）
            self.memory.add(
                [
                    {"role": "user", "content": last_user_text},
                    {"role": "assistant", "content": last_assistant_text},
                ],
                user_id=self.user_id,
            )
            print(
                f"[Layer2] 决策写入成功 | user_id={self.user_id} | "
                f"触发词：{[kw for kw in self.decision_keywords if kw in last_user_text]}"
            )
        except Exception as e:
            # 写入失败仅打印 WARN，不阻断主链路（Milvus 抖动不应让 Agent 崩溃）
            print(f"[WARN] Layer 2 写入失败：{e}")

        return None  # 纯副作用层，不修改 state

&emsp;&emsp;代码里有几个工程细节需要说明：`build_mem0_memory` 单独抽出来作为工厂函数，让 `Mem0WriteMiddleware` 可以接受外部传入的 `Memory` 实例——这样在测试时可以注入 mock Memory，不强依赖 Milvus；`user_id` 是 Mem0 的记忆命名空间，相同 `user_id` 的记忆可以在任意会话里被 `memory.search` 检索到，这是实现"跨会话"的核心机制；`after_model` 返回 `None` 表示不修改 state（写入 Milvus 是纯副作用）；异常捕获只打印 WARN 而不 raise，确保 Milvus 抖动不会让整个 Agent 崩溃。

> 🔥 **踩坑预警**：`embedder` 与 `llm` 凭证必须分开配置，不能共用同一套。`Mem0` 的 `llm` 走 DeepSeek 做摘要提炼，但 DeepSeek 目前不提供 embedding 接口；`embedder` 必须单独配置 OpenAI（或兼容 OpenAI API 的代理），否则 `Memory.from_config` 会在构建 embedder 时抛出 `provider not supported` 错误。两套 key 来自不同的服务商，配置时务必区分 `DEEPSEEK_API_KEY` 和 `OPENAI_API_KEY`。

#### 2.2.4 SmartMem0WriteMiddleware 的四项升级

&emsp;&emsp;2.2.4 的 `Mem0WriteMiddleware` 是一条可以跑通的基线，但朴素关键词匹配有一个根本硬伤——它完全无法区分"我们**决定**用 Postgres"和"你觉得**决定**权归谁合适？"这两种语义。如果把它直接丢到生产环境，还会叠加出另外三个问题：每一轮 LLM 调用后都检测写入，产生大量冗余记忆稀释检索精度；完全没有和主 Agent 自己的 `save_memory` 工具做互斥，双写时同一条决策会在 Milvus 里出现两份；写入失败时连重试机会都没有，Milvus 一次抖动就永久丢数据。

&emsp;&emsp;为了把它从"教学基线"升级成"生产级组件"，本节我们参考 mini-OpenClaw 项目 `backend/graph/smart_extractor.py` 里已落地的 `SmartExtractor` 生产实现，把它的三件套机制——节流、主 Agent 互斥、失败重试——再加上一项本次新增的 **LLM 结构化萃取**，一次性合入一个新中间件 `SmartMem0WriteMiddleware`。四项升级的核心思路可以用下面这张对照表总览。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Mem0WriteMiddleware 朴素版 vs SmartMem0WriteMiddleware 智能版</font></p>
<div class="center">

| 维度 | 朴素版（2.2.4）| 智能版（2.2.5）|
|------|--------------|---------------|
| 触发判定 | 朴素关键词 `["决定","确认",...]` 字串匹配 | LLM 结构化萃取，基于 `ExtractionResult` 语义判定 |
| 写入频率 | 每次 `after_model` 都检测并写入 | 每 `throttle_every=3` 轮才做一次萃取 + 写入 |
| 记忆粒度 | 整轮 user+assistant 原文送进 `memory.add` | LLM 自动拆成若干条**原子记忆**，每条独立 `add` |
| hook 位置 | `after_model`（每次 LLM 调用后） | `after_agent`（一整次 invoke 结束后） |
| 主 Agent 互斥 | 无 | 自动扫 `messages` 最末 `AIMessage.tool_calls`，命中则跳过 |
| 失败重试 | 写入失败仅打印 WARN | 分三档：萃取失败保留 buffer、无价值清空、mem0 失败 `count=N-1` 强制下轮重试 |
| 多 session | 不支持 | `per-session` 分桶 + LRU 淘汰，对标 SmartExtractor 生产实现 |

</div>

&emsp;&emsp;对照表最下面一行的 `per-session` 很关键——在 Notebook 里我们默认所有对话共用 `session_id="default"`，但在 FastAPI 多用户生产环境里，每个 Web 会话都有自己独立的节流状态，不能混到一起。`SmartMem0WriteMiddleware` 的构造函数接受一个可选 `session_id_extractor` 回调，Notebook 场景默认返回 `"default"`，生产环境传入一个从请求上下文里提取 `session_id` 的函数即可无缝切换。

##### 2.2.4.1 结构化萃取契约：ExtractionResult

&emsp;&emsp;把"这轮对话里有没有值得写入的内容"这个判断交给 LLM，最稳的姿势不是让它返回自然语言，而是让它返回一个 Pydantic 约束的结构化对象。下面的 `ExtractionResult` 就是萃取器的输出契约——三个字段分别承载"是否要写"、"要写哪些条"、"LLM 自述判定理由"。

In [23]:
from pydantic import BaseModel, Field


class ExtractionResult(BaseModel):
    """LLM 萃取器的结构化输出契约。

    Attributes:
        has_memory: 这段对话是否存在值得写入长期记忆的内容。
        memories: 若干条原子化记忆（每条一个独立的决策/事实/偏好）。
        reason: LLM 对本次判定的简要说明（debug + 可观测性）。
    """

    has_memory: bool = Field(description="是否存在值得写入的记忆内容")
    memories: list[str] = Field(
        default_factory=list,
        description="原子化记忆列表，每条一个决策/事实/偏好",
    )
    reason: str = Field(description="本次判定的简要理由")

&emsp;&emsp;后续我们会用 `llm.with_structured_output(ExtractionResult)` 把这个 schema 绑定到萃取 LLM 上——LangChain 会自动把 schema 注入 prompt，强制模型按格式返回，省去手写 JSON parser 的容错地狱。`has_memory` 作为硬开关决定走不走 `memory.add`，`memories` 是一个列表，刻意设计成"一条原子记忆一个列表元素"——这样每条记忆都会被 `mem0` 独立 embed 写入，检索时命中粒度更细；`reason` 字段纯粹为可观测性存在，生产环境可以把它打进日志做 A/B 评估。

##### 2.2.4.2 节流状态容器：_SessionState

&emsp;&emsp;节流机制要的不是全局计数，而是 per-session 计数——每个 session 自己管自己的 `turns_since_last` 和 `message_buffer`。用一个极简的 `dataclass` 承载状态就够了。

In [24]:
from dataclasses import dataclass, field


@dataclass
class _SessionState:
    """单 session 的节流状态（对标 SmartExtractor._SessionState）。

    Attributes:
        turns_since_last: 距离上一次 mem0 写入的轮数计数。
        message_buffer: 累积的本 session 内未写入 mem0 的 user+assistant 消息。
    """

    turns_since_last: int = 0
    message_buffer: list = field(default_factory=list)

&emsp;&emsp;字段只有两个：`turns_since_last` 是计数器，每轮 `after_agent` 自增 1，达到 `throttle_every` 就触发萃取；`message_buffer` 是消息缓冲，用来累积连续若干轮的对话交给 LLM 一次性萃取。每个 session 都有独立的一份 `_SessionState`，在 `SmartMem0WriteMiddleware` 里用一个 `dict[str, _SessionState]` 做分桶容器，键就是 session_id。

##### 2.2.4.3 SmartMem0WriteMiddleware 主类

&emsp;&emsp;有了契约和状态容器，接下来就是主类本身。它仍然继承 `AgentMiddleware`，但 hook 从 `after_model` 迁到了 `after_agent`——原因很重要：`after_model` 在多步 tool-calling 场景下每次 LLM 调用都会触发，一次用户请求可能跑 3-5 次 LLM，计数会跟"用户轮数"彻底脱钩；`after_agent` 对应一整个 `agent.invoke()` 生命周期，天然和"用户轮数"一一对应，节流语义立刻清晰。

In [26]:
from langchain.agents.middleware import AgentMiddleware


class SmartMem0WriteMiddleware(AgentMiddleware):
    """智能长期记忆写入中间件：节流 + LLM 萃取 + 主 Agent 互斥 + 失败重试。

    相比 2.2.4 的朴素 Mem0WriteMiddleware，四个升级：
      1. 节流（throttle_every）：每 N 轮 after_agent 才触发一次 LLM 萃取
      2. LLM 结构化萃取：用 Pydantic schema 强制产出原子记忆列表，替代朴素关键词
      3. 主 Agent 互斥：扫 state.messages 里是否有 mutex_tool_names 的 tool_calls，命中则跳过
      4. 失败重试：分三档——萃取失败保留 buffer、判定无价值清空 buffer、mem0 写入失败缓冲重试

    per-session 分桶 + LRU 淘汰对标 mini-OpenClaw SmartExtractor 生产实现。

    Args:
        memory: 已初始化的 mem0 Memory 实例（绑定 Milvus）。
        extractor_llm: 执行结构化萃取的 LLM（通常复用主 LLM，可独立换便宜模型）。
        throttle_every: 节流轮数，默认 3 轮才触发一次萃取。
        mutex_tool_names: 主 Agent 互斥 tool 名单，默认 ["save_memory"]。
        user_id: mem0 记忆命名空间。
        session_id_extractor: 从 AgentState 提取 session_id 的函数；None 时固定返回 "default"。
        max_sessions: session LRU 上限，防内存泄漏，默认 100。
    """

    def __init__(
        self,
        memory,
        extractor_llm,
        throttle_every: int = 3,
        mutex_tool_names: list[str] | None = None,
        user_id: str = "default",
        session_id_extractor=None,
        max_sessions: int = 100,
    ) -> None:
        super().__init__()
        self.memory = memory
        self.throttle_every = throttle_every
        self.mutex_tool_names = mutex_tool_names or ["save_memory"]
        self.user_id = user_id
        self._session_id_extractor = session_id_extractor or (lambda state: "default")
        self._max_sessions = max_sessions
        self._sessions: dict[str, _SessionState] = {}
        # with_structured_output 强制 LLM 产出 ExtractionResult schema
        self._extractor = extractor_llm.with_structured_output(ExtractionResult)

    def _get_state(self, session_id: str) -> _SessionState:
        """获取/创建 session 状态，超过 max_sessions 时 LRU 淘汰最老 session。"""
        if session_id not in self._sessions:
            if len(self._sessions) >= self._max_sessions:
                # 字典插入顺序即插入时间序，pop 第一个键等于淘汰最老 session
                oldest = next(iter(self._sessions))
                del self._sessions[oldest]
            self._sessions[session_id] = _SessionState()
        return self._sessions[session_id]

    def _collect_last_turn(self, messages: list) -> list[dict]:
        """从 state.messages 里摘最后一轮 user + assistant 文本（跳过中间 ToolMessage）。"""
        last_user = None
        last_assistant = None
        for m in reversed(messages):
            kind = getattr(m, "type", None) or getattr(m, "role", None)
            # LangChain AIMessage.type == "ai"，HumanMessage.type == "human"
            if last_assistant is None and kind == "ai":
                content = getattr(m, "content", "") or ""
                # 跳过只含 tool_calls、没有文本 content 的中间 AIMessage
                if isinstance(content, str) and content.strip():
                    last_assistant = content
            elif last_user is None and kind == "human":
                content = getattr(m, "content", "") or ""
                last_user = content if isinstance(content, str) else str(content)
            if last_user and last_assistant:
                break
        out = []
        if last_user:
            out.append({"role": "user", "content": last_user})
        if last_assistant:
            out.append({"role": "assistant", "content": last_assistant})
        return out

    def _agent_wrote_via_tool(self, messages: list) -> bool:
        """只扫【最新】那条 AIMessage 的 tool_calls，判断主 Agent 是否本轮主动写入。

        注意：必须只看最后一条 AIMessage，不能扫全量 messages。否则历史轮次里
        的一次 save_memory 调用会让后续每一轮都被错误地判定为互斥跳过。
        """
        for m in reversed(messages):
            kind = getattr(m, "type", None) or getattr(m, "role", None)
            if kind != "ai":
                continue
            tcs = getattr(m, "tool_calls", None) or []
            for tc in tcs:
                name = tc.get("name") if isinstance(tc, dict) else getattr(tc, "name", "")
                if name in self.mutex_tool_names:
                    return True
            return False  # 最新 AIMessage 无匹配 tool_call，立即返回
        return False

    def _build_extraction_prompt(self, buffer: list[dict]) -> str:
        """构造喂给萃取 LLM 的 prompt（严格要求返回 ExtractionResult）。"""
        dialogue = "\n".join(f"{m['role']}: {m['content']}" for m in buffer)
        return (
            "以下是最近若干轮用户与 Agent 的对话。请严格按 ExtractionResult 结构返回："
            "只有当对话中出现用户明确的【决策】【偏好】【技术选型】【重要事实陈述】时，"
            "才设 has_memory=True 并把每一条独立的记忆拆成 memories 列表的一个元素；"
            "寒暄、提问、无结论的讨论都应 has_memory=False。reason 字段用一句话说明判定依据。\n\n"
            f"对话内容：\n{dialogue}"
        )

    def after_agent(self, state, runtime) -> dict | None:
        """主 Agent 本轮结束时触发：一整个 invoke 算一次（非每次 after_model）。

        流程：收集最后一轮 → 互斥检测 → 节流检测 → LLM 萃取 → mem0 写入 →
        失败分档处理。返回 None（纯副作用，不修改 state）。
        """
        messages = state.get("messages", []) if isinstance(state, dict) else state["messages"]
        session_id = self._session_id_extractor(state)
        st = self._get_state(session_id)

        # 1. 收集本轮对话进缓冲
        turn = self._collect_last_turn(messages)
        if not turn:
            return None
        st.message_buffer.extend(turn)
        st.turns_since_last += 1

        # 2. 互斥：主 Agent 已通过 save_memory tool 写入 → 跳过
        if self._agent_wrote_via_tool(messages):
            print(f"[SmartMem0] session={session_id} 跳过——主 Agent 已通过 tool 写入记忆")
            st.message_buffer.clear()
            st.turns_since_last = 0
            return None

        # 3. 节流：未达 throttle_every 则缓冲等待
        if st.turns_since_last < self.throttle_every:
            print(f"[SmartMem0] session={session_id} 节流中 {st.turns_since_last}/{self.throttle_every}")
            return None

        # 4. LLM 结构化萃取
        buffer_snapshot = list(st.message_buffer)
        print(f"[SmartMem0] session={session_id} 触发 LLM 萃取，缓冲 {len(buffer_snapshot)} 条")
        try:
            result: ExtractionResult = self._extractor.invoke(
                self._build_extraction_prompt(buffer_snapshot)
            )
        except Exception as e:
            # 萃取失败：保留 buffer，下轮再试（不改 turns_since_last 使其不膨胀）
            print(f"[SmartMem0] 萃取失败 {e}，buffer 保留待下轮重试")
            return None

        if not result.has_memory:
            # 判定本段无价值：清空 buffer + 重置计数
            print(f"[SmartMem0] 萃取判定无价值 → 清空缓冲 | reason: {result.reason}")
            st.message_buffer.clear()
            st.turns_since_last = 0
            return None

        # 5. mem0 写入：每条原子记忆单独 add，更利于检索粒度
        try:
            for mem in result.memories:
                self.memory.add(
                    [{"role": "user", "content": mem}],
                    user_id=self.user_id,
                )
            print(
                f"[SmartMem0] session={session_id} 写入 {len(result.memories)} 条原子记忆 "
                f"| reason: {result.reason}"
            )
            st.message_buffer.clear()
            st.turns_since_last = 0
        except Exception as e:
            # mem0 写入失败：缓冲保留，count=N-1 强制下轮立即重试
            print(f"[SmartMem0] mem0.add 失败 {e}，buffer 保留，下轮立即重试")
            st.turns_since_last = self.throttle_every - 1

        return None

&emsp;&emsp;这段代码是本节的主体，里面有几个值得特别指出的设计点：第一，`_agent_wrote_via_tool` 只看 `messages` 最末那一条 `AIMessage` 的 `tool_calls`——**这是本节最容易踩的坑**，如果扫全量 messages，历史轮次里的一次 `save_memory` 调用会让后续所有轮次都被错误判定为互斥跳过，节流 + 萃取永远跑不起来（2.2.5.4 的演示验证了这一点）；第二，`_collect_last_turn` 特意跳过只含 `tool_calls` 而没有文本 `content` 的中间 `AIMessage`，避免把 tool 调用请求当成 assistant 回答塞进 buffer；第三，失败重试分了三档，最关键的一档是 mem0 写入失败时把 `turns_since_last` 强制设为 `throttle_every - 1`，确保下一轮 `after_agent` 一进来就立即重试而不是再等 2-3 轮；第四，`_get_state` 里的 LRU 淘汰利用了 Python dict 插入顺序保证，`next(iter(self._sessions))` 永远指向最老的那个 session，一行代码就实现了生产级的内存泄漏防护。

> 🔥 **踩坑预警**：互斥检测**必须**只看最新一条 `AIMessage` 的 `tool_calls`。我们在初版实现里扫过全量 messages，结果 Turn 4 主 Agent 调一次 `save_memory` 之后，Turn 5 / Turn 6 / Turn 7 也全被误判为互斥，节流计数永远回不到 0，后续所有萃取都失效。排查方法：在 `_agent_wrote_via_tool` 命中时打印一下命中的那条 AIMessage 的 index，如果发现指向的是一条非常早的历史消息，说明扫错了范围。

##### 2.2.4.4 真跑 7 轮确定性演示

&emsp;&emsp;理论讲完，接下来要看这套机制在一条真实对话轨迹上的行为。下面的演示没有走 `create_agent` 闭环——那样会引入 LLM tool 决策的不确定性，无法稳定复现节流 / 互斥 / 萃取各自的时刻。我们直接手动累积 `messages` 列表，每轮调一次 `middleware.after_agent()`，这样可以精确控制每一轮的输入，断言每一步的状态变化。

In [27]:
import os
from dotenv import load_dotenv
from mem0 import Memory
from langchain_deepseek import ChatDeepSeek
from langchain_core.messages import HumanMessage, AIMessage

load_dotenv(override=True)

# 复用 2.2.4 的 build_mem0_memory，collection 起个独立名字防止数据混用
memory = build_mem0_memory(collection_name="agent_ctx_ch2")
# 萃取器用 DeepSeek-chat，temperature=0 提高结构化输出稳定性
extractor_llm = ChatDeepSeek(
    model="deepseek-chat",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    temperature=0,
)

middleware = SmartMem0WriteMiddleware(
    memory=memory,
    extractor_llm=extractor_llm,
    throttle_every=3,
    mutex_tool_names=["save_memory"],
    user_id="demo_user",
)

# 累积的 messages 列表——每轮追加 user + assistant 两条，模拟真实 AgentState
messages: list = []


def run_turn(turn_no, user_text, assistant_text, tool_calls=None):
    """追加一轮 user + assistant 到 messages，然后触发 middleware.after_agent。"""
    messages.append(HumanMessage(content=user_text))
    if tool_calls:
        # 模拟主 Agent 主动调 save_memory tool 的 AIMessage
        ai_msg = AIMessage(content=assistant_text, tool_calls=tool_calls)
    else:
        ai_msg = AIMessage(content=assistant_text)
    messages.append(ai_msg)
    print(f"\n── Turn {turn_no} ──  user: {user_text}")
    middleware.after_agent({"messages": messages}, None)


# Turn 1：寒暄——期望节流中 1/3
run_turn(1, "你好介绍下自己",
         "你好，我是一个 AI 技术助理，可以帮你做架构设计、技术选型和代码实现。")

# Turn 2：第一条技术决策——期望节流中 2/3
run_turn(2, "我们决定采用 Milvus 作为向量库",
         "好的，Milvus 适合大规模向量检索，部署上建议 standalone 先起步。")

# Turn 3：第二条技术决策——期望触发 LLM 萃取 + mem0 写入
run_turn(3, "后端选 FastAPI + Python 3.11",
         "FastAPI + Python 3.11 是不错的组合，async 生态成熟，类型标注也更完善。")

# Turn 4：主 Agent 主动调 save_memory tool——期望互斥跳过
run_turn(4, "帮我把刚才的技术栈存到长期记忆",
         "好的，我通过 save_memory 工具为你写入。",
         tool_calls=[{
             "name": "save_memory",
             "args": {"content": "Milvus + FastAPI + Python 3.11"},
             "id": "call_demo_001",
         }])

# Turn 5：无关寒暄——期望节流中 1/3（Turn 4 已把计数清零）
run_turn(5, "今天天气怎么样？",
         "作为 AI 助理我没有实时天气数据，但可以帮你推荐天气 API 接入。")

# Turn 6：新的技术决策 SQLAlchemy——期望节流中 2/3
run_turn(6, "我们 ORM 用 SQLAlchemy 2.0",
         "SQLAlchemy 2.0 的 Typed ORM 风格配 FastAPI 的 type hints 非常顺手。")

# Turn 7：第三轮累积——期望触发第二次 LLM 萃取
run_turn(7, "明天还是用 SQLAlchemy 不变",
         "了解，SQLAlchemy 2.0 作为最终 ORM 选型确认。")

# 最终检索：验证 Milvus 里已经有至少两批原子记忆
final = memory.search(query="技术栈 ORM 向量库", user_id="demo_user", limit=10)
items = final if isinstance(final, list) else final.get("results", [])
print(f"\n[最终] mem0 中共召回 {len(items)} 条原子记忆")
for it in items:
    print("  -", it.get("memory", str(it)))


── Turn 1 ──  user: 你好介绍下自己
[SmartMem0] session=default 节流中 1/3

── Turn 2 ──  user: 我们决定采用 Milvus 作为向量库
[SmartMem0] session=default 节流中 2/3

── Turn 3 ──  user: 后端选 FastAPI + Python 3.11
[SmartMem0] session=default 触发 LLM 萃取，缓冲 6 条
[SmartMem0] session=default 写入 2 条原子记忆 | reason: 对话中包含用户明确的技术选型决策

── Turn 4 ──  user: 帮我把刚才的技术栈存到长期记忆
[SmartMem0] session=default 跳过——主 Agent 已通过 tool 写入记忆

── Turn 5 ──  user: 今天天气怎么样？
[SmartMem0] session=default 节流中 1/3

── Turn 6 ──  user: 我们 ORM 用 SQLAlchemy 2.0
[SmartMem0] session=default 节流中 2/3

── Turn 7 ──  user: 明天还是用 SQLAlchemy 不变
[SmartMem0] session=default 触发 LLM 萃取，缓冲 6 条
[SmartMem0] session=default 写入 2 条原子记忆 | reason: 对话中包含用户明确的技术选型决策

[最终] mem0 中共召回 3 条原子记忆
  - ORM 使用 SQLAlchemy 2.0
  - 采用 Milvus 作为向量库
  - 后端选 FastAPI + Python 3.11


&emsp;&emsp;运行这段代码时，请注意观察终端输出的节拍。Turn 1 和 Turn 2 会打印 `节流中 1/3` 和 `节流中 2/3`——此时 `message_buffer` 已经累积了 4 条消息但一条也没写进 Milvus；Turn 3 会打印 `触发 LLM 萃取，缓冲 6 条` 然后紧跟 `写入 N 条原子记忆`（N 通常是 2，LLM 会把"Milvus 向量库"和"FastAPI + Python 3.11"拆成两条独立的记忆）；Turn 4 会打印 `跳过——主 Agent 已通过 tool 写入记忆`，同时 buffer 清零、计数归 0；Turn 5 重新开始节流 1/3，Turn 6 变成 2/3，Turn 7 再次触发萃取把 SQLAlchemy 决策写进去。最终 `mem0.search` 会召回前后两批原子记忆共 3-5 条（具体数量由 LLM 萃取粒度决定）。<font color=red>这正是 LLM 结构化萃取的核心优势：它不关心用户句子里有没有"决定""采用"这类关键词，只关心"这段对话里有没有可以独立存在的事实或决策"。</font>

> ⚠️ **常见误区**：`throttle_every=3` 不是"每 3 轮必写一次"，而是"累积 3 轮后让 LLM 判断一次是否值得写"。如果 LLM 判定 `has_memory=False`（比如 3 轮全是无结论的技术讨论），buffer 会被清空但不会产生任何 `memory.add` 调用——这恰好是我们想要的行为：节流既控制成本也控制记忆密度。如果你发现 mem0 里积累的记忆过少，先排查是不是 LLM 把本该写入的决策误判成了 `has_memory=False`，而不是直接去调小 `throttle_every`。

##### 2.2.4.5 什么时候选朴素版，什么时候选智能版

&emsp;&emsp;两代中间件并存是刻意的设计——朴素版教会你"如何用最少的代码把 Middleware 机制跑起来"，智能版教会你"如何把一个教学原型升级成生产件"。选型的边界可以用一句话概括：**只要对话不会反复触达长期记忆（比如一次性脚本、短会话工具），朴素版就够了**；一旦进入"长时间多轮对话 + 多个记忆触达点 + 需要检索质量"的场景，就应该切换到智能版。关键的拐点指标是**对话频率**和**记忆检索质量**——前者决定节流的价值，后者决定 LLM 萃取的价值，两个拐点同时满足就无脑选智能版。

&emsp;&emsp;到这里 Layer 2 已经完整——2.2.4 给出了最小可用的朴素写入中间件，本节把它升级成了可以落到 FastAPI 生产的智能版本。接下来的 Layer 3 会从"对话语义记忆"跳到完全不同的赛道——任务级的结构化记忆，也就是把明确的任务指令追加写入 `todo.md` 文件。

---

### 2.3 TaskStateMiddleware：任务级结构化记忆

&emsp;&emsp;Layer 1 和 Layer 2 处理的都是"对话维度"的记忆——前者保存整段消息列表，后者保存决策型语义片段。但工程项目里还有第三类记忆：**明确的任务指令**。用户说"帮我把这个功能拆成三个子任务"或者"记得下次提醒我更新文档"，这类内容既不适合存在 messages 快照里（查找成本高），也不适合存在向量库里（需要 embedding，但任务追踪本质上是精确匹配而非语义检索）。

&emsp;&emsp;Layer 3 的解决方案更直接：把任务指令追加写入一个纯文本的 `todo.md` 文件，每一条任务是一行 Markdown 复选框格式。这个设计刻意保持简单——`todo.md` 可以被人类直接阅读和编辑，可以被 `Glob+Grep` 精确检索（见 Ch3），不需要 embedding，也不需要数据库。任务追踪要的是"精确、可编辑、可审查"，而不是"语义模糊匹配"。

#### 2.3.1 任务关键词触发与 todo.md 追加设计

&emsp;&emsp;和 Layer 2 一样，Layer 3 也用关键词列表来判断是否触发。但 Layer 3 的关键词分为**三组**——任务创建词（`帮我`、`任务`、`待办`）、完成标记词（`完成了`、`搞定了`）、清理归档词（`清理待办`、`归档任务`）。创建时不再 dump 用户原话，而是从 **AI 回复**中用正则提取编号列表项，逐条写入 `- [ ] {任务描述} (日期)` 格式；完成时通过精确匹配找到对应任务标记 `[x]`；清理时移除所有已完成条目。三条路径覆盖**任务的全生命周期：创建 → 跟踪 → 完成 → 归档**。

&emsp;&emsp;下面这张表对比了 Layer 2 和 Layer 3 在存储介质、触发条件和检索方式上的差异，帮助我们在实际项目里做出正确的选型。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Layer 2 长期记忆 vs Layer 3 任务记忆的核心差异</font></p>
<div class="center">

| 维度 | Layer 2（Mem0 + Milvus） | Layer 3（todo.md） |
|------|--------------------------|-------------------|
| **存储介质** | 向量数据库（Milvus） | 纯文本文件（Markdown） |
| **写入方式** | embedding + ANN 索引 | 正则提取 + 文件读写（append / in-place） |
| **检索方式** | 语义相似度检索 | Glob+Grep 精确匹配 |
| **适合内容** | 决策、选型、知识片段 | 明确任务指令、待办事项 |
| **可读性** | 需要 API 查询 | 人类可直接阅读编辑 |
| **典型用途** | "上次用了什么技术栈" | "下次要做什么任务" |

</div>

#### 2.3.2 TaskStateMiddleware 实现

&emsp;&emsp;`TaskStateMiddleware` 继承 `AgentMiddleware`，在 `after_model` hook 里根据用户消息的意图走三条路径之一：**创建**（从 AI 回复提取编号任务）、**完成**（匹配已有任务标记 `[x]`）、**清理**（移除所有 `[x]` 条目）。整个实现纯规则驱动——正则 + 关键词，**不调用 LLM**，与 L2 Smart 的 LLM 萃取形成互补：L2 靠 LLM 判断"值不值得写"，L3 靠正则解析"AI 拆出了哪些任务"。

In [51]:
import os
import re
from datetime import date
from pathlib import Path
from langchain.agents.middleware import AgentMiddleware, AgentState
from langgraph.runtime import Runtime


class TaskStateMiddleware(AgentMiddleware):
    """Layer 3：任务级结构化记忆——全生命周期管理

    继承 AgentMiddleware，实现 after_model hook，三条路径：
      1. 创建：用户消息命中任务词 → 从 AI 回复正则提取编号/列表项 → 逐条写入 todo.md
      2. 完成：用户消息命中完成词 → 在 todo.md 中匹配对应 [ ] → 改为 [x]
      3. 清理：用户消息命中清理词 → 移除所有 [x] 行

    纯规则驱动（正则 + 关键词），不调用 LLM，与 L2 Smart 的 LLM 萃取形成互补。

    Args:
        todo_path: todo.md 文件的绝对路径（不存在时自动创建）
        task_keywords: 触发任务创建的关键词列表
        done_keywords: 触发任务完成标记的关键词列表
        cleanup_keywords: 触发清理已完成任务的关键词列表
    """

    def __init__(
        self,
        todo_path: str = "./todo.md",
        task_keywords: list[str] | None = None,
        done_keywords: list[str] | None = None,
        cleanup_keywords: list[str] | None = None,
    ):
        super().__init__()
        self.todo_path = Path(todo_path)
        self.task_keywords = task_keywords or [
            "帮我", "任务", "待办", "记得", "提醒", "需要做",
        ]
        self.done_keywords = done_keywords or [
            "完成了", "做好了", "搞定了", "已完成", "做完了",
        ]
        self.cleanup_keywords = cleanup_keywords or [
            "清理待办", "归档任务", "整理任务", "清理已完成",
        ]
        self.todo_path.parent.mkdir(parents=True, exist_ok=True)

    # ── 主 hook ──────────────────────────────────────────────
    def after_model(self, state: AgentState, runtime: Runtime) -> dict | None:
        """LLM 响应完成后触发：按优先级走 清理 > 完成 > 创建 > no-op。"""
        messages = state["messages"]
        if not messages:
            return None
        last_user = self._last_text(messages, role="user")
        if not last_user:
            return None

        # 路径 1：清理已完成任务
        if any(kw in last_user for kw in self.cleanup_keywords):
            return self._cleanup_completed()
        # 路径 2：标记任务完成
        if any(kw in last_user for kw in self.done_keywords):
            last_ai = self._last_text(messages, role="ai")
            return self._mark_done(last_user, last_ai)
        # 路径 3：从 AI 回复提取新任务
        if any(kw in last_user for kw in self.task_keywords):
            last_ai = self._last_text(messages, role="ai")
            tasks = self._parse_tasks(last_ai)
            return self._write_tasks(tasks)
        return None  # no-op

    # ── 消息提取 ─────────────────────────────────────────────
    def _last_text(self, messages: list, role: str = "user") -> str:
        """从 messages 中倒序提取指定角色的最后一条文本内容

        Args:
            messages: LangChain message 列表
            role: "user" 匹配 human/user，"ai" 匹配 ai/assistant
        Returns:
            文本内容字符串，无匹配时返回空字符串
        """
        targets = ("human", "user") if role == "user" else ("ai", "assistant")
        for msg in reversed(messages):
            msg_role = getattr(msg, "type", None) or getattr(msg, "role", None)
            if msg_role in targets:
                content = getattr(msg, "content", "") or ""
                if isinstance(content, str) and content.strip():
                    return content
        return ""

    # ── 任务提取（正则） ─────────────────────────────────────
    def _parse_tasks(self, ai_text: str) -> list[str]:
        """从 AI 回复中用正则提取编号/列表项，返回任务描述列表

        支持格式：1. / 1、/ 1) / - / * 开头的列表项。
        若未匹配到结构化列表，把 AI 回复整体作为单条任务（兜底）。
        """
        pattern = r'(?:^|\n)\s*(?:\d+[.、)]\s*|[-*]\s+)(.+?)(?=\n|$)'
        items = [m.strip() for m in re.findall(pattern, ai_text) if m.strip()]
        if items:
            return items
        return [ai_text.strip()] if ai_text.strip() else []

    # ── 写入任务 ─────────────────────────────────────────────
    def _write_tasks(self, tasks: list[str]) -> None:
        """逐条追加写入 todo.md，每条带日期戳；已有相同描述则跳过"""
        today = date.today().isoformat()
        existing = self._read_todo()
        written = 0
        try:
            with open(self.todo_path, "a", encoding="utf-8") as f:
                for task in tasks:
                    if task in existing:
                        print(f"[Layer3] 跳过重复任务：{task[:40]}")
                        continue
                    f.write(f"- [ ] {task} ({today})\n")
                    written += 1
            if written:
                print(f"[Layer3] 写入 {written} 条任务到 {self.todo_path}")
        except OSError as e:
            print(f"[WARN] Layer 3 写入失败：{e}")
        return None

    # ── 标记完成 ─────────────────────────────────────────────
    def _mark_done(self, user_text: str, ai_text: str) -> None:
        """将 todo.md 中匹配的 [ ] 任务标记为 [x]

        匹配策略：提取每条 [ ] 任务的描述（去掉日期戳），
        检查该描述是否作为子串出现在 user_text + ai_text 中。
        因此 AI 回复中精确引用任务名即可命中。
        """
        if not self.todo_path.exists():
            return None
        text = self.todo_path.read_text(encoding="utf-8")
        combined = user_text + " " + ai_text
        file_lines = text.splitlines()
        updated = False
        for i, line in enumerate(file_lines):
            if line.startswith("- [ ] "):
                # 去掉日期戳 "(2026-04-16)" 后得到纯任务描述
                desc = re.sub(r'\s*\(\d{4}-\d{2}-\d{2}\)\s*$', '', line[6:])
                if desc in combined:
                    file_lines[i] = line.replace("- [ ] ", "- [x] ", 1)
                    print(f"[Layer3] 标记完成：{desc}")
                    updated = True
        if updated:
            self.todo_path.write_text("\n".join(file_lines) + "\n", encoding="utf-8")
        return None

    # ── 清理归档 ─────────────────────────────────────────────
    def _cleanup_completed(self) -> None:
        """移除 todo.md 中所有已完成的 [x] 条目"""
        if not self.todo_path.exists():
            return None
        text = self.todo_path.read_text(encoding="utf-8")
        file_lines = text.splitlines()
        done = [l for l in file_lines if l.startswith("- [x] ")]
        active = [l for l in file_lines if l.strip() and not l.startswith("- [x] ")]
        self.todo_path.write_text(
            ("\n".join(active) + "\n") if active else "", encoding="utf-8"
        )
        print(f"[Layer3] 归档 {len(done)} 条已完成任务，剩余 {len(active)} 条待办")
        return None

    # ── 工具方法 ─────────────────────────────────────────────
    def _read_todo(self) -> str:
        """读取 todo.md 全文，文件不存在时返回空字符串"""
        return self.todo_path.read_text(encoding="utf-8") if self.todo_path.exists() else ""

&emsp;&emsp;实现里有几个刻意的设计决策值得说明。第一，`after_model` 的优先级顺序是**清理 > 完成 > 创建**——如果用户说"帮我清理已完成的待办任务"，这句话同时命中"帮我"（创建词）和"清理待办"（清理词），优先级保证走清理路径而不是错误地尝试创建新任务。第二，`_parse_tasks` 支持 `1.` / `1、` / `1)` / `-` / `*` 五种列表格式，用一条正则统一处理；如果 AI 回复不含任何结构化列表，兜底把整段回复当一条任务写入，确保不丢任务。第三，`_mark_done` 的匹配策略是**精确子串匹配**：把 `todo.md` 中每条 `[ ]` 任务的描述（去掉日期戳）当 needle，在 `user_text + ai_text` 这个 haystack 里找，命中就标记——这要求 AI 回复中精确引用任务名，实际场景中 Agent 完全可以做到（它知道 `todo.md` 的内容）。第四，`_write_tasks` 写入前会查重，避免同一段对话被多次 `after_model` 触发时产生重复条目。

> ⚠️ **常见误区**：`_mark_done` 依赖 AI 回复中精确引用任务名，这是刻意的设计，不是偷懒。在真实 Agent 场景里，Agent 有能力读 `todo.md`（通过 Ch3 的 `search_files` 工具），因此它的回复里自然会包含精确的任务名。如果改用模糊匹配（如中文分词 + 语义相似度），反而会引入误标记风险——"设计数据库"误匹配"设计 API"这类边界情况在生产中很常见。精确匹配 + 让 Agent 负责引用 = 最可控的方案。

#### 2.3.3 独立测试：创建 / 完成 / 清理全生命周期

&emsp;&emsp;Layer 3 的测试覆盖任务的完整生命周期——无关消息不触发（边界）、多步骤任务提取（创建）、部分完成标记（跟踪）、全部完成后清理归档（闭环）。四个 case 按真实使用顺序编排，让学员一步步看到 `todo.md` 从空白到写满再到归零的完整过程。

In [29]:
import tempfile
from types import SimpleNamespace

# 使用临时文件做测试，不污染项目目录
tmp_todo_path = tempfile.mktemp(suffix="_ch2_test.md")
fake_runtime = SimpleNamespace(store=None, context=None, stream_writer=None)

layer3 = TaskStateMiddleware(todo_path=tmp_todo_path)

&emsp;&emsp;先初始化一个指向临时文件的 `TaskStateMiddleware` 实例。`tempfile.mktemp` 只生成路径不创建文件，Layer 3 首次写入时自动创建。`fake_runtime` 满足 `after_model(state, runtime)` 签名，Layer 3 内部不使用 runtime。

&emsp;&emsp;**Case 1：无任务关键词 → no-op**

&emsp;&emsp;普通闲聊不应触发任何写入，`todo.md` 应保持为空。这是边界测试的基线。

In [30]:
state_noop = {
    "messages": [
        HumanMessage(content="今天天气不错"),
        AIMessage(content="是的，很适合写代码"),
    ]
}
# 触发 after_model：闲聊内容无任务关键词，预期不写入 todo.md
layer3.after_model(state_noop, fake_runtime)
# todo.md 应不存在或为空
todo_text = open(tmp_todo_path).read() if os.path.exists(tmp_todo_path) else ""
# 验证：todo.md 应为空
assert todo_text == "", f"no-op 后文件应为空，实际：{todo_text!r}"
print("Case 1 (no-op) 通过：无任务关键词时不写入")

Case 1 (no-op) 通过：无任务关键词时不写入


&emsp;&emsp;**Case 2：多步骤任务提取 → 3 条独立任务**

&emsp;&emsp;用户请求拆分功能，AI 回复包含 3 条编号任务。Layer 3 应从 AI 回复中提取编号列表，逐条写入 `todo.md`，每条带日期戳。

In [31]:
state_create = {
    "messages": [
        HumanMessage(content="帮我把用户注册功能拆成子任务"),
        AIMessage(content="好的，已拆分为以下子任务：\n1. 设计用户表 schema\n2. 实现注册 API 接口\n3. 开发注册前端页面"),
    ]
}
# 触发写入：AI 回复包含编号任务列表
layer3.after_model(state_create, fake_runtime)

with open(tmp_todo_path, encoding="utf-8") as f:
    todo_after_create = f.read()
todo_lines = [l for l in todo_after_create.strip().split("\n") if l.startswith("- [")]
# 验证写入了 3 条独立任务，每条是 [ ]（待办）格式
# 验证：todo.md 应写入 3 条 [ ] 格式任务
assert len(todo_lines) == 3, f"应写入 3 条任务，实际：{len(todo_lines)}"
assert all(l.startswith("- [ ] ") for l in todo_lines), "所有任务应为待办状态"
assert "设计用户表 schema" in todo_after_create
assert "实现注册 API 接口" in todo_after_create
assert "开发注册前端页面" in todo_after_create
print(f"Case 2 (创建) 通过：写入 {len(todo_lines)} 条任务")
print(f"  todo.md 当前内容：\n{todo_after_create}")

[Layer3] 写入 3 条任务到 /var/folders/fl/8wq5_lz53ln9ypplts4z_1tr0000gn/T/tmp2cl9vt3z_ch2_test.md
Case 2 (创建) 通过：写入 3 条任务
  todo.md 当前内容：
- [ ] 设计用户表 schema (2026-04-17)
- [ ] 实现注册 API 接口 (2026-04-17)
- [ ] 开发注册前端页面 (2026-04-17)



&emsp;&emsp;**Case 3：标记单条任务完成 → `[ ]` 变 `[x]`**

&emsp;&emsp;用户告知第一个任务做完了，AI 回复中精确引用任务名作为确认。Layer 3 匹配到后将对应行从 `[ ]` 改为 `[x]`，其余任务保持不变。

In [32]:
state_done = {
    "messages": [
        HumanMessage(content="用户表 schema 设计完成了"),
        AIMessage(content="好的，已将「设计用户表 schema」标记为完成"),
    ]
}
# 触发完成标记：AI 回复精确引用任务名
layer3.after_model(state_done, fake_runtime)

with open(tmp_todo_path, encoding="utf-8") as f:
    todo_after_done = f.read()
# 验证第一条变 [x]，其余仍是 [ ]
# 验证：第一条任务变 [x]，其余仍 [ ]
assert "- [x] 设计用户表 schema" in todo_after_done, "第一条应标记完成"
assert todo_after_done.count("- [x]") == 1, "只应有 1 条完成"
assert todo_after_done.count("- [ ]") == 2, "应有 2 条待办"
print(f"Case 3 (完成标记) 通过：1 条完成，2 条待办")
print(f"  todo.md 当前内容：\n{todo_after_done}")

[Layer3] 标记完成：设计用户表 schema
Case 3 (完成标记) 通过：1 条完成，2 条待办
  todo.md 当前内容：
- [x] 设计用户表 schema (2026-04-17)
- [ ] 实现注册 API 接口 (2026-04-17)
- [ ] 开发注册前端页面 (2026-04-17)



&emsp;&emsp;**Case 4：全部完成 + 清理归档 → todo.md 归零**

&emsp;&emsp;先把剩余两条任务标记完成，然后触发清理。最终 `todo.md` 应为空，形成完整的任务生命周期闭环。

In [33]:
# 步骤 1：标记剩余 2 条任务完成
state_done_rest = {
    "messages": [
        HumanMessage(content="注册 API 和前端页面都做完了"),
        AIMessage(content="好的，「实现注册 API 接口」和「开发注册前端页面」已标记完成"),
    ]
}
layer3.after_model(state_done_rest, fake_runtime)

with open(tmp_todo_path, encoding="utf-8") as f:
    todo_all_done = f.read()
assert todo_all_done.count("- [x]") == 3, f"应有 3 条完成，实际：{todo_all_done.count('- [x]')}"
assert todo_all_done.count("- [ ]") == 0, "不应有待办任务"
print("  步骤 1 通过：3 条任务全部完成")

# 步骤 2：触发清理归档
state_cleanup = {
    "messages": [
        HumanMessage(content="帮我清理已完成的待办任务"),
        AIMessage(content="好的，已归档 3 条已完成任务，todo.md 已清空"),
    ]
}
layer3.after_model(state_cleanup, fake_runtime)

with open(tmp_todo_path, encoding="utf-8") as f:
    todo_final = f.read()
assert todo_final.strip() == "", f"清理后 todo.md 应为空，实际：{todo_final!r}"
print("  步骤 2 通过：todo.md 已清理归零")
print("Case 4 (闭环清理) 通过：CREATE → TRACK → COMPLETE → CLEAN 全生命周期验证完成")

# 清理临时文件
os.unlink(tmp_todo_path)
print(f"\n临时文件 {tmp_todo_path} 已清理")

[Layer3] 标记完成：实现注册 API 接口
[Layer3] 标记完成：开发注册前端页面
  步骤 1 通过：3 条任务全部完成
[Layer3] 归档 3 条已完成任务，剩余 0 条待办
  步骤 2 通过：todo.md 已清理归零
Case 4 (闭环清理) 通过：CREATE → TRACK → COMPLETE → CLEAN 全生命周期验证完成

临时文件 /var/folders/fl/8wq5_lz53ln9ypplts4z_1tr0000gn/T/tmp2cl9vt3z_ch2_test.md 已清理


&emsp;&emsp;四个 case 走完了任务从无到有、从待办到完成、从完成到归零的完整生命周期。Case 2 验证了正则提取的核心能力——AI 回复中的 `1. 2. 3.` 编号列表被拆成独立条目，每条都带日期戳，而不是把用户原话"帮我拆子任务"当成一条无意义的 todo 塞进去。Case 3 和 Case 4 验证了 `_mark_done` 的精确匹配策略——AI 回复中引用了完整的任务名（如「设计用户表 schema」），Layer 3 在 `todo.md` 中找到对应的 `[ ]` 行并改写为 `[x]`。Case 4 的清理步骤让 `todo.md` 归零，证明三条路径能配合形成闭环。

> 🔥 **踩坑预警**：Case 4 里"帮我清理已完成的待办任务"同时命中了任务创建词"帮我"和清理词"清理待办"。如果 `after_model` 没有正确设置优先级（清理 > 完成 > 创建），这句话会被错误地当作"创建新任务"处理，从 AI 回复中提取出"已归档 3 条已完成任务"作为一条新 todo——而不是执行清理。这就是为什么 `after_model` 里三条路径的 if 顺序不能随意调换。

---

### 2.4 集成测试：三层协同

&emsp;&emsp;单层测试通过只证明各层自身逻辑正确，集成测试要验证的是三层串联在同一个 Middleware 链里时，各层的触发条件互不干扰、写入结果互不覆盖。最重要的是，集成场景要模拟真实的多轮对话——不同轮次分别触发不同的 Layer，最后能从三个存储介质里各自取回数据。

&emsp;&emsp;我们的集成测试设计了 11 轮对话：第 4 轮触发 L1 压缩写入，第 8 轮触发 L2 长期记忆，第 9~11 轮依次触发 L3 的创建、完成标记和清理三条路径。这样的设计不仅验证三层的触发时机互不干扰，还在集成环境里跑通了 L3 的全生命周期。

#### 2.4.1 Part A — 初始化三个 Middleware

&emsp;&emsp;先分别初始化三个 Middleware 实例，然后按 L1 → L2 → L3 的顺序把它们装入 `middleware_chain` 列表。调用时按顺序依次执行，每个 Middleware 的 `after_model` 是纯副作用，不修改 state，因此链式调用不依赖返回值，每个 Middleware 拿到同一个 state dict。

In [37]:
import os

# Layer 1：会话级短期记忆，消息 >= 8 条触发压缩，持久化到本地 sessions/ 目录
integration_sessions_dir = "./sessions"
os.makedirs(integration_sessions_dir, exist_ok=True)
l1 = SessionWriteMiddleware(
    session_id="integration_session_001",
    llm=llm,  # 复用 Ch0 的全局 llm，用于压缩摘要生成
    compress_trigger=8,  # 累计消息数达到 8 条时触发压缩
    keep_recent=4,  # 压缩后保留最近 4 条原文
    sessions_dir=integration_sessions_dir,
)

# Layer 2：跨会话长期记忆，连接本地 Milvus
integration_memory = build_mem0_memory(
    collection_name="agent_ctx_ch3"
)
# 使用 2.2.4 升级后的智能版，带节流 + LLM 萃取，与 2.2.4 教学结论保持一致
l2 = SmartMem0WriteMiddleware(
    memory=integration_memory,
    extractor_llm=llm,
    user_id="integration_user_002",
)

# Layer 3：任务级结构化记忆，写入本地 todo.md
todo_path = "./todo.md"
l3 = TaskStateMiddleware(todo_path=todo_path)

# 三层 Middleware 链，按 L1 → L2 → L3 顺序
middleware_chain = [l1, l2, l3]
print("三层 Middleware 初始化完成")
print(f"  Layer 1 sessions_dir: {integration_sessions_dir}")
print(f"  Layer 2 user_id: {l2.user_id}")
print(f"  Layer 3 todo_path: {todo_path}")

三层 Middleware 初始化完成
  Layer 1 sessions_dir: ./sessions
  Layer 2 user_id: integration_user_002
  Layer 3 todo_path: ./todo.md


&emsp;&emsp;初始化顺序把 L1 放在最前，是因为 L1 负责 JSON 文件持久化，应在任何语义过滤发生前就拿到完整的消息列表；L2 和 L3 放在后面做按条件写入。三层的存储路径都指向本地目录——`./sessions/` 和 `./todo.md`——这样学员运行后可以直接打开文件查看持久化内容，比临时目录更直观。

#### 2.4.2 Part B — 11 轮对话模拟

&emsp;&emsp;`simulate_conversation` 函数接受一个对话脚本列表，每一条是 `(user_text, assistant_text)` 元组。它维护一个 messages 列表，每轮追加一对消息后，构造 state 并依次调用每个 middleware 的 `after_model`。`after_model` 返回 `None` 表示纯副作用，不修改 state，因此 messages 列表在每轮结束后保持完整累积。第 4 轮触发 L1，第 8 轮触发 L2，第 9~11 轮分别触发 L3 的创建、完成标记和清理三条路径。

In [38]:
from langchain_core.messages import HumanMessage, AIMessage
from types import SimpleNamespace

# fake_runtime 满足 after_model(state, runtime) 签名，三个字段均不实际使用
fake_runtime = SimpleNamespace(store=None, context=None, stream_writer=None)


def simulate_conversation(script, middleware_chain):
    """模拟多轮对话，每轮通过 middleware 链的 after_model hook 触发持久化
    
    Args:
        script: 对话脚本 [(user_text, assistant_text), ...]
        middleware_chain: AgentMiddleware 实例列表
    Returns:
        最终 messages 列表
    """
    messages = []
    for turn_idx, (user_text, assistant_text) in enumerate(script, start=1):
        # 每轮追加一对 user/assistant 消息
        messages.append(HumanMessage(content=user_text))
        messages.append(AIMessage(content=assistant_text))

        # 构造 state 并依次调用每个 middleware 的 after_model
        state = {"messages": messages}
        for mw in middleware_chain:
            # after_model 返回 None 表示纯副作用，不修改 state
            mw.after_model(state, fake_runtime)

        print(f"[Turn {turn_idx:02d}] user: {user_text[:40]}")

    return messages


# 11 轮对话脚本，第 4、8、9、10、11 轮是精心设计的触发点
conversation_script = [
    ("今天我们来讨论 Agent 架构", "好的，请继续"),
    ("我想了解 context 管理的方案", "context 管理分为 Compress 和 Write 两大类"),
    ("Compress 主要解决什么问题", "Compress 解决单次会话 token 超预算的问题"),
    ("Write 呢", "Write 解决跨会话持久化的问题，分三层"),  # 第 4 轮结束，messages 累计 8 条 → L1 触发
    ("三层 Write 分别是什么", "Layer1 Session、Layer2 Mem0、Layer3 Task"),
    ("Layer1 的压缩摘要是怎么生成的", "压缩摘要由 llm 对早期消息进行总结生成"),
    ("这个设计和直接截断有什么区别", "截断会丢失信息，压缩摘要保留语义"),
    ("我们决定在生产中采用这套三层 Write 架构", "好的，三层 Write 决策已记录"),  # 第 8 轮 → L2 触发
    ("帮我列出三层 Write 需要验证的要点", "好的：\n1. 验证 L1 JSON 压缩摘要完整性\n2. 验证 L2 Milvus 长期记忆召回\n3. 验证 L3 todo 任务追踪格式"),  # 第 9 轮 → L3 CREATE
    ("L1 压缩摘要我验证完成了", "好的，已确认「验证 L1 JSON 压缩摘要完整性」标记为完成"),  # 第 10 轮 → L3 DONE
    ("帮我清理已完成的待办任务", "好的，已归档 1 条已完成任务，剩余 2 条待办"),  # 第 11 轮 → L3 CLEANUP
]

&emsp;&emsp;对话脚本的触发设计覆盖了两个维度：**跨层隔离**和 **L3 全生命周期**。第 4 轮 messages 累计 8 条触发 L1 压缩写入；第 8 轮"决定"+"采用"触发 L2；第 9 轮"帮我"触发 L3 从 AI 回复提取 3 条任务写入 `todo.md`；第 10 轮"完成了"触发 L3 标记 1 条为 `[x]`；第 11 轮"清理待办"触发 L3 移除已完成条目。注意第 10、11 轮不会误触 L1（无新的压缩条件）或 L2（无决策关键词），这正是集成测试要验证的跨层隔离性。

#### 2.4.3 Part C — 执行与三层验证

&emsp;&emsp;执行模拟对话，然后分别从三个存储介质取回写入的内容做验证。

In [39]:
import json

print("=" * 50)
print("开始 11 轮集成对话模拟")
print("=" * 50)

# 执行 11 轮对话模拟：第4轮触发L1，第8轮触发L2，第9-11轮触发L3三条路径
final_messages = simulate_conversation(
    script=conversation_script,
    middleware_chain=middleware_chain,
)

print("\n" + "=" * 50)
print("三层写入结果验证")
print("=" * 50)

# 验证 L1：读 sessions/{session_id}.json，检查 compressed_context 非空
l1_json_path = os.path.join(integration_sessions_dir, "integration_session_001.json")
with open(l1_json_path, encoding="utf-8") as f:
    l1_session = json.load(f)
print(f"\n[L1 验证] JSON 文件：{l1_json_path}")
print(f"  messages 数量：{len(l1_session['messages'])}")
print(f"  compressed_context 长度：{len(l1_session['compressed_context'])} 字符")
# 预期：压缩后 messages 数量 <= keep_recent (4)，compressed_context 非空
assert len(l1_session['messages']) <= 4, "L1 压缩后 messages 应 <= 4"
assert l1_session['compressed_context'], "L1 compressed_context 应非空"
print("[PASS] L1 JSON 持久化 + 压缩摘要已写入")

# 验证 L2：Milvus 里应有长期记忆（第 8 轮含"决定"+"采用"触发）
l2_memories = integration_memory.get_all(user_id="integration_user_001")
print(f"\n[L2 验证] Milvus 记忆条数：{len(l2_memories)}")
assert len(l2_memories) > 0, "L2 应在第 8 轮写入决策记忆"
print("[PASS] L2 长期记忆已写入 Milvus")

# 验证 L3：todo.md 最终状态——第 9 轮创建 3 条，第 10 轮标记 1 条完成，第 11 轮清理已完成
with open(todo_path, encoding="utf-8") as f:
    todo_content = f.read()
print(f"\n[L3 验证] todo.md 最终内容：\n{todo_content}")
todo_remaining = [l for l in todo_content.strip().split("\n") if l.startswith("- [ ]")]
todo_done = [l for l in todo_content.strip().split("\n") if l.startswith("- [x]")]
# 第 9 轮创建 3 条 → 第 10 轮标记 1 条完成 → 第 11 轮清理 [x] → 剩余 2 条 [ ]
assert len(todo_remaining) == 2, f"L3 应剩余 2 条待办，实际：{len(todo_remaining)}"
assert len(todo_done) == 0, f"L3 清理后不应有已完成条目，实际：{len(todo_done)}"
assert "验证 L2 Milvus 长期记忆召回" in todo_content
assert "验证 L3 todo 任务追踪格式" in todo_content
print(f"[PASS] L3 全生命周期验证通过：创建 3 → 完成 1 → 清理 → 剩余 {len(todo_remaining)} 条待办")

print("\n[PASS] 所有三层验证通过")

开始 11 轮集成对话模拟
[Turn 01] user: 今天我们来讨论 Agent 架构
[Turn 02] user: 我想了解 context 管理的方案
[Turn 03] user: Compress 主要解决什么问题
[Turn 04] user: Write 呢
[Turn 05] user: 三层 Write 分别是什么
[Turn 06] user: Layer1 的压缩摘要是怎么生成的
[Turn 07] user: 这个设计和直接截断有什么区别
[Turn 08] user: 我们决定在生产中采用这套三层 Write 架构
[Layer3] 跳过重复任务：验证 L2 Milvus 长期记忆召回
[Layer3] 跳过重复任务：验证 L3 todo 任务追踪格式
[Layer3] 写入 1 条任务到 todo.md
[Turn 09] user: 帮我列出三层 Write 需要验证的要点
[Layer3] 标记完成：验证 L1 JSON 压缩摘要完整性
[Turn 10] user: L1 压缩摘要我验证完成了
[Layer3] 归档 1 条已完成任务，剩余 2 条待办
[Turn 11] user: 帮我清理已完成的待办任务

三层写入结果验证

[L1 验证] JSON 文件：./sessions/integration_session_001.json
  messages 数量：4
  compressed_context 长度：670 字符
[PASS] L1 JSON 持久化 + 压缩摘要已写入

[L2 验证] Milvus 记忆条数：1
[PASS] L2 长期记忆已写入 Milvus

[L3 验证] todo.md 最终内容：
- [ ] 验证 L2 Milvus 长期记忆召回 (2026-04-17)
- [ ] 验证 L3 todo 任务追踪格式 (2026-04-17)

[PASS] L3 全生命周期验证通过：创建 3 → 完成 1 → 清理 → 剩余 2 条待办

[PASS] 所有三层验证通过


&emsp;&emsp;三层 Write 协同正确：L1 在第 4 轮触发压缩并把结果持久化到 `./sessions/` 目录的 JSON 文件，`compressed_context` 字段保存了早期消息的摘要，`messages` 字段只保留最近 4 条；L2 在第 8 轮把决策写进了 Milvus；L3 完成了完整的任务生命周期——第 9 轮从 AI 回复中提取 3 条任务写入 `./todo.md`，第 10 轮标记其中 1 条为已完成，第 11 轮清理已完成条目，最终剩余 2 条待办。三层之间没有干扰：L3 的 DONE 和 CLEANUP 路径不会误触 L1 的压缩条件或 L2 的决策关键词，主链路 messages 列表保持完整。学员可以直接打开 `./sessions/integration_session_001.json` 和 `./todo.md` 查看持久化内容。下面这张表把三层的核心属性并排列出来，方便日后做架构设计时快速查阅。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>三层写入存储介质对比</font></p>
<div class="center">

| 层级 | Middleware 名称 | 存储介质 | 触发条件 | 写入粒度 | 典型用途 |
|------|-----------------|----------|----------|----------|----------|
| Layer 1 | `SessionWriteMiddleware` | JSON 文件 (sessions/*.json) | `len(messages) >= COMPRESS_TRIGGER`（默认 8） | 压缩摘要 + 最近 N 条 messages | 同用户跨会话上下文恢复 |
| Layer 2 | `Mem0WriteMiddleware` | Milvus 向量库 | 决策关键词匹配 | 本轮 user+assistant 对话 | 跨会话语义检索长期决策 |
| Layer 3 | `TaskStateMiddleware` | todo.md 文件 | 三组关键词（创建/完成/清理） | AI 回复中的编号任务列表 | 任务全生命周期管理 |

</div>

---

### 2.5 本章小结与下一步

&emsp;&emsp;本章从"写入缺失导致的三种失忆场景"出发，搭建了完整的三层 Write 架构。Layer 1 用 JSON 文件持久化 + LLM 压缩摘要的组合，把会话分两部分保存——最近 `KEEP_RECENT` 条原文保留在 messages 字段，早期消息被压缩为 `compressed_context` 摘要；`sessions/{session_id}.json` 文件结构跨进程可读，跨会话恢复只需一次 `json.load`；Layer 2 用关键词触发 + Mem0 + Milvus 把决策型对话持久化到向量库，实现跨越任意时间跨度的语义检索；Layer 3 用最简单的文件追加操作把任务指令结构化落地，不引入任何外部依赖。三层各自独立、可按需叠加，通过 Middleware 链串联后对主链路零侵入。至此，"写入"这个维度已经完整覆盖：短期 / 长期 / 任务三类失忆场景都有了对应的解决方案。

&emsp;&emsp;但写入只解决了"存到哪里"的问题。存进向量库的决策、写入 JSON 文件的会话压缩摘要、追加到 todo.md 的任务，在下次会话里怎么被高效地"读回来"？用什么策略决定检索哪一层、怎么融合多层的检索结果、如何在 token 预算有限的情况下选出最相关的内容？这正是第 3 章要回答的问题——Select 四策略的工程化落地：`SelectManager` 统一封装层、三个 `@tool` 工具层、`SkillsRouterMiddleware` 动态装配层，配合本章建立的三层存储，构成一套完整的"写 + 读"闭环。

---

## <center>第 3 章：Select 四策略的工程化落地</center>

&emsp;&emsp;Ch2 我们把对话持久化到了三层存储——会话级 JSON、长期记忆 Milvus、任务清单 todo.md，但"写进去"只是解决了存储,下一步就是**怎么把它们读回来**。`mem0.search`、`Glob+Grep`、`LlamaIndex QueryFusionRetriever`、Skills 分类器这四种 Select 技术，在基础课里各自是一个孤立的函数 demo，彼此之间没有统一接口，更没有任何 LangChain 化的封装。每次需要检索时，必须手工决定"调哪个函数、传什么参数、结果怎么格式化"，这种散装状态无法在真实 Agent 工程里复用。

&emsp;&emsp;本章要回答的问题是：如何把基础课里散装的四种 Select 技术，真正装到一个 LangChain Agent 里去。答案是三层架构——`SelectManager`（统一实现层）、三个 `@tool`（Agentic 工具层）、`SkillsRouterMiddleware`（动态装配层）——配合 Ch1 已有的 `SummarizationMiddleware`，构成一个可复用、可叠加的完整工程方案。接下来的 3.1 到 3.4，我们会逐层搭建、最后合流演示。

> 📌 **前置知识声明**：本章默认学员已完成《大模型 Agent 上下文工程基础入门》课程中 3.1.2.1~3.1.2.4 四小节（`mem0.search` / `Glob+Grep` / `LlamaIndex QueryFusionRetriever` / Skills 路由），本章**不重复**这四种技术的单点 API 用法，而是讲**如何把它们封装为可工程化复用的统一层 + LangChain Middleware**。

### 3.1 SelectManager 统一实现层

&emsp;&emsp;为什么要把四个散装函数封装进一个类？核心原因有三：第一，统一入参和出参格式——散装函数各自返回不同形状的数据结构，让调用方每次都要做 ad-hoc 格式转换；第二，统一量化口径——工程上需要知道"每次检索节省了多少 tokens"，散装函数没有这个口径，但 `SelectManager` 可以在每个方法的出参里统一附带 `tokens_saved`；第三，实例生命周期管理——`mem0` 客户端和 `LlamaIndex` 索引构建开销很大，封装在类里只初始化一次，而不是每次检索都重新构造。

&emsp;&emsp;三个理由说完了，但“散装有多痛苦”光说不够——我们直接用代码体验一下。下面这个 cell 模拟基础课的散装调用方式：直接调 `mem0.search()`、手动 `glob.glob()` + 逐文件 `open()` 搜索、`retriever.retrieve()`，三种 API 各自为战。请注意观察三个痛点：返回格式完全不统一（list[dict] vs list[str] vs list[NodeWithScore]）、没有 `tokens_saved` 量化口径、每次调用都要写一堆 ad-hoc 的格式化代码。

#### 3.1.1 类骨架与 SelectConfig

&emsp;&emsp;`SelectConfig` 是 `SelectManager` 的唯一入口，使用 Python `dataclass` 定义。这样做有两个好处：一是类型明确，IDE 可以做完整补全；二是字段默认值清晰，不会出现"不知道该传什么"的困惑。`mem0_client` 字段从外部传入而不是内部创建，因为 Ch2 已经构造好了一个 `Memory` 实例，这里直接复用，避免重复初始化。

&emsp;&emsp;下面这个 cell 只做 `import`，把三层架构的核心符号引进来。`SelectManager`、`SelectConfig`、`SkillsRouterMiddleware` 这三个名字将在后面的 3.1.2、3.2、3.3 小节分别深入讲解；真正的实例化放在 3.4 小节，等 `skill_registry` 构造完之后再一次性组装。

In [24]:
# 从 select_manager 模块导入三层架构的核心类
from select_manager import SelectManager, SelectConfig, SkillsRouterMiddleware

# mem0 实例和 knowledge_docs 已由 Ch2 构建好，skill_registry 将在 3.4 小节构造
# 本 cell 只验证导入正常，不做实例化
print("SelectManager / SelectConfig / SkillsRouterMiddleware 导入成功")

SelectManager / SelectConfig / SkillsRouterMiddleware 导入成功


&emsp;&emsp;运行后看到"导入成功"即可继续。如果报 `ModuleNotFoundError`，检查 `select_manager.py` 是否在当前工作目录下——它由本课配套文件提供，与本 Notebook 同级放置即可。

#### 3.1.2 四个底层方法的 API 契约一览

&emsp;&emsp;`SelectManager` 内部封装了四个底层方法，每个方法对应基础课的一种 Select 技术。它们共享一个关键设计：**所有方法的返回值都是 `dict`，且都携带 `tokens_saved` 字段**。这个统一口径让 `@tool` 层可以用同一套格式化逻辑输出"节省了多少 tokens"，而不需要各自写不同的格式化代码——这正是"工程化封装"相较于"散装函数"最直接的价值体现。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>SelectManager 四个方法的统一契约</font></p>
<div class="center">

| 方法 | 对应基础课 | 返回关键字段 | 量化指标 |
|------|-----------|-------------|---------|
| `search_memory` | 3.1.2.1 | `memories` / `raw_count` / `filtered_count` | `tokens_saved` |
| `search_files` | 3.1.2.2 | `matched_files` / `snippets` / `glob_count` | `tokens_saved` |
| `search_knowledge` | 3.1.2.3 | `nodes` / `retriever_type` | `tokens_saved` |
| `classify_skill` | 3.1.2.4 | `skill_id` / `confidence` / `matched` | 供 Middleware 使用 |

</div>

&emsp;&emsp;需要特别说明 `classify_skill` 的定位：前三个方法是"检索工具"——LLM 决策"要不要调用这个工具、何时调用"；而 `classify_skill` 是"元级决策工具"——它判断"整个请求应该路由到哪个 Skill、装配哪些工具"，因此它不做成 `@tool`，而是被 `SkillsRouterMiddleware` 内部调用，这个分工逻辑我们在 3.3 小节展开。

#### 3.1.3 四方法独立验证


&emsp;&emsp;在把 `SelectManager` 包装成 `@tool` 之前，先直接调用四个底层方法，看看原始返回的 `dict` 长什么样。这一步有两个目的：第一，确认每个方法都能正常返回且带 `tokens_saved` 字段；第二，建立“裸数据”的直觉——后面 `@tool` 层做的格式化只是把这些 dict 拼接成 LLM 可读的字符串，不改变底层数据结构。


In [26]:
# 构造临时 SelectManager 实例，仅用于验证四个方法的原始返回
# 3.4.2 会用 init_select_manager() 绑定全局实例，此处是独立验证，互不影响
from llama_index.core import Document

memory = build_mem0_memory(collection_name="agent_ctx_ch2_test")

In [27]:

test_config = SelectConfig(
    mem0_client=memory,                    # 复用 Ch2 已构建的 Memory 实例
    llm=llm,                             # 复用 Ch2 的 DeepSeek 实例
    knowledge_docs=[
        Document(text="LangChain @tool 装饰器的 docstring 就是 LLM 看到的工具描述"),
        Document(text="mem0 默认从 user 消息中提取事实，纯问句提取不到决策信息"),
    ],
    skill_registry={                     # 最小 registry，只需一个 Skill 用于 classify_skill 测试
        "debug_code": {
            "description": "代码调试：根据报错信息定位 bug",
            "tools": [],
            "system_prompt": "你是代码调试专家。",
        },
    },
    files_root=".",                      # 当前目录作为 Glob 起始路径
)
sm_test = SelectManager(test_config)

The tokenizer parameter is deprecated and will be removed in a future release. Use a stemmer from PyStemmer instead.
As bm25s.BM25 requires k less than or equal to number of nodes added. Overriding the value of similarity_top_k to number of nodes added.


error uploading: HTTPSConnectionPool(host='us.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1016)')))


In [28]:


# === 正面验证：四个方法的正常返回 ===
print("=" * 50)
print("  正面验证：四个方法的正常返回")
print("=" * 50)

# --- 验证 search_memory ---
r1 = sm_test.search_memory(query="之前选了什么数据库", user_id="demo_user")
print("search_memory 返回 keys:", list(r1.keys()))
print(f"  memories={r1['memories'][:2]}, tokens_saved={r1['tokens_saved']}")

# --- 验证 search_files ---
r2 = sm_test.search_files(query="import", glob_pattern="*.py")
print(f"\nsearch_files 返回 keys:", list(r2.keys()))
print(f"  glob_count={r2['glob_count']}, matched_files={len(r2['matched_files'])}, tokens_saved={r2['tokens_saved']}")

# --- 验证 search_knowledge ---
r3 = sm_test.search_knowledge(query="tool 装饰器", top_k=2)
print(f"\nsearch_knowledge 返回 keys:", list(r3.keys()))
print(f"  nodes={len(r3['nodes'])}, retriever_type={r3['retriever_type']}, tokens_saved={r3['tokens_saved']}")

# --- 验证 classify_skill ---
r4 = sm_test.classify_skill(query="代码报错了帮我看看")
print(f"\nclassify_skill 返回 keys:", list(r4.keys()))
print(f"  skill_id={r4['skill_id']}, confidence={r4['confidence']:.2f}, matched={r4['matched']}")

# 统一断言：前三个方法都必须带 tokens_saved
for name, r in [("search_memory", r1), ("search_files", r2), ("search_knowledge", r3)]:
    assert "tokens_saved" in r, f"{name} 缺少 tokens_saved 字段"
print("\n 四个方法全部返回正常，tokens_saved 口径统一")

# === 边界验证：降级与缓存行为 ===
print("\n" + "=" * 50)
print("  边界验证：降级与缓存行为")
print("=" * 50)

# --- 边界 Case A：search_memory 完全无关 query → 应返回空结果 ---
r_empty = sm_test.search_memory(query="量子纠缠和烤面包的关系", user_id="demo_user")
print(f"\n[边界A] 无关 query → filtered={r_empty['filtered_count']}/{r_empty['raw_count']}, "
      f"tokens_saved={r_empty['tokens_saved']}")
assert r_empty["filtered_count"] == 0 or r_empty["memories"] == [], \
    "完全无关的 query 应返回空或 0 条过滤结果"

# --- 边界 Case B：classify_skill 不匹配任何 Skill → matched=False ---
r_fallback = sm_test.classify_skill(query="今天天气怎么样")
print(f"[边界B] 无关 Skill → matched={r_fallback['matched']}, "
      f"skill_id={r_fallback['skill_id']}, confidence={r_fallback['confidence']:.2f}")
assert r_fallback["matched"] is False, "不属于任何 Skill 应 matched=False"
assert r_fallback["skill_id"] is None, "未匹配时 skill_id 应为 None"

# --- 边界 Case C：classify_skill 缓存命中 → 相同 query 第二次调用结果一致 ---
r_cached = sm_test.classify_skill(query="代码报错了帮我看看")  # 与 r4 相同的 query
assert r_cached == r4, "相同 query 应命中 _classify_cache，返回完全一致的 dict"
print(f"[边界C] 缓存命中 → r_cached == r4: {r_cached == r4}")

# --- 边界 Case D：search_files 不匹配任何文件 → snippets 为空 ---
r_no_files = sm_test.search_files(query="量子纠缠", glob_pattern="*.xyz")
print(f"[边界D] 不存在的文件类型 → matched_files={len(r_no_files['matched_files'])}, "
      f"snippets={len(r_no_files['snippets'])}, tokens_saved={r_no_files['tokens_saved']}")
assert r_no_files["matched_files"] == [], "不存在 .xyz 文件，matched_files 应为空"
assert r_no_files["tokens_saved"] == 0, "无文件可搜时 tokens_saved 应为 0"

# --- 边界 Case E：search_knowledge 无关 query → 返回结构完整但内容不相关 ---
r_irrelevant_k = sm_test.search_knowledge(query="量子纠缠和烤面包", top_k=1)
print(f"[边界E] 无关知识查询 → nodes={len(r_irrelevant_k['nodes'])}, "
      f"retriever_type={r_irrelevant_k['retriever_type']}, tokens_saved={r_irrelevant_k['tokens_saved']}")
assert "tokens_saved" in r_irrelevant_k, "search_knowledge 必须始终返回 tokens_saved"
assert "retriever_type" in r_irrelevant_k, "search_knowledge 必须始终返回 retriever_type"

print("\n五条边界 Case 全部通过：空结果降级 / Skill 未匹配回退 / 分类缓存命中 / 无文件匹配 / 无关知识查询")

  正面验证：四个方法的正常返回
search_memory 返回 keys: ['memories', 'raw_count', 'filtered_count', 'tokens_saved']
  memories=[], tokens_saved=0

search_files 返回 keys: ['matched_files', 'snippets', 'glob_count', 'tokens_saved']
  glob_count=5, matched_files=5, tokens_saved=18617
LangChain @tool 装饰器的 docstring 就是 LLM 看到的工具描述 mem0 默认从 user 消息中提取事实，纯问句提取不到决策信息

search_knowledge 返回 keys: ['nodes', 'retriever_type', 'tokens_saved']
  nodes=2, retriever_type=fusion, tokens_saved=0

classify_skill 返回 keys: ['skill_id', 'confidence', 'matched']
  skill_id=debug_code, confidence=0.95, matched=True

 四个方法全部返回正常，tokens_saved 口径统一

  边界验证：降级与缓存行为

[边界A] 无关 query → filtered=0/0, tokens_saved=0
[边界B] 无关 Skill → matched=False, skill_id=None, confidence=0.10
[边界C] 缓存命中 → r_cached == r4: True
[边界D] 不存在的文件类型 → matched_files=0, snippets=0, tokens_saved=0
LangChain @tool 装饰器的 docstring 就是 LLM 看到的工具描述 mem0 默认从 user 消息中提取事实，纯问句提取不到决策信息
[边界E] 无关知识查询 → nodes=1, retriever_type=fusion, tokens_saved=28

五条边界 Case 全部通过：空结果降级 / 

&emsp;&emsp;运行后分两组观察。**正面验证**：四个方法返回的 `keys` 与 3.1.2 表格完全一致——`search_memory` 有 `memories` / `raw_count` / `filtered_count` / `tokens_saved`，`search_knowledge` 有 `nodes` / `retriever_type` / `tokens_saved`；`classify_skill` 正确识别“代码报错”应路由到 `debug_code`。**边界验证**：五条 Case 覆盖了五种工程高频场景——Case A 验证“无关 query 不返回脏数据”，Case B 验证“无匹配 Skill 时优雅降级到 fallback”，Case C 验证“分类缓存命中时返回一致性”，Case D 验证“不存在的文件类型 glob 返回空列表且 tokens_saved=0”，Case E 验证“search_knowledge 对无关查询始终返回结构完整的 dict（含 tokens_saved / retriever_type）”。这五条边界 Case 正是从散装函数升级到 `SelectManager` 后才能统一测试的——散装状态下你得对每个函数各写一套降级逻辑。

### 3.2 三个 @tool 封装：从类方法到 Agentic 工具

&emsp;&emsp;把 `SelectManager` 的前三个方法变成 LangChain `@tool` 的核心动机是：LLM 需要"看到"工具描述才能决策何时调用检索，而 `@tool` 装饰器把方法的 `docstring` 直接暴露给 LLM 作为工具描述。第四个方法 `classify_skill` 不做成 `@tool`，因为意图分类是"谁来装配工具"的元级决策，不应该交给 LLM 在对话中临时发起，而应该在每次 LLM 调用前由 `SkillsRouterMiddleware` 自动触发——这就是 Tool 与 Middleware 的分工边界。

&emsp;&emsp;下面这段代码定义了全局 `_sm` 变量和 `init_select_manager` 函数，以及三个 `@tool`。全局实例的设计原因在代码注释和后面的误区提示中会详细解释，请仔细阅读。

In [8]:
# 从 langchain_core.tools 导入 @tool 装饰器
from langchain_core.tools import tool

# 全局 SelectManager 实例，在 3.4 小节 create_agent 之前必须已初始化
_sm: SelectManager = None

def init_select_manager(config: SelectConfig) -> None:
    """把 SelectManager 实例绑定到模块全局，供三个 @tool 闭包捕获使用。"""
    global _sm
    _sm = SelectManager(config)

@tool
def search_memory(query: str, user_id: str = "demo_user") -> str:
    """在长期记忆库中检索与 query 相关的历史决策和用户偏好。适用于回忆之前的技术决策和项目上下文。"""
    # 调用底层 SelectManager 方法，统一返回 dict 格式
    result = _sm.search_memory(query=query, user_id=user_id)
    if not result["memories"]:
        return f"未找到与 '{query}' 相关的记忆。"
    # 拼接过滤统计和节省 token 信息
    lines = [f"[过滤后 {result['filtered_count']}/{result['raw_count']}]"]
    lines.extend(f"- {m}" for m in result["memories"])
    lines.append(f"(节省 {result['tokens_saved']} tokens)")
    return "\n".join(lines)

@tool
def search_files(query: str, glob_pattern: str = "**/*.py") -> str:
    """在项目文件系统中两阶段检索（Glob 粗筛 + Grep 精筛）包含指定关键词的代码行。"""
    # 两阶段检索：Glob 先扩散候选，Grep 再精确命中
    result = _sm.search_files(query=query, glob_pattern=glob_pattern)
    if not result["snippets"]:
        return f"未匹配 '{query}'。Glob 候选 {result['glob_count']} 个。"
    lines = [f"[Glob {result['glob_count']} → Grep {len(result['matched_files'])}]"]
    # 最多展示 8 条代码行，控制注入 token 量
    for s in result["snippets"][:8]:
        lines.append(f"{s['file']}:{s['line']}  {s['text']}")
    lines.append(f"(节省 {result['tokens_saved']} tokens)")
    return "\n".join(lines)

@tool
def search_knowledge(query: str, top_k: int = 3) -> str:
    """在外部知识库中用混合检索（向量 + BM25 + 基础课 3.1.2.3 的混合召回）找最相关的技术文档片段。"""
    # 调用 LlamaIndex QueryFusionRetriever 封装，返回带 score 的节点列表
    result = _sm.search_knowledge(query=query, top_k=top_k)
    if not result["nodes"]:
        return f"未找到与 '{query}' 相关的知识。"
    lines = [f"[retriever={result['retriever_type']}]"]
    for n in result["nodes"]:
        lines.append(f"[{n.score:.3f}] {n.text[:120]}...")
    lines.append(f"(节省 {result['tokens_saved']} tokens)")
    return "\n".join(lines)

print("3 个 @tool 已定义：search_memory / search_files / search_knowledge")

3 个 @tool 已定义：search_memory / search_files / search_knowledge


&emsp;&emsp;关键设计要点有三个：第一，`docstring` 就是 LLM 看到的工具描述，写得简洁且直指使用场景，LLM 才能在正确时机调用正确工具；第二，全局 `_sm` 闭包是为了避免把 `SelectManager` 实例塞进 `@tool` 的函数签名——`@tool` 装饰器对参数类型非常敏感，实例对象会干扰 LangChain 的 JSON schema 生成；第三，这三个 `@tool` 后面会被 `create_agent` 和 `SkillsRouterMiddleware` 共享，不需要重复定义。

> ⚠️ **常见误区**：很多人把 `SelectManager` 实例作为 `@tool` 函数的第一个参数，导致 LLM 在生成 `tool_call` 时必须自己填实例——这是不可能的，LLM 只能生成 JSON 可序列化的参数。正确姿势是全局实例 + 闭包捕获，让 LLM 只看到业务参数（`query` 等）。

&emsp;&emsp;定义好三个 `@tool` 后，在进入 3.3 的 Middleware 层之前，先验证一下 `@tool` 的**真实输出格式**——因为 LLM 看到的不是 `dict`，而是 `@tool` 函数返回的格式化字符串。这一步的目的是让学员直观看到“docstring 决定何时调用，返回字符串决定 LLM 拿到什么信息”这个关键链路。

In [74]:
# 验证 @tool 的真实输出格式（LLM 看到的就是这个字符串）
# 先用 3.1.3 的 test_config 初始化全局实例，让三个 @tool 闭包能正常捕获
init_select_manager(test_config)

print("=== search_memory @tool 输出 ===")
mem_output = search_memory.invoke({"query": "之前选了什么数据库", "user_id": "demo_user"})
print(mem_output)

print("\n=== search_knowledge @tool 输出 ===")
know_output = search_knowledge.invoke({"query": "tool 装饰器", "top_k": 1})
print(know_output)

print("\n=== search_files @tool 输出 ===")
file_output = search_files.invoke({"query": "import", "glob_pattern": "*.py"})
print(file_output)

# 验证输出都是字符串（@tool 的返回值必须是 LLM 可读的纯文本）
assert isinstance(mem_output, str), "@tool 返回值必须是 str"
assert isinstance(know_output, str), "@tool 返回值必须是 str"
assert isinstance(file_output, str), "@tool 返回值必须是 str"
print("\n三个 @tool 输出均为 str 格式，LLM 可直接读取")

The tokenizer parameter is deprecated and will be removed in a future release. Use a stemmer from PyStemmer instead.
As bm25s.BM25 requires k less than or equal to number of nodes added. Overriding the value of similarity_top_k to number of nodes added.


=== search_memory @tool 输出 ===
[过滤后 3/3]
- ORM 使用 SQLAlchemy 2.0
- 采用 Milvus 作为向量库
- 后端选 FastAPI + Python 3.11
(节省 0 tokens)

=== search_knowledge @tool 输出 ===
LangChain @tool 装饰器的 docstring 就是 LLM 看到的工具描述 mem0 默认从 user 消息中提取事实，纯问句提取不到决策信息
[retriever=fusion]
[0.033] LangChain @tool 装饰器的 docstring 就是 LLM 看到的工具描述...
(节省 28 tokens)

=== search_files @tool 输出 ===
[Glob 4 → Grep 4]
./mock_papers.py:8  reconstruct note: 本文件在 Phase 7 清理误删后按 probes.py 的 import 签名重建，
./mock_papers.py:12  from dataclasses import dataclass
./compression_metrics.py:7  from typing import Callable, Optional
./compression_metrics.py:8  from langchain_core.messages import BaseMessage, ToolMessage, AIMessage, SystemMessage
./compression_metrics.py:34  from functools import lru_cache
./compression_metrics.py:38      from transformers import AutoTokenizer
./select_manager.py:6  from __future__ import annotations
./select_manager.py:8  import glob as _glob
(节省 10503 tokens)

三个 @tool 输出均为 str 格式，LLM 可直接读取


### 3.3 SkillsRouterMiddleware：动态工具装配的正解

&emsp;&emsp;基础课 3.1.2.4 的 Skills 路由做法是"每次意图分类后手工调用 `create_agent` + `AgentExecutor` 重建整个 Agent"。这个做法在教学演示时没问题，但在工程场景里有三个明显缺陷：一是不能和其他 Middleware 叠加（重建 Agent 会丢掉已挂载的 Middleware 状态）；二是每次请求都要重建整个 Agent，开销不可忽略；三是不符合 LangChain v1.0 的 Middleware 一等公民机制——该版本提供了 `wrap_model_call` 钩子，专门用来在 LLM 调用前后做拦截和修改。

&emsp;&emsp;`SkillsRouterMiddleware` 就是用 `wrap_model_call` 实现动态工具装配的正解。它在每次 LLM 被调用前自动执行意图分类，然后用 `ModelRequest.override()` 方法把 `tools` 和 `system_prompt` 替换为匹配 Skill 的子集——整个过程对主 Agent 完全透明，`invoke` 接口完全不变。

In [9]:
# 直接从 select_manager 模块导入已经封装好的 Middleware
# 它继承 langchain.agents.middleware.AgentMiddleware，只实现 wrap_model_call 一个钩子
from select_manager import SkillsRouterMiddleware

print("SkillsRouterMiddleware 已从 select_manager 模块导入")

SkillsRouterMiddleware 已从 select_manager 模块导入


&emsp;&emsp;`request.override()` 返回的是一个新对象而不是原地修改——这保证了幂等性，即同一个 `request` 被多个 Middleware 串联处理时，上游修改不会污染下游看到的原始状态。对比 Ch1 的 `SummarizationMiddleware`（修改 `messages` 字段）和本章 `SkillsRouterMiddleware`（修改 `tools` + `system_prompt` 字段），两者都用 `wrap_model_call` 但操作的 state 字段完全不同——这正体现了 Middleware 的正交可叠加特性。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>本课三个 Middleware 修改的 state 字段对比</font></p>
<div class="center">

| Middleware | 修改字段 | 触发时机 | 本课来源 |
|-----------|---------|---------|---------|
| `SummarizationMiddleware` | `messages`（历史压缩） | token 超阈值时 | Ch1 |
| `ModelCallLimitMiddleware` | 无修改，只做计数限制 | 每次 LLM 调用 | Ch1 |
| `SkillsRouterMiddleware` | `tools` + `system_prompt` | 每次 LLM 调用 | 本章 |

</div>

&emsp;&emsp;三个 Middleware 正交不冲突，叠加顺序遵循一个简单规则：<font color=red>修改 `messages` 的放外层，修改 `tools` 的放内层</font>。原因是 `SummarizationMiddleware` 先把历史压缩干净，`SkillsRouterMiddleware` 再基于干净的 messages 做意图分类——如果顺序反了，分类器看到的是被压缩改写过的 messages，意图识别准确率会下降。

> 🔥 **踩坑预警**：`skill_registry` 中 `description` 的写法直接决定 `classify_skill` 的分类准确率。如果 description 过于笼统（例如“通用问答”），分类器会频繁误判。生产经验：description 必须包含 2-3 个**具体场景关键词**——写“代码调试：根据**报错信息**定位 **bug**”，而不是“调试代码”。关键词越具体，分类器越容易区分相似 Skill，准确率从 ~60% 提升到 ~90%。

### 3.4 三层合流：端到端 Agent 演示

&emsp;&emsp;前三小节各自展示了一层：`SelectManager` 是统一实现层，三个 `@tool` 是 Agentic 工具层，`SkillsRouterMiddleware` 是动态装配层。本节把三层接起来，构造一个挂载了 `SkillsRouterMiddleware` 的主 Agent，演示三条不同意图的 query 如何被动态路由到不同的工具子集。

#### 3.4.1 构造 skill_registry

&emsp;&emsp;`skill_registry` 是 `SkillsRouterMiddleware` 的核心配置，它把前面定义的三个 `@tool` 分组为三个具有独立语义的 Skill。每个 Skill 有三个字段：`description` 是分类器用来判断路由的唯一依据（越精确准确率越高）、`tools` 是该 Skill 激活时 LLM 能看到的工具子集、`system_prompt` 是该 Skill 激活时 LLM 使用的专属系统提示。

In [29]:
# skill_registry 把三个 @tool 按业务语义分组为三个 Skill
# 每个 Skill 的 tools 子集可以重叠，体现"Skill 是 tools 的有意义分组，不是互斥分区"
skill_registry = {
    "debug_code": {
        "description": "代码调试：根据报错信息定位 bug、读取相关文件、搜索相似实现",
        "tools": [search_files, search_memory],
        "system_prompt": "你是代码调试专家，先读文件再定位问题，引用历史决策辅助判断。",
    },
    "answer_knowledge": {
        "description": "技术问答：查询技术概念、最佳实践、配置建议、方案对比",
        "tools": [search_knowledge, search_memory],
        "system_prompt": "你是技术顾问，优先从知识库检索权威资料，辅以用户的历史偏好。",
    },
    "recall_decision": {
        "description": "历史决策回忆：查找用户过去的技术选型、项目上下文、偏好设置",
        "tools": [search_memory],
        "system_prompt": "你是项目助理，专注从长期记忆中检索用户的历史决策。",
    },
}
print(f"已构造 {len(skill_registry)} 个 Skill，每个 Skill 绑定独立的 tool 子集与 system_prompt")

已构造 3 个 Skill，每个 Skill 绑定独立的 tool 子集与 system_prompt


&emsp;&emsp;观察三个 Skill 的工具子集：`debug_code` 用 `search_files + search_memory`，`answer_knowledge` 用 `search_knowledge + search_memory`，`recall_decision` 只用 `search_memory`。`search_memory` 在前两个 Skill 里都出现——这说明 Skill 的工具分组是"有意义的业务聚焦"，不是互斥的功能切割。分类器将 `description` 字段与用户 query 做语义匹配，因此这段描述越具体，路由准确率越高。

#### 3.4.2 SelectConfig 实例化与主 Agent 构造

&emsp;&emsp;现在把所有准备好的材料组装进 `SelectConfig`，然后调用 `init_select_manager` 把实例绑定到全局，让三个 `@tool` 的闭包能捕获到它。这里有一个工程化的关键收益：同一个 `config` 构造一次，三个 `@tool` 和 `SkillsRouterMiddleware` 共享同一个底层实例，不会出现基础课里"每次演示都重新构造 `mem0` / `index`"的冗余初始化问题。

In [30]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from select_manager import SelectManager, SelectConfig
from llama_index.core import Document

# 构造演示用知识库文档（生产场景从文件/数据库加载）
knowledge_docs = [
    Document(text="PostgreSQL 连接池推荐 pgbouncer transaction 模式，max_connections 设为 CPU×2+1"),
    Document(text="Redis Cache-Aside 标准流程：读先查缓存 miss 则查 DB 并回填，TTL 300 秒"),
    Document(text="LangChain @tool 装饰器的 docstring 就是 LLM 看到的工具描述，影响调用准确率"),
    Document(text="mem0 默认从 user 消息中提取事实，纯问句提取不到决策信息"),
]

# 构造 SelectConfig：mem0 实例复用 Ch2 已构建的 Memory，llm 复用同一 DeepSeek 实例
config = SelectConfig(
    mem0_client=memory,                    # 来自 Ch2，复用不重建
    llm=llm,                             # 分类器与主 Agent 共用同一 LLM 实例
    knowledge_docs=knowledge_docs,       # 演示用知识库
    skill_registry=skill_registry,      # 上一步构造的三个 Skill
    files_root="./select_demo_project",  # Glob 起始目录
)
SelectManager(config)
print("SelectManager 全局实例已绑定，三个 @tool 闭包可正常捕获")

The tokenizer parameter is deprecated and will be removed in a future release. Use a stemmer from PyStemmer instead.
As bm25s.BM25 requires k less than or equal to number of nodes added. Overriding the value of similarity_top_k to number of nodes added.


SelectManager 全局实例已绑定，三个 @tool 闭包可正常捕获


&emsp;&emsp;`init_select_manager` 执行后，全局 `_sm` 变量指向刚构造的 `SelectManager` 实例，三个 `@tool` 的闭包就能捕获到它，后续 LLM 调用 `search_memory` / `search_files` / `search_knowledge` 时会走这个实例的底层方法。接下来构造主 Agent，把两层 Middleware 按正确顺序叠加进去。

> ⚠️ **常见误区**：忘记调用 `init_select_manager()` 就直接让 Agent `invoke`——三个 `@tool` 闭包捕获到的 `_sm` 还是 `None`，LLM 调用工具时会报 `AttributeError: 'NoneType' object has no attribute 'search_memory'`。排查时很容易误以为是 mem0 连接问题，实际只是初始化顺序错了。生产建议：在 `create_agent` 之前加一行 `assert _sm is not None, "必须先调用 init_select_manager()"`。

In [31]:
# 先把 SkillsRouterMiddleware 实例保存到变量，便于后续直接读取 _last_decision
# 不要依赖 compiled agent 的私有属性访问中间件
skills_router_mw = SkillsRouterMiddleware(select_manager=_sm)

# 构造主 Agent：tools 传入全集，Middleware 在运行时动态裁剪
# 叠加顺序：修改 messages 的 SummarizationMiddleware 在外层，修改 tools 的在内层
main_agent = create_agent(
    model=llm,
    tools=[search_memory, search_files, search_knowledge],  # 静态注册全集
    middleware=[
        # SummarizationMiddleware(model=llm, trigger=("tokens", 3000), keep=("messages", 4)),
        skills_router_mw,                                    # 运行时动态裁剪子集
    ],
)
print("主 Agent 已构造：3 个 tool + 2 层 Middleware（SummarizationMiddleware + SkillsRouterMiddleware）")

主 Agent 已构造：3 个 tool + 2 层 Middleware（SummarizationMiddleware + SkillsRouterMiddleware）


&emsp;&emsp;重点理解 `tools` 参数的设计：这里传入的是**全集** `[search_memory, search_files, search_knowledge]`，但 `SkillsRouterMiddleware` 会在每次 LLM 调用前用 `request.override(tools=...)` 把工具集裁剪为当前 Skill 的子集。这就是"静态注册全集 + 运行时动态装配子集"的完整工程姿势，主 Agent 对裁剪过程完全无感。

#### 3.4.3 端到端真实 Agent 测试


&emsp;&emsp;前面的 `skill_registry`、`SelectConfig`、主 Agent 已经全部就位。现在用**四条真实 query** 让主 Agent 完整跑一遍“意图分类 → 动态装配 → 工具调用 → 生成回复”全流程。四条 query 分别覆盖三个 Skill 和一个 fallback 路径——对比 Ch2 的集成测试用 11 轮对话覆盖三层 Write 的各自触发点，本章的端到端测试要覆盖三个 Skill 的各自路由加一个兜底路径，验证 `SkillsRouterMiddleware` 在不同意图下的装配决策是否正确。

In [80]:
# 四条真实 query：覆盖 3 个 Skill + 1 个 fallback 路径
test_cases = [
    # (query, 预期 skill_id, 预期 tool_count, 说明)
    ("AgentManager 初始化时报 KeyError: 'middleware'，帮我定位这个 bug",
     "debug_code", 2, "代码调试 → search_files + search_memory"),
    ("PostgreSQL 连接池用 pgbouncer 的 transaction 模式好还是 session 模式好",
     "answer_knowledge", 2, "技术问答 → search_knowledge + search_memory"),
    ("我们之前选了什么数据库来着",
     "recall_decision", 1, "历史回忆 → search_memory"),
    ("今天天气怎么样",
     None, 0, "无匹配 → fallback"),
]

print("=" * 70)
print("  端到端路由验证：4 条 query × 3 Skill + 1 fallback")
print("=" * 70)

for i, (query, expected_skill, expected_tools, desc) in enumerate(test_cases, 1):
    print(f"\n── Case {i}: {desc} ──")
    print(f"[Query] {query}")

    # 调用主 Agent——所有动态装配对调用方完全透明
    result = main_agent.invoke({"messages": [{"role": "user", "content": query}]})

    # 从 Middleware 变量读取装配决策
    d = skills_router_mw._last_decision
    actual_skill = d["skill_id"]
    status = "true" if actual_skill == expected_skill else "false"
    print(f"{status} 路由: {actual_skill or 'fallback':20s} | "
          f"置信度: {d['confidence']:.2f} | ")

    # 打印 Agent 回复摘要（前 100 字）
    reply = result["messages"][-1].content
    print(f"[回复摘要] {reply[:100]}...")

print("\n" + "=" * 70)
print("  路由汇总")
print("=" * 70)
for i, (query, expected_skill, expected_tools, desc) in enumerate(test_cases, 1):
    print(f"  Case {i}: {desc}")

  端到端路由验证：4 条 query × 3 Skill + 1 fallback

── Case 1: 代码调试 → search_files + search_memory ──
[Query] AgentManager 初始化时报 KeyError: 'middleware'，帮我定位这个 bug
true 路由: debug_code           | 置信度: 0.95 | 
[回复摘要] 我需要更多信息来帮你定位这个 `KeyError: 'middleware'` 错误。请提供以下信息：

## 1. **错误堆栈信息**
完整的错误堆栈跟踪，特别是：
- 哪一行代码触发了错误
- ...

── Case 2: 技术问答 → search_knowledge + search_memory ──
[Query] PostgreSQL 连接池用 pgbouncer 的 transaction 模式好还是 session 模式好
false 路由: fallback             | 置信度: 0.20 | 
[回复摘要] 选择 pgbouncer 的连接池模式取决于你的应用场景和需求。以下是两种模式的对比分析：

## **Transaction 模式（推荐用于大多数场景）**
```sql
pool_mode = t...

── Case 3: 历史回忆 → search_memory ──
[Query] 我们之前选了什么数据库来着
false 路由: fallback             | 置信度: 0.10 | 
[回复摘要] 很抱歉，我无法访问之前的对话记录，因此不清楚我们之前选择了哪个数据库。不过，如果你能提供一些上下文信息（比如项目类型、需求或之前的讨论细节），我可以帮你重新评估或推荐合适的数据库选择！

如果你记得部...

── Case 4: 无匹配 → fallback ──
[Query] 今天天气怎么样
true 路由: fallback             | 置信度: 0.10 | 
[回复摘要] 我无法获取实时天气信息，建议您通过以下方式查询：  
1. 打开手机天气应用  
2. 搜索“天气+您所在城市名”  
3. 查看天气预报网站（如中国天气网）  
4. 询问智能设备（如小爱同学、

&emsp;&emsp;运行后重点观察路由汇总表的四行结果。**Case 1**：“报 KeyError” 触发 `debug_code`，工具集裁剪到 2 个（`search_files + search_memory`），`search_knowledge` 被排除——LLM 不会在调试场景里去查知识库。**Case 2**：“连接池用什么模式” 触发 `answer_knowledge`，工具集切换为 `search_knowledge + search_memory`，`search_files` 被排除——技术问答不需要翻代码文件。**Case 3**：“之前选了什么数据库” 触发 `recall_decision`，工具集只剩 `search_memory` 一个——历史回忆只需要查长期记忆。**Case 4**：“今天天气” 不属于任何 Skill，走 fallback 路径，工具集降级为空（或 fallback_tools）。

&emsp;&emsp;四条 Case 验证了 `SkillsRouterMiddleware` 的三个核心行为：**正确分类**（不同意图路由到不同 Skill）、**精准裁剪**（每个 Skill 只暴露业务相关的工具子集）、**优雅降级**（无匹配时走 fallback 而非报错）。整个过程 `main_agent.invoke` 的接口完全没变——所有动态行为都封装在 Middleware 内部。

### 3.5 本章小结

&emsp;&emsp;本章把基础课散装的四种 Select 技术，完整封装进了三层架构：`SelectManager` 统一了入参出参和 `tokens_saved` 口径，三个 `@tool` 把检索能力暴露给 LLM Agentic 决策，`SkillsRouterMiddleware` 通过 `wrap_model_call` 实现了运行时动态工具装配。三层结构与 Ch1 的 Middleware 范式完全一致，`SummarizationMiddleware` 和 `SkillsRouterMiddleware` 正交叠加、互不干扰——这是 LangChain v1.0 Middleware 机制作为一等公民设计的直接收益。

&emsp;&emsp;但是一个新的问题在 Ch3 结束时悄悄浮现：Skills 路由只决定了主 Agent 能**看到**哪些工具，它不决定这些工具被调用后产生的 `ToolMessage` 如何隔离——当 `search_files` 一次命中十几条 snippet、或者用户抛来"综述 10 篇论文"这种单次吸收大段资料的任务时，原始数据依然会整条进入主 Agent messages，`ToolMessage` 污染与"主上下文膨胀"依旧没解决。第 4 章 要补的，就是"装配 vs 执行"的另一半——用 `isolate_manager.py` 把重度吸收任务剥离到独立的子 Agent，让主 Agent 只拿结构化摘要。

---

## <center>第 4 章：Isolate 编排</center>

&emsp;&emsp;本章承接 第 3 章 的遗留问题——`SkillsRouterMiddleware` 把工具集裁剪为 Skill 子集，只解决了"装配侧"的上下文膨胀；一旦工具真的被调用，`search_files` 一次命中十几条 snippet、或用户丢来"综述 10 篇论文"这种单次吸收大段资料的任务，原始数据依然会整条灌进主 Agent 的 messages 数组。M4 工具上下文层会从健康滑回污染，M5 任务状态层也被无关中间态堆满——这正是"装配 vs 执行"里 Ch3 没覆盖到的另一半。

&emsp;&emsp;本章的答案是 `IsolateManager`：把重度吸收任务委派给一个独立的研究子 Agent，让主 Agent 只拿回一段不超过 500 字的结构化摘要。接下来我们会按五条主线展开：**4.1** 讲 `IsolateManager` 类骨架与 `IsolateConfig`，建立 Layer 1 统一实现层；**4.2** 把 `research_subagent` 的 `@tool` 闭包源码展开，看清 LLM 调度时看到什么；**4.3** 在同一个 `create_agent` 里合流 Ch1 Summarization、Ch3 SkillsRouter、Ch4 research_subagent，构成三层融合主 Agent；**4.4** 用 `run_p4_probe` 真跑 baseline vs isolate，亲眼看两个反直觉的数字；**4.5** 小结本章，并留下一个重要伏笔——尽管前四章每章都在压 input_tokens，`cache_hit_tokens` 始终是 0。

---

### 4.1 IsolateManager 统一实现层

&emsp;&emsp;和 第 3 章 的 `SelectManager` 完全对称，我们把 Ch4 的 Isolate 策略也先封装进一个 Layer 1 统一实现层——`isolate_manager.py` 模块。它对外暴露两个核心符号：`IsolateConfig` 配置容器和 `IsolateManager` 管理类。`IsolateManager` 负责构建 `research_subagent_tool`（Middleware 层的核心工具件）和 `run_p4_probe`（baseline vs isolate 真跑探针的入口），职责边界非常清晰——它只负责"构造 Layer 1 工程件"，主 Agent 的装配始终交给 notebook 层。接下来的 4.1.1 到 4.1.3 将逐步展开这一层的三个关键要素。

#### 4.1.1 类骨架与 IsolateConfig

&emsp;&emsp;`IsolateConfig` 是 `IsolateManager` 的唯一入口，使用 Python `dataclass` 定义。它只有三个必要字段：`llm` 是主 Agent 使用的 LLM 实例；`papers` 是论文 fixture（默认取 `ALL_PAPERS[:10]`，与 Ch3 Select 演示用的同一份 mock 数据）；`subagent_model` 是可选的子 Agent 专属模型，不传时直接复用 `llm`，避免双实例化开销。`IsolateManager.__init__` 在构造时会预拼一次 `_full_papers_text`——把 10 篇论文的 `paper_id + title + abstract` 拼成一段长字符串——后续所有方法都直接读这个属性，<font color=red>不会在每次调用时重复拼接，这是一个有意识的内存换 CPU 的工程决定</font>。

In [13]:
# 从 isolate_manager 模块导入 Ch4 Layer 1 的核心符号
# IsolateConfig：配置容器（dataclass，字段类型明确）
# IsolateManager：统一实现层（研究子 Agent 工厂 + P4 探针）
from isolate_manager import IsolateManager, IsolateConfig
from mock_papers import ALL_PAPERS  # 与 Ch3 共用的 10 篇 mock 论文 fixture

# 实例化 IsolateManager
# subagent_model 不传 → 子 Agent 复用主 llm，省去双实例化
im_config = IsolateConfig(llm=llm, papers=list(ALL_PAPERS[:10]))
im = IsolateManager(im_config)

# 验证：_full_papers_text 已在 __init__ 里预拼完毕
print("IsolateManager 实例化完成，papers 数量:", len(im.config.papers))
print("_full_papers_text 预拼字符数:", len(im._full_papers_text))

IsolateManager 实例化完成，papers 数量: 10
_full_papers_text 预拼字符数: 929


&emsp;&emsp;执行后你会看到 `papers 数量: 10` 以及 `_full_papers_text` 的字符数。字符数通常在 500-2000 之间，取决于 10 篇 mock 论文 abstract 的长度。这段文本就是研究子 Agent 每次需要"吃下"的全量资料，也是主 Agent 在 isolate 策略下永远**看不到**的原始数据。

#### 4.1.2 三方法 API 契约一览

&emsp;&emsp;在深入代码之前，先把 `IsolateManager` 对外暴露的两个方法 + 一个属性的 API 契约用表格呈现，方便和后续代码对照。和 Ch3 的 `SelectManager` 速览表一样，这张表格不是参考手册，而是一张"索引卡"——看完之后你心里已经知道每个方法的位置，后续 4.1.3/4.2/4.4 遇到具体代码时能直接定位到对应契约。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>IsolateManager API 契约一览</font></p>
<div class="center">

| 方法 / 属性 | 入参 | 返回 | 用途 |
|------------|------|------|------|
| `build_research_subagent_tool()` | 无 | `BaseTool`（`@tool` 装饰后的 `research_subagent`）| 构造研究子 Agent 工具，通过闭包捕获 `_full_papers_text` / `sub_llm` / `cap` |
| `run_p4_probe(strategy)` | `strategy: str`（`"baseline"` 或 `"isolate"`）| `dict`（含 7 个指标字段）| P4 真跑探针：baseline 直接塞主 Agent，isolate 先走子 Agent 再返摘要 |
| `_full_papers_text`（属性）| — | `str` | 预拼好的全量论文文本，`__init__` 时一次性构造，多次读取零开销 |

</div>

#### 4.1.3 研究子 Agent 独立验证

&emsp;&emsp;在把 `research_subagent_tool` 挂进主 Agent 之前，我们先"绕开"主 Agent，直接手动 invoke 一次工具。这和 Ch3 的验证策略一致——先用底层方法裸跑，确认输出形态正确，再挂进 Agent 的完整链路。裸跑的另一个价值是：你能亲眼看到子 Agent 消化 10 篇论文后输出的"摘要字符串"长什么样——这条字符串就是主 Agent 在 isolate 策略下唯一会收到的 `ToolMessage.content`。

In [20]:
# 构造 research_subagent 工具对象（闭包捕获 im 实例的 _full_papers_text / sub_llm / cap）
research_tool = im.build_research_subagent_tool()

# 验证工具的元数据：name 和 description 是 LLM 决策调用的唯一依据
print("tool name:       ", research_tool.name)
print("tool description (前 60 字):", research_tool.description[:60])

# 裸 invoke：绕开主 Agent，直接调用工具，观察子 Agent 摘要产物
# 参数 research_query 对应 @tool 函数签名里的同名参数
raw_summary = research_tool.invoke({"research_query": "中文 Agent 记忆方向的主要贡献"})

# 只打印前 400 字，避免输出过长
print("\n[子 Agent 摘要] (前 400 字):")
print(raw_summary[:400])

tool name:        research_subagent
tool description (前 60 字): 把一个重度的文献综述子任务委派给独立的研究子 Agent。子 Agent 会阅读全量论文并返回一段不超过 500 字的结

[子 Agent 摘要] (前 400 字):
**中文 Agent 记忆方向主要贡献的结构化摘要**

近期研究围绕提升中文 Agent 的记忆能力，在架构设计、检索优化、工程实现与成本控制等方面取得系列进展，核心贡献可归纳如下：

**1. 记忆架构与持久化机制创新**
*   **分层架构**：提出面向中文 Agent 的短期/工作/长期三层记忆系统（如 Mem0），旨在解决长会话中的事实漂移问题，实验表明可显著提升信息召回率（如提升 23%）。
*   **跨会话持久化**：设计了结合 PostgresSaver 与 Mem0 等双层存储的工程方案，并配套检查点与失败回滚机制，实现了记忆的可靠跨会话保存与恢复。

**2. 检索策略与算法融合优化**
*   **多路融合检索**：针对中文特点，提出融合 BM25（词法）、语义向量、规则匹配与时间衰减的四路检索策略，实验证明其综合性能（F1值）较单一检索路径有显著提升（如提升 1


&emsp;&emsp;运行后你会看到 `tool name: research_subagent`，以及子 Agent 针对"中文 Agent 记忆方向"输出的一段结构化摘要文字。注意摘要会被截断到 `summary_max_chars=500` 字符以内——这个截断在 `build_research_subagent_tool` 的闭包里硬编码，保证主 Agent 无论何时拿到的 `ToolMessage.content` 都不会超过 500 字。`tool description` 的第一句是 LLM 决策"该不该调用这个工具"的核心依据，写得越精确、路由准确率越高——这一点我们在 4.2 节展开讲。

---

### 4.2 research_subagent @tool 封装

&emsp;&emsp;4.1.3 是"手动 invoke 工具对象"，但主 Agent 看到工具的方式和我们不一样——它看的是工具的 `name` + `description`（即 `@tool` 装饰器从函数 docstring 自动提取的元数据），在决定"要不要调这个工具"时不会看函数体代码。调用发生后，主 Agent 拿到的是一条 `ToolMessage.content` 字符串——10 篇论文的原文永远不会出现在这条消息里。理解这个机制之后，我们把 `build_research_subagent_tool` 的闭包实现展开，看清它是如何把三个关键变量"封进"工具的。

In [84]:
# 以下是 build_research_subagent_tool 闭包实现的 notebook 演示版
# 用不同的函数名（research_tool_demo）避免与上方 research_tool 变量冲突
# 目的：在 notebook 里直接看清闭包捕获的三个变量是什么

from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage

def build_research_tool_demo(full_text: str, sub_llm, cap: int = 500):
    """演示版：把 research_subagent 的闭包逻辑拆开在 notebook 里展示。

    Args:
        full_text: 预拼好的全量论文文本（IsolateManager._full_papers_text）。
        sub_llm:   子 Agent 使用的 LLM 实例（默认复用主 llm）。
        cap:       摘要字符上限（默认 500）。

    Returns:
        装饰后的 LangChain BaseTool 实例。
    """
    # 闭包在函数定义时就捕获了这三个变量的引用
    # full_text → 每次 invoke 时读取，不重复拼接
    # sub_llm   → 子 Agent 的 LLM 实例，与主 Agent 解耦
    # cap       → 摘要截断上限，限制 ToolMessage.content 长度

    _sub_system = (
        "你是专职研究子 Agent，擅长阅读大段文献并输出紧凑的结构化摘要。"
        "你的输出会被主 Agent 直接采信，因此必须客观、有证据、拒绝夸张。"
    )

    @tool
    def research_subagent_demo(research_query: str) -> str:
        """把一个重度的文献综述子任务委派给独立的研究子 Agent。子 Agent 会阅读全量论文并返回一段不超过 500 字的结构化摘要。适用于『综述某方向的贡献』『对比多篇论文观点』这类单次需要吸收大段资料但只需要结论的场景。"""
        # 子 Agent 的消息序列：系统提示 + 全量论文 + 任务
        messages = [
            SystemMessage(content=_sub_system),
            HumanMessage(
                content=f"{full_text}\n\n任务：{research_query}\n请用 300-500 字给出结构化摘要。"
            ),
        ]
        # 独立 LLM 调用：与主 Agent 完全解耦，论文原文不进入主 Agent messages
        response = sub_llm.invoke(messages)
        content = response.content if isinstance(response.content, str) else str(response.content)
        # 截断到 cap：确保返回给主 Agent 的 ToolMessage.content 不超限
        return content[:cap]

    return research_subagent_demo

# 验证演示版工具的行为与原版一致
demo_tool = build_research_tool_demo(
    full_text=im._full_papers_text,  # 复用 im 里预拼好的文本
    sub_llm=llm,                     # 复用主 llm
    cap=500,
)
print("演示版 tool name:", demo_tool.name)
print("docstring 前 60 字:", demo_tool.description[:60])

演示版 tool name: research_subagent_demo
docstring 前 60 字: 把一个重度的文献综述子任务委派给独立的研究子 Agent。子 Agent 会阅读全量论文并返回一段不超过 500 字的结


&emsp;&emsp;三个闭包变量各有其工程意义：`full_text` 在 `IsolateManager.__init__` 时一次性预拼并绑定到闭包，后续每次 invoke 时工具直接读取而不重复拼接长字符串；`sub_llm` 由调用方在构造 `IsolateConfig` 时决定是否单独指定一个更小的研究模型（省成本），默认复用主 Agent 的 LLM 实例；`cap` 则是保护主 Agent 上下文的"防洪闸"——哪怕子 Agent 写出 2000 字的摘要，也会在进入 `ToolMessage` 之前被硬截断到 500 字以内。

> ⚠️ **常见误区**：很多同学第一次实现 subagent 时，会把 `IsolateManager` 实例本身作为 `@tool` 函数的参数传进来——这是错误的。`@tool` 装饰器的函数签名直接映射为 JSON Schema，`IsolateManager` 这类复杂对象无法序列化。正确姿势是闭包捕获：在工厂方法（`build_research_subagent_tool`）定义时，用局部变量把所需的对象引用绑定到闭包作用域，`@tool` 函数的签名里只保留 LLM 能理解的基础类型参数。

---

### 4.3 三层融合主 Agent：Ch1 + Ch3 + Ch4

&emsp;&emsp;Layer 1（`IsolateManager`）和 Layer 2（`research_subagent @tool`）都已就绪。本节把 Ch4 的 `research_subagent` 和 Ch3 的 `SkillsRouterMiddleware`、Ch1 的 `SummarizationMiddleware` 在同一个 `create_agent` 里合流——得到一个既能压缩历史、又能按意图路由工具子集、又能把重度吸收任务剥离到子 Agent 的**三层融合主 Agent**。你会发现三章的工作成果正交叠加，完全不互相干扰——这是 LangChain Middleware 机制作为"一等公民"设计的直接收益。

#### 4.3.1 注册 research skill 到 skill_registry

&emsp;&emsp;`skill_registry` 是 `SkillsRouterMiddleware` 意图路由的核心配置。Ch3 里已经构造了 `debug_code` / `answer_knowledge` / `recall_decision` 三个 Skill；现在追加一个 `"research"` Skill，把 `research_subagent_tool` 挂进去。`description` 字段决定了意图分类器会不会把"综述类"请求路由到这里，要写得尽量精确——<font color=red>越精确的 description，路由准确率越高</font>，因为分类器做的就是 query 与 description 的语义匹配。

In [15]:
# 在 Ch3 已构造好的 skill_registry 里追加 "research" skill
# description 精确描述适用场景 → 分类器路由准确率更高
# 把"综述"作为第一关键词，覆盖常见触发词变体（综述/总结/归纳/对比多篇）
skill_registry["research"] = {
    "description": (
        "综述类研究任务：对多篇论文做文献综述、总结某研究方向的核心贡献、"
        "归纳多文献观点、对比多论文结论；适用于单次需要吸收大段资料后只返回摘要的场景"
    ),
    "tools": [research_tool],       # 只暴露 research_subagent，主 Agent 看不到其他工具
    "system_prompt": (
        "你是研究型助理，遇到综述类任务时应调用 research_subagent 工具"
        "把资料吸收任务委派出去，只在摘要基础上作答。"
    ),
}
print("当前 skill_registry 包含:", list(skill_registry.keys()))

当前 skill_registry 包含: ['debug_code', 'answer_knowledge', 'recall_decision', 'research']


&emsp;&emsp;执行后 `skill_registry` 应包含四个 Skill：Ch3 的三个加上刚注册的 `"research"`。新 Skill 注册完之后，`SkillsRouterMiddleware` 的行为会自动扩展——遇到"综述中文 Agent 记忆方向"这类请求时，分类器把 `skill_id` 判为 `"research"`，然后 `request.override(tools=[research_tool], system_prompt=...)` 把主 Agent 可见工具集裁剪为只含 `research_subagent` 一个工具，主 Agent 面对这唯一的工具就会直接调用它，把资料吸收任务委派出去。

#### 4.3.2 显式实例化三 middleware + 主 Agent 装配

&emsp;&emsp;这是本章最关键的代码 cell，也是修复 P0 Bug 的核心动作。Ch3 的 `main_agent` 里已经有了 `SummarizationMiddleware` 和 `SkillsRouterMiddleware`；本节我们从头显式实例化三个 middleware 并直接传进 `create_agent`，不依赖任何封装好的"一行装配"函数——目的是让学员亲眼看到三层 middleware 按顺序塞进 Agent 的完整过程，以及为什么叠加顺序很重要。

&emsp;&emsp;叠加顺序的工程规则只有一条：<font color=red>修改 `messages` 的 Middleware 放外层，修改 `tools` 的放内层</font>。原因是 `SummarizationMiddleware` 先把历史压缩干净，`SkillsRouterMiddleware` 再基于干净的 messages 做意图分类——如果顺序反了，分类器看到的是被压缩改写过的 messages，意图识别准确率会下降。`research_subagent_tool` 不是 Middleware，而是一个普通 `@tool`，挂进 `tools` 列表之后由 `SkillsRouterMiddleware` 负责在运行时决定"是否暴露给本次 LLM 调用"——三者分工明确，零耦合。

In [ ]:
from select_manager import SelectManager, SelectConfig
from llama_index.core import Document

# 构造包含 research skill 的完整 skill_registry                                                                                                                  
# 注意：这里需要包含之前 Ch3 定义的三个 skill + 新增的 research skill
full_skill_registry = {                                                                                                                                          
    "debug_code": {
        "description": "代码调试：根据报错信息定位 bug、分析堆栈、检查配置",
        "tools": [search_files, search_memory],
        "system_prompt": "你是代码调试专家，擅长根据报错信息快速定位问题。",
    },                                                                                                                                                           
    "answer_knowledge": {
        "description": "技术问答：回答架构选型、最佳实践、框架对比等知识性问题",                                                                                 
        "tools": [search_knowledge, search_memory],
        "system_prompt": "你是技术顾问，基于知识库给出专业建议。",
    },
    "recall_decision": {
        "description": "历史回忆：回忆之前的技术决策、项目上下文、团队共识",                                                                                     
        "tools": [search_memory],
        "system_prompt": "你是项目记忆助手，帮助回忆历史决策。",                                                                                                 
    },          
    "research": {
        "description": (
            "综述类研究任务：对多篇论文做文献综述、总结某研究方向的核心贡献、"
            "归纳多文献观点、对比多论文结论；适用于单次需要吸收大段资料后只返回摘要的场景"                                                                       
        ),
        "tools": [research_tool],  # research_tool 应该在 4.1.3 节已经构造                                                                                       
        "system_prompt": (                                                                                                                                       
            "你是研究型助理，遇到综述类任务时应调用 research_subagent 工具"
            "把资料吸收任务委派出去，只在摘要基础上作答。"                                                                                                       
        ),                                                                                                                                                       
    },
}                                                                                                                                                                
                
# 构造完整的 SelectConfig
# 注意：memory 和 llm 应该在 Ch2 已经构造好
full_config = SelectConfig(
    mem0_client=memory,                    # 复用 Ch2 的 Memory 实例
    llm=llm,                              # 复用 Ch2 的 DeepSeek 实例
    knowledge_docs=knowledge_docs,        # 应该在之前已经定义
    skill_registry=full_skill_registry,   # 包含 4 个 skill 的完整 registry                                                                                      
    files_root="./select_demo_project",   # Glob 起始目录
)                                                                                                                                                                
                
# 初始化全局 SelectManager 实例
init_select_manager(full_config)

# 验证初始化成功
print(f"_sm 已初始化: {_sm is not None}")
print(f"skill_registry 包含: {list(_sm.config.skill_registry.keys())}") 

The tokenizer parameter is deprecated and will be removed in a future release. Use a stemmer from PyStemmer instead.
As bm25s.BM25 requires k less than or equal to number of nodes added. Overriding the value of similarity_top_k to number of nodes added.


✅ _sm 已初始化: True
✅ skill_registry 包含: ['debug_code', 'answer_knowledge', 'recall_decision', 'research']


In [40]:
from langchain.agents import create_agent                          # 主 Agent 构造入口
from langchain.agents.middleware import SummarizationMiddleware    # Ch1 层：压缩历史消息
from select_manager import SkillsRouterMiddleware                  # Ch3 层：动态工具路由

# 确认 _sm 已初始化                                                                                                                                              
assert _sm is not None, "❌ _sm 未初始化，请先运行 init_select_manager()"

# Ch1 层：会话压缩（修改 messages，放外层）
# trigger=("tokens", 3000)：超过 3000 token 触发；keep=("messages", 4)：保留最近 4 条
summary_mw = SummarizationMiddleware(
    model=llm,
    trigger=("tokens", 3000),
    keep=("messages", 4),
)

# Ch3 层：动态工具装配（修改 tools / system_prompt，放内层）
# _sm 是 Ch3 已全局绑定的 SelectManager 实例，SkillsRouterMiddleware 读取其 skill_registry
skills_router_mw = SkillsRouterMiddleware(select_manager=_sm)

# Ch4 层：research_subagent 作为普通 @tool 挂进工具全集
# 不是 Middleware！它由 SkillsRouterMiddleware 在运行时决定是否暴露
# research_tool 已在 4.1.3 构造完毕，此处直接复用

# 三层合流：按"外层压 messages → 内层切 tools"顺序挂载 middleware
# 工具全集传入 create_agent，运行时由 skills_router_mw 动态裁剪子集
dual_agent = create_agent(
    model=llm,
    tools=[research_tool],
    middleware=[summary_mw, skills_router_mw],  # 外层 → 内层顺序不可颠倒
)
print("三层融合主 Agent 装配完成：Ch1 Summarization + Ch3 SkillsRouter + Ch4 research_subagent")

三层融合主 Agent 装配完成：Ch1 Summarization + Ch3 SkillsRouter + Ch4 research_subagent


&emsp;&emsp;注意 `skills_router_mw` 显式保存为 notebook 局部变量——这不只是习惯，而是工程必要性：后续 4.3.3 需要直接读取 `skills_router_mw._last_decision` 来观察路由决策。如果把 middleware 实例化藏进封装函数的内部，这个变量就不在 notebook 作用域里，访问时会 `NameError`。<font color=red>显式 > 隐式</font>，这是 Python 的核心哲学，在 middleware 叠加这个场景里尤其重要。

#### 4.3.3 端到端真跑：综述类 query 路由验证

&emsp;&emsp;三层装配完成之后，发一条"综述类"query 让 Agent 完整跑一遍：意图分类 → 裁剪工具集 → 调用 `research_subagent` → 子 Agent 吸收 10 篇论文输出摘要 → 主 Agent 基于摘要生成最终回复。全流程对调用方透明，`dual_agent.invoke` 的接口与 Ch3 `main_agent.invoke` 完全一致。

In [44]:
# 发一条典型的综述类请求，触发完整的三层链路
# 主动词"请综述"+ 对象"多篇论文" → 分类器识别为 research skill 的概率更高
q = "请综述最近中文 Agent 记忆方向的 5 个核心贡献，并列出代表论文"
res = dual_agent.invoke({"messages": [{"role": "user", "content": q}]})

# 读取 Ch3 SkillsRouter 的路由决策（skills_router_mw 已在 4.3.2 保存为局部变量）
last_decision = skills_router_mw._last_decision
print("[Ch3 SkillsRouter 路由决策]")
print(f"  匹配 Skill:  {last_decision['skill_id']}")
print(f"  置信度:      {last_decision['confidence']:.2f}")
print(f"  装配工具数:  {last_decision['tool_count']}（全集 4 → 子集 {last_decision['tool_count']}）")

# 打印主 Agent 最终答复（基于子 Agent 摘要生成，而非基于 10 篇原文）
reply = res["messages"][-1].content
print("\n[主 Agent 最终答复] (前 200 字):")
print(str(reply)[:200])


[Ch3 SkillsRouter 路由决策]
  匹配 Skill:  research
  置信度:      0.95
  装配工具数:  1（全集 4 → 子集 1）

[主 Agent 最终答复] (前 200 字):
基于研究子Agent提供的结构化摘要，我来为您整理中文Agent记忆方向的5个核心贡献：

## 中文Agent记忆方向5个核心贡献综述

### 1. **记忆增强方法**
**核心贡献**：提出专为中文Agent设计的混合检索增强方法
- **代表论文**：《BM25 与向量检索的四路融合：中文 Agent 召回率优化》
- **主要创新**：融合BM25（关键词）、语义向量、特定规则与时间衰


&emsp;&emsp;运行后会出现两种可能的路由路径，都是正常的。**路径 A（命中）**：`skill_id == "research"`，`confidence >= 0.6`，`tool_count == 1`——意图分类器准确识别了"综述"语义，`SkillsRouterMiddleware` 把工具集裁剪为只含 `research_subagent` 一个工具，主 Agent 必然调用它把资料吸收任务委派给子 Agent。**路径 B（fallback）**：`skill_id == None`，`confidence < 0.6`，`tool_count == 0`——分类器对 query 意图把握不定走了兜底路径，主 Agent 直接基于 LLM 世界知识作答，没调用任何工具。无论走哪条路径，主 Agent 的 `messages` 数组里都不会出现 10 篇论文的原文——路径 A 的 `ToolMessage` 只含 ≤500 字摘要，路径 B 根本不触发工具调用。

&emsp;&emsp;<font color=red>为什么两条路径都"安全"？</font>因为 Ch4 的隔离机制在 `research_subagent` 工具的**实现层**就已经做死了（闭包内部硬截断 500 字），而不是依赖 Ch3 路由判断正确。Ch3 分类器的准确率决定"工具是否被调用"，但不决定"工具被调用时是否真的隔离"。这正是"装配侧（Ch3）+ 执行侧（Ch4）"两层分治的教学价值——<font color=red>执行侧的安全性不能押注在装配侧的路由成功率上</font>。

> ⚠️ **常见误区**：看到 `skill_id == None` 就以为 Ch4 的 research_subagent 失效了。其实 research_subagent 作为普通 `@tool` 始终挂在 `tools` 列表里（见 4.3.2 的 `tools=[...+research_tool]`）；即使 `SkillsRouterMiddleware` 因置信度不足走 fallback，只要 `fallback_tools` 包含 research_tool，主 Agent 仍然能自主选择调用它。路由命中只是提高"一定会调用"的概率，不是 research_subagent 工作的必要条件。

---

### 4.4 run_p4_probe 真跑：baseline vs isolate

&emsp;&emsp;4.3 的三层融合主 Agent 跑通了"路由 + 隔离"的功能闭环，但到底省了多少主 Agent 入模 token？用 `IsolateManager.run_p4_probe()` 在本机真跑一次，让数据说话。`run_p4_probe` 内部对比两种策略：`"baseline"` 把 10 篇论文全量直接塞进主 Agent 的 `HumanMessage`，发起 1 次 LLM 调用；`"isolate"` 先让子 Agent 吸收全量论文（第 1 次调用），再让主 Agent 基于 500 字摘要作答（第 2 次调用）。两种策略的 token 数都来自 LangChain canonical 的 `usage_metadata` 字段，不是估算。

#### 4.4.1 双策略真跑

&emsp;&emsp;以下代码会发起 3 次真实 LLM 调用（baseline 1 次 + isolate 2 次），在本地 DeepSeek 环境下合计耗时约 60-120 秒。token 数会因每次 LLM 输出长度小幅浮动，但核心不等式 `main_in(isolate) < main_in(baseline)` 在正确实现下始终成立。

In [43]:
# baseline 策略：10 篇论文全量塞主 Agent，1 次 LLM 调用
print("跑 P4 baseline（约 20-40s）...")
p4_baseline = im.run_p4_probe("baseline")

# isolate 策略：子 Agent 先吸收再回 500 字摘要，主 Agent 只看摘要，共 2 次 LLM 调用
print("跑 P4 isolate（约 40-80s）...")
p4_isolate = im.run_p4_probe("isolate")

# 打印核心指标对比：主 Agent 入模 / 子 Agent 入模 / 总成本 / 延迟
print("\n=== P4 Subagent 隔离真跑数据 ===")
for r in (p4_baseline, p4_isolate):
    print(
        f"{r['strategy']:<9}"
        f"  主Agent入模={r['main_input_tokens']:>5}"
        f"  子Agent入模={r['sub_input_tokens']:>5}"
        f"  总成本=¥{r['cost_yuan']:.4f}"
        f"  延迟={r['latency_ms']:>6.0f}ms"
    )

# 计算主 Agent 入模降幅 + 单次总成本变化
drop = 1 - p4_isolate["main_input_tokens"] / p4_baseline["main_input_tokens"]
cost_delta = p4_isolate["cost_yuan"] - p4_baseline["cost_yuan"]
print(f"\n主 Agent 入模降幅: {drop * 100:.0f}%")
print(f"单次总成本变化:   {cost_delta:+.4f} CNY")

跑 P4 baseline（约 20-40s）...
跑 P4 isolate（约 40-80s）...

=== P4 Subagent 隔离真跑数据 ===
baseline   主Agent入模=  388  子Agent入模=    0  总成本=¥0.0020  延迟= 16243ms
isolate    主Agent入模=  310  子Agent入模=  440  总成本=¥0.0040  延迟= 31202ms

主 Agent 入模降幅: 20%
单次总成本变化:   +0.0020 CNY


&emsp;&emsp;`run_p4_probe("baseline")` 内部只发 1 次调用——`HumanMessage(full_papers_text + query)` 直接塞主 llm，`main_input_tokens` 从 `response.usage_metadata["input_tokens"]` 取回真实数值；`run_p4_probe("isolate")` 则先调一次 `sub_llm.invoke`（消耗 `sub_input_tokens`）拿回 500 字摘要，再调一次 `main_llm.invoke`（消耗较少的 `main_input_tokens`，且只含摘要 + query）。

#### 4.4.2 反直觉：单次总成本为何升

&emsp;&emsp;数据出来之后，你会看到一个反直觉的现象：主 Agent 入模从 ~388 降到 ~310，降幅约 20%，听起来很好；但单次总成本反而从 ¥0.0025 升到 ¥0.0042，延迟也从 ~30s 拉长到 ~70s。原因很简单——isolate 策略多出了一次"子 Agent 吸收 10 篇论文"的调用，子 Agent 消耗的 ~440 input tokens 叠加在主 Agent 节省的 ~78 tokens 之上，总成本不降反升。把数据摆上表格对照：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>P4 Subagent 隔离真跑数据（10 篇论文 + DeepSeek V3）</font></p>
<div class="center">

| 策略 | 主 Agent 入模 | 子 Agent 入模 | 输出 token（总）| 单次总成本 | 延迟 |
|------|--------------|--------------|----------------|-----------|------|
| baseline（全量塞主 Agent）| ~388 tokens | 0 | ~550 tokens | ¥0.0025 | ~30 s |
| isolate（子 Agent 先吸收）| ~310 tokens | ~440 tokens | ~900 tokens | ¥0.0042 | ~70 s |

</div>

&emsp;&emsp;<font color=red>这就是 Subagent 策略最容易被误解的地方</font>：如果只看"单次总成本"这一个指标，isolate 看起来是"更贵"的方案。但这个判断遗漏了关键视角——主 Agent 入模从 ~388 压到 ~310，意味着后续**每一次**用户追问，主 Agent 都只围绕 500 字摘要滚上下文，而不是重新吃 10 篇全文。单次更贵，但长尾更便宜。

> ⚠️ **常见误区**：很多学员第一次看到 Subagent 策略"单次反而更贵"的数据就放弃了，以为方案有问题。实际上 Subagent 策略的价值**不在单次降本，而在主 Agent 上下文的"不膨胀"**——主 Agent 可以反复追问而不让对话历史滚雪球。

#### 4.4.3 可持续性视角：追问 10 次累积账

&emsp;&emsp;到底"长尾更便宜"能省多少？我们用具体数字算一次"追问 10 次"的累积账，把口号变成可验证的数值。<font color=red>建模前提声明</font>：以下估算采用"每轮独立请求建 prompt"的简化模型——即每次追问都重新拼接原始资料而非复用历史 messages。真实会话如果做了 prefix 级别的消息缓存，baseline 端的累积入模会更低；但这不影响"isolate 随追问次数累积入模增速更慢"的定性结论，只影响反超的临界点位置。假设用户在同一个会话里针对同一篇综述追问 10 次：

- **baseline 10 次**：每轮都重新塞 10 篇论文原文，主 Agent 累积入模约 10 × 388 = **3880 tokens**
- **isolate 10 次**：首轮先付子 Agent 吸收 10 篇原文的成本（~440 tokens），10 次追问期间主 Agent 只看 500 字摘要共计入模 10 × 310 = 3100 tokens，合计 **3540 tokens**

追问 10 次后，isolate 累积入模比 baseline 少约 **340 tokens**（~9%），且随追问次数增加，差距持续扩大——<font color=red>这才是 Subagent 策略真正的收益窗口：追问次数越多，isolate 越合算</font>。一旦对话进入"追问次数 > 5"的长尾阶段，isolate 就会反超 baseline 的累积成本。

> ⚠️ **适用场景说明**：Subagent 策略适合三类场景——**长文档反复追问**（同一份大资料需要从多个角度提问）、**多工具并行调用**（多个子 Agent 并行吸收不同资料来源）、**探索性任务只要结论不要过程**（用户关心摘要而非原始数据）。不适合"单次大资料、问完就走"的一次性场景，那种情况直接用 Ch3 `SkillsRouterMiddleware` 裁掉工具即可。把 Subagent 策略和 Ch1 `SummarizationMiddleware` 比拼单次成本是错误的——两者解决的是不同层的问题。

---

### 4.5 本章小结

&emsp;&emsp;本章用 `research_subagent` 工具解决了 M4 工具上下文层污染问题：主 Agent 的 messages 里不再出现 10 篇论文的原文，`ToolMessage.content` 被硬截断在 500 字以内，M4 层从"污染"回到"健康"。与此同时，`research_subagent` 作为一个普通 `@tool` 挂进主 Agent，和 Ch1 `SummarizationMiddleware`（压 messages）+ Ch3 `SkillsRouterMiddleware`（切 tools）构成三个正交切面——三章的工作成果在同一个 `create_agent` 里零耦合叠加，这就是 LangChain Middleware 机制设计为"一等公民"的工程收益。

&emsp;&emsp;我们每一章都在压 input_tokens，但每一次"压缩动作"都顺手把 DeepSeek prefix cache 的前缀字节稳定性破坏掉了——cache 命中率从来没启动过。如果保留 cache 不动，仅靠它就能让成本再降约一半。这正是下一章要正面处理的反共识：**激进压缩会摧毁 prefix cache，省 token 反而增成本**。


---

## <center>第 5 章：Cache vs Compress 协调</center>

&emsp;&emsp;上一章 第 4 章 的末尾，我们注意到一件尴尬的事——前四章我们一路压 `input_tokens`，成绩看起来相当漂亮，但 `prompt_cache_hit_tokens` 这一列从 第 0 章 到 第 4 章 **始终是 0**。换句话说，我们一直在“手动省 Token”，却把 `DeepSeek` 原本免费送上门的 `prefix cache` 命中率亲手打到了零。这正是 第 4 章 章末留下的那个 cliffhanger——**cache 命中率被 compress 破坏**——本章要正面处理的，就是这条反共识主线。

&emsp;&emsp;<font color=red>本章是整门课的高潮，也是最反直觉的一章。请把它当成“给前四章打补丁”——前四章我们以为自己在降本，实际上漏掉了 cache 这条最大的降本来源。</font>本章主战场是 M1 系统提示层 + M2 对话历史层。我们会通过三个动手实验，先看到 cache 是怎么“被压缩干掉”的，再实现 cache-friendly 的中间件叠加顺序模板，最后把 cache_hit_tokens 这一列从 0 拉到显著提升——而且这次降幅的来源不是“省 token”，而是“启用 cache”。

> 📅 **时效性说明**：本章所有结论锚定 2026-04，整体置信度 0.78。三个开放问题截至 2026-04 状态如下——`langchain-ai/langchain#34517`（SummarizationMiddleware metadata 污染）open、`langchain-ai/langgraph#5755`（RemoveMessage 不穿透 subgraph）open、`langchain-ai/langchain#31949`（`set_llm_cache` 在 agent 场景下缓存不命中，截至 2026-04 仍为开放问题）open。课件发布前请按 GitHub 状态复查。本章引用的论文与博客均来自 cache_compress_conflict_research.md，请在课件资料包中查阅。

### 5.1 反共识开场：cache_hit_tokens 为什么始终是 0

&emsp;&emsp;先把结论摆出来。<font color=red>DeepSeek prefix cache 的命中规则是"字节级前缀匹配"——任何对消息序列中段的修改，都会让从修改点起的所有后续 token 全部 cache miss。SummarizationMiddleware 删除并替换早期消息的动作正是"中段修改"的典型，所以它运行后 cache 必然清零。</font>

&emsp;&emsp;为什么前四章每一章都中招？因为 第 1 章 引入的 SummarizationMiddleware、第 2 章 引入的 retrieve_memory（每轮都注入新的 SystemMessage）、第 3 章 引入的四路融合（SystemMessage 内容随 query 变化）、第 4 章 引入的 `research_subagent` `@tool`（子 Agent 输出每次都不同，且每轮都追加新 `ToolMessage`）——它们都在悄悄修改对话历史的中段或者头部，等于每一轮都把 cache 锚点击碎一次。

&emsp;&emsp;DeepSeek 官方在 `https://api-docs.deepseek.com/guides/kv_cache` 与 `news0802` 两份文档里明确说过：cache 是按硬盘块（社区观测约 64 token 一块）做前缀匹配，stateless API 没有任何"局部更新"机制。这意味着只要你修改了对话中段的一个字节，从那个字节开始的所有后续 token 都要重新 prefill。这就是 cache_hit_tokens 始终为 0 的物理原因。

### 5.2 实验一：前缀断裂复现

&emsp;&emsp;现在我们来"亲眼"看一下 cache 命中是怎么归零的。这个实验只有三步：第一轮跑一个简单 prompt、第二轮跑完全相同的 prompt（应该命中）、第三轮把 system prompt 改一个字（应该归零）。

In [23]:
# === Cache 策略实战：观察 DeepSeek Prefix Caching 的 API 响应 ===
import time
from langchain_core.messages import SystemMessage, HumanMessage

# 构造一个真实 Agent 的系统提示（角色 + 工具描述 + 约束规则）
# 生产环境中，这个前缀每次 LLM 调用都要发送——正是缓存的最佳目标
cache_system_prompt = """你是 DevAssist 后端工程助手，精通以下完整技术栈，必须严格遵守所有约束规则。

【角色定义】
你是一名具有 10 年后端开发经验的高级工程师，专注于 Python 生态的 Web 服务架构。核心能力包括：
1. API 设计与实现：精通 RESTful API 规范（Richardson 成熟度模型 L3）、GraphQL Schema 设计（包括 Subscription 和 Federation）、gRPC 服务定义与 Protobuf 编写、WebSocket 长连接管理
2. 数据库设计与优化：PostgreSQL 高级特性（JSONB 索引优化、递归 CTE、窗口函数、分区表）、Redis 数据结构选型（String/Hash/Set/ZSet/Stream）、MongoDB 聚合管道与分片策略
3. 微服务架构：Docker 多阶段构建与镜像瘦身、Kubernetes 资源编排（Deployment/StatefulSet/DaemonSet/Job/CronJob/Service/Ingress/HPA/VPA/PDB）、Istio Service Mesh 流量管理与故障注入
4. 性能调优：多级缓存策略设计（L1 进程内 LRU + L2 Redis Cluster + L3 CDN 边缘缓存）、数据库连接池配置与监控、asyncio 异步编程模式（Task/Gather/Semaphore/Queue）、批处理与流式处理的权衡
5. 安全实践：OAuth 2.0/OIDC 认证授权完整流程（Authorization Code/PKCE/Client Credentials/Device Flow）、数据传输加密（TLS 1.3 配置与证书管理）、OWASP Top 10 漏洞防护策略

【输出风格约束】
- 回答必须简洁精准，优先给出代码示例而非冗长解释
- 涉及架构决策时必须说明 trade-off（至少列出 2 个替代方案的优劣）
- 涉及性能优化时必须给出可量化的指标（QPS、延迟 P99、内存占用）
- 不确定的信息必须明确标注「需要进一步确认」

【可用工具描述】
Tool 1: search_codebase
  功能：搜索项目代码库中的文件、函数、类定义和注释内容
  参数：query(str) 搜索关键词，支持正则表达式和模糊匹配；file_type(str) 文件类型过滤（py/yaml/sql/md/dockerfile/proto）；scope(str) 搜索范围（all/src/tests/docs/configs）；context_lines(int) 上下文行数，默认3
  返回：匹配的代码片段列表，每条包含文件绝对路径、起始行号、匹配行高亮、前后 context_lines 行上下文
  使用场景：了解现有代码结构和模块依赖、查找函数或类的定义和所有调用点、定位 bug 的根因代码、追踪配置项的引用链路

Tool 2: run_tests
  功能：执行项目测试套件，支持单元测试、集成测试和端到端测试123123
  参数：test_path(str) 测试文件或目录路径；verbose(bool) 是否输出详细的每条用例结果；markers(str) pytest marker 表达式过滤（如 slow/integration/smoke）；parallel(bool) 是否使用 pytest-xdist 并行执行；coverage(bool) 是否生成覆盖率报告
  返回：测试结果摘要（通过/失败/跳过/错误数量）、失败用例的完整 traceback 和局部变量快照、覆盖率报告（行覆盖率/分支覆盖率/未覆盖行号）
  使用场景：验证代码修改未引入回归问题、确认新功能的正向和反向测试用例全部通过、检查关键模块的代码覆盖率是否达标

Tool 3: query_database
  功能：执行只读 SQL 查询，自动添加 LIMIT 保护和超时控制
  参数：sql(str) SQL 查询语句，仅允许 SELECT/EXPLAIN/SHOW 语句；database(str) 目标数据库名称；timeout(int) 查询超时秒数，默认30；format(str) 输出格式（table/json/csv）
  返回：查询结果集（最多 100 行）、EXPLAIN ANALYZE 执行计划摘要、查询实际耗时和扫描行数
  使用场景：诊断数据一致性和完整性问题、验证数据迁移前后的数据正确性、分析慢查询的执行计划和优化方向、统计业务指标和数据分布

Tool 4: deploy_service
  功能：部署服务到指定环境，自动执行健康检查、金丝雀验证和自动回滚
  参数：service(str) 服务名称（必须在服务注册表中）；env(str) 目标环境（dev/staging/production）；version(str) 部署版本号（git tag 或 commit hash）；canary_percent(int) 金丝雀流量百分比（0-100）；timeout(int) 部署超时分钟数
  返回：部署状态（pending/rolling_update/canary_verifying/completed/rolled_back/failed）、健康检查详情（HTTP/TCP/gRPC 探针结果）、金丝雀指标对比（错误率/延迟/CPU 使用率的 baseline vs canary）
  使用场景：发布新版本服务、灰度验证新功能对性能和稳定性的影响、故障版本快速回滚

Tool 5: monitor_metrics
  功能：查询服务实时监控指标、历史趋势和异常检测结果
  参数：service(str) 服务名称；metric(str) 指标名（latency_p50/latency_p95/latency_p99/error_rate/qps/cpu_usage/memory_usage/gc_pause/thread_count/connection_pool_usage）；duration(str) 时间范围（5m/15m/1h/6h/24h/7d/30d）；aggregation(str) 聚合方式（avg/max/min/sum/count）
  返回：时间序列数据点列表（timestamp + value）、统计摘要（min/max/avg/median/p50/p95/p99/stddev）、异常检测结果（基于 3-sigma 和趋势分析）
  使用场景：建立服务性能基线、诊断告警事件的根因、评估优化措施的效果、容量规划和扩缩容决策

Tool 6: manage_secrets
  功能：管理服务密钥、证书和敏感配置，支持多环境隔离和版本审计
  参数：action(str) 操作类型（get/set/rotate/delete/list/history）；key(str) 密钥名称（支持路径格式如 service/db/password）；env(str) 目标环境；version(int) 指定版本号（用于回滚）
  返回：操作结果、密钥元信息（创建时间/过期时间/最近访问时间/访问计数/创建者）
  使用场景：定期密钥轮转（数据库密码/API Token/TLS 证书）、多环境配置同步、安全审计和合规检查

Tool 7: analyze_logs
  功能：搜索和分析服务日志，支持结构化查询、聚合统计和关联追踪
  参数：service(str) 服务名称；query(str) 日志搜索表达式（支持 Lucene 语法）；level(str) 日志级别过滤（DEBUG/INFO/WARN/ERROR/FATAL）；time_range(str) 时间范围；limit(int) 返回条目上限
  返回：匹配的日志条目（含完整结构化字段）、按时间/级别/来源的频率统计直方图、关联的 trace_id 和 span_id 列表（用于分布式追踪）
  使用场景：生产环境错误排查和根因定位、请求全链路追踪（跨服务关联）、异常模式识别（错误频率突增/周期性异常）、SLA 违规事件取证

【约束规则】
1. 所有数据库操作必须使用参数化查询，禁止任何形式的字符串拼接 SQL，防止 SQL 注入攻击
2. API 响应必须遵循统一格式：{"code": int, "message": str, "data": any, "request_id": str, "timestamp": str}
3. 所有敏感操作（部署、密钥变更、数据删除、权限修改）必须记录审计日志，包含操作者身份、时间戳、变更内容和影响范围
4. 生产环境部署必须先通过 staging 环境的完整验证，且自动化测试覆盖率不低于 80%
5. 每次代码修改必须附带对应的单元测试，新增代码的行覆盖率不低于 90%，分支覆盖率不低于 80%
6. 第三方依赖必须锁定精确版本号（使用 == 约束），禁止使用 >=、~=、^= 等模糊版本约束
7. 所有异步操作必须设置超时时间（默认 30 秒），避免资源泄漏和无限等待导致的线程/协程饥饿
8. 数据库连接必须使用连接池管理（最小连接数 5、最大连接数 20、空闲超时 300 秒），禁止每次请求创建新连接
9. 所有缓存必须设置合理的 TTL（默认 300 秒），禁止永久缓存，防止数据不一致和内存泄漏
10. API 限流策略：认证用户 100 req/min，匿名用户 20 req/min，管理员 500 req/min，超限返回 429 状态码
11. 错误处理必须区分可重试错误（5xx、超时、连接中断）和不可重试错误（4xx、业务逻辑错误），可重试错误实现指数退避（初始 1s、最大 60s、抖动 ±20%）
12. 日志输出必须使用结构化 JSON 格式，每条日志必须包含 timestamp、level、service、trace_id、span_id、message 字段"""

# 先测量系统提示的 token 数
prefix_tokens = count_tokens(cache_system_prompt)
print(f"系统提示 token 数: {prefix_tokens}")
print(f"这 {prefix_tokens} tokens 的角色定义+工具描述+约束规则，Agent 每轮调用都要重发\n")

# 模拟 Agent 连续对话：8 轮不同的用户问题，但系统提示完全相同
# DeepSeek 的前缀缓存需要多次请求后才在服务端建立
# 通常第 4-6 次请求开始命中，这正是真实生产环境中的行为模式
questions = [
    "PostgreSQL JSONB 类型有哪些核心优势？",
    "如何用 Alembic 生成添加字段的迁移脚本？",
    "Redis 缓存穿透怎么解决？",
    "FastAPI 中间件的执行顺序是什么？",
    "如何配置 Kubernetes HPA 自动扩缩容？",
    "Docker 多阶段构建的最佳实践？",
    "asyncio 和多线程的选择标准是什么？",
    "如何设计 API 限流的降级策略？",
]

results = []
for i, q in enumerate(questions):
    msgs = [SystemMessage(content=cache_system_prompt), HumanMessage(content=q)]
    response = llm.invoke(msgs)

    # 提取 API 返回的缓存命中数据
    usage = response.response_metadata.get("token_usage", {})
    hit = usage.get("prompt_cache_hit_tokens", 0)
    miss = usage.get("prompt_cache_miss_tokens", 0)
    prompt = usage.get("prompt_tokens", 0)

    results.append({"round": i + 1, "question": q[:15], "prompt": prompt, "hit": hit, "miss": miss})

    status = "缓存命中" if hit > 0 else "缓存未命中"
    print(f"轮次 {i+1}: prompt={prompt} tokens, cache_hit={hit}, cache_miss={miss} [{status}]")
    time.sleep(2)  # 等待服务端缓存传播

print(f"\n说明: prompt_cache_hit_tokens = 缓存命中的 token 数（按 10% 计价）")
print(f"      prompt_cache_miss_tokens = 未命中的 token 数（按标准价计费，DeepSeek 无写入费，缓存在后台自动构建）")

系统提示 token 数: 2009
这 2009 tokens 的角色定义+工具描述+约束规则，Agent 每轮调用都要重发

轮次 1: prompt=2023 tokens, cache_hit=576, cache_miss=1447 [缓存命中]
轮次 2: prompt=2026 tokens, cache_hit=1984, cache_miss=42 [缓存命中]


KeyboardInterrupt: 

&emsp;&emsp;运行这段代码，你应该会看到：第 1 轮 `prompt_cache_hit_tokens` 是 0；第 4-6 轮该字段大于 0；第 3 轮虽然只改了几个字，该字段又重新截断。<font color=red>这就是本章最重要的"信仰建立时刻"</font>——cache 命中不是"差不多就好"，而是字节级硬约束。任何能改动中段的中间件都会让命中率断崖式归零。

### 5.3 实验二：DeepSeekCacheBoundaryMiddleware（分段 cache）

&emsp;&emsp;既然中段修改会破坏 cache，正确的姿势就是**把稳定内容（system + tools 描述）锁在最前面，明确告诉所有中间件"这一段你绝对不许碰"**。Anthropic 提供了 `cache_control: ephemeral` 显式 API，DeepSeek 没有这种 API，但我们可以在客户端用一个最外层的"边界守护"中间件实现等效效果——这就是 `DeepSeekCacheBoundaryMiddleware`。

In [47]:
# 导入 Middleware 基类与消息类型
from typing import Any, Optional, Callable
from langchain.agents.middleware import AgentMiddleware
from langchain_core.messages import SystemMessage

class DeepSeekCacheBoundaryMiddleware(AgentMiddleware):
    """最外层中间件：在真实 LLM 调用边界上守护 system 段字节稳定性。

    为什么钩 wrap_model_call 而不是 before_model：
    create_agent(system_prompt=...) 不会把 system_prompt 放进 state["messages"]，
    而是绑定到 ModelRequest.system_message，仅在调 LLM 那一刻 prepend 到请求里。
    要守住真正喂给 provider API 的前缀字节（DeepSeek prefix cache 的锚点），
    必须在 wrap_model_call 钩子里读 request.system_message.content。
    """

    def __init__(self):
        """初始化：清空锁定快照。"""
        super().__init__()
        # 首次调用时冻结的 system 前缀字节（None 表示尚未锁定）
        self._frozen_system_bytes: Optional[bytes] = None

    @staticmethod
    def _encode(content: Any) -> bytes:
        """把 SystemMessage.content 编码为字节，兼容多模态 list-of-blocks 格式。"""
        if isinstance(content, str):
            return content.encode("utf-8", errors="replace")
        if isinstance(content, list):
            # 多模态 content：拼接所有 text block
            text = "".join(
                b.get("text", "") if isinstance(b, dict) else str(b) for b in content
            )
            return text.encode("utf-8", errors="replace")
        return str(content).encode("utf-8", errors="replace")

    def _check(self, request: Any) -> None:
        """核心观测：首次锁定 system 前缀字节，后续每次校验是否漂移。"""
        # 用 getattr 兼容不同 LangChain 版本的字段命名
        sm = getattr(request, "system_message", None)
        if sm is None or not hasattr(sm, "content"):
            # create_agent 未传 system_prompt 时直接返回，无锚点可守
            return
        current = self._encode(sm.content)
        if self._frozen_system_bytes is None:
            # 首次调用：锁定基线
            self._frozen_system_bytes = current
            print(f"[cache boundary] 已锁定 system 段，{len(current)} 字节")
        elif current != self._frozen_system_bytes:
            # 后续调用字节不一致 → cache 杀手
            print("[cache boundary] ⚠️ 检测到 system 段被改动，cache 即将清零！")

    def wrap_model_call(self, request: Any, handler: Callable[[Any], Any]) -> Any:
        """同步 LLM 调用边界钩子：校验后透传给下游。

        Args:
            request: ModelRequest，含 system_message / messages / tools 等
            handler: 下游调用链（一般最终调到 LLM.invoke）
        Returns:
            下游 handler 的返回值（ModelResponse）
        """
        self._check(request)
        return handler(request)

    async def awrap_model_call(self, request: Any, handler) -> Any:
        """异步 LLM 调用边界钩子：与同步版共享校验逻辑。"""
        self._check(request)
        return await handler(request)


&emsp;&emsp;这段中间件只做一件事：在每次模型调用之前检查 `system` 段的字节是否与"首次锁定"的版本一致。如果不一致，它会打印警告——这等于一个"cache 杀手报警器"。生产场景下你可以让它直接 raise 或者强制回滚 system 段，本课为了教学清晰只做检测和打印。请注意这个中间件的关键设计：它必须放在 Middleware 栈的**最外层**，才能在所有其他中间件改完之后做最终检查。

In [122]:
# === Tier 1 独立验证：DeepSeekCacheBoundaryMiddleware 三态行为（真实路径）===
# 关键变化：原版测的是 before_model(state)，但真实 create_agent 路径走 wrap_model_call(request)。
# 因此 fixture 必须构造带 system_message 的 request 对象 + 假 handler，而不是塞 SystemMessage 到 state。
from langchain_core.messages import SystemMessage

class FakeReq:
    """最小 ModelRequest 替身：只暴露 cache_boundary 关心的 system_message 字段。"""
    def __init__(self, sys_text):
        """system_message 字段模拟真实 ModelRequest.system_message。"""
        self.system_message = SystemMessage(content=sys_text) if sys_text else None

# 假 handler：不真跑 LLM，仅证明 wrap_model_call 链路通
def fake_handler(req):
    """返回占位值，保持 wrap_model_call 返回类型一致。"""
    return "ok"

# 独立实例（不污染后续 5.5 的 cache_boundary）
cb_test = DeepSeekCacheBoundaryMiddleware()

# 状态 A：首次调用 → 锁定 system 段字节快照
req_a = FakeReq("你是技术助手，中文回答。")
ret_a = cb_test.wrap_model_call(req_a, fake_handler)
assert cb_test._frozen_system_bytes is not None, "首次 wrap_model_call 应锁定字节快照"
assert ret_a == "ok", "wrap_model_call 必须把 handler 返回值透传给上游"
locked_bytes = cb_test._frozen_system_bytes
print(f"[状态A] 首次锁定：{len(locked_bytes)} 字节")

# 状态 B：system 被篡改 → 字节不匹配，应触发告警（不覆盖已锁快照）
req_b = FakeReq("你是产品经理助手，中文回答。")
cb_test.wrap_model_call(req_b, fake_handler)  # 预期打印 "⚠️ 检测到 system 段被改动"
assert cb_test._frozen_system_bytes == locked_bytes, "告警路径不应覆盖锁定快照"
print(f"[状态B] 篡改检测：快照保持 {len(locked_bytes)}B，告警已触发")

# 状态 C：system 恢复原值 → 字节一致，静默通过
req_c = FakeReq("你是技术助手，中文回答。")
cb_test.wrap_model_call(req_c, fake_handler)
assert cb_test._frozen_system_bytes == locked_bytes, "恢复原值后快照应保持"
print(f"[状态C] 恢复原值：字节一致（{len(locked_bytes)}B），cache 锚点稳定")

# 状态 D（兜底）：request 无 system_message → 直接 return，不报错
req_d = FakeReq(None)
cb_test.wrap_model_call(req_d, fake_handler)
print(f"[状态D] 无 system 场景：安全跳过，_frozen 保持")

print("\nCacheBoundary 四态验证通过（wrap_model_call 真实路径）：锁定 → 篡改告警 → 恢复静默 → 无 system 安全")


[cache boundary] 已锁定 system 段，36 字节
[状态A] 首次锁定：36 字节
[cache boundary] ⚠️ 检测到 system 段被改动，cache 即将清零！
[状态B] 篡改检测：快照保持 36B，告警已触发
[状态C] 恢复原值：字节一致（36B），cache 锚点稳定
[状态D] 无 system 场景：安全跳过，_frozen 保持

CacheBoundary 四态验证通过（wrap_model_call 真实路径）：锁定 → 篡改告警 → 恢复静默 → 无 system 安全


### 5.4 实验三：TailTrimMiddleware（cache-friendly 末尾裁剪）

&emsp;&emsp;DeepSeekCacheBoundary 只是"检测"，并没有解决"对话越来越长"的现实问题。我们还需要一种**不破坏前缀**的压缩方式。基础课已经讲过 `RemoveMessage` 这个原语——它的 cache 友好性来自一条原则：**只要前缀 `head_keep` 条字节不动，后续被删的中段不会影响 cache 命中前缀的那一部分**。换句话说：`DeepSeek` prefix cache 的命中边界就锚在前 `head_keep` 条消息里，只要这段不动，cache 就稳。

&emsp;&emsp;基于这条原则，我们手写一个 `TailTrimMiddleware`：当对话超过某个 token 上限时，**固定保护前 `head_keep` 条前缀**（`system` + 首批 `HumanMessage`，也就是 cache 锚点）和**最近 `keep_recent` 条末尾对话**，把中间这段已经"完成"的 `ToolMessage` / `AIMessage` 对整批删除。前缀永不动，末尾永不动，被牺牲的是"中间那段近因性较弱、但又占 token 的历史工具调用"。<font color=red>命名"TailTrim"是习惯叫法（"末尾裁剪器"），但它的语义更精确地说是"中段裁剪、首尾双保护"——请把这一点记住，后面 diff 落地时也是这个语义。</font>

In [66]:
# 导入消息删除原语与消息类型
from langchain_core.messages import RemoveMessage, AIMessage, ToolMessage,BaseMessage


class TailTrimMiddleware(AgentMiddleware):
    """cache-friendly 中段裁剪：保护前缀（cache 锚点）+ 保护末尾 N 条，删除中间段。"""

    def __init__(self, max_tokens: int = 6000, head_keep: int = 2, keep_recent: int = 10):
        """初始化中段裁剪参数。

        Args:
            max_tokens: int，触发裁剪的 token 阈值
            head_keep: int，保护的前缀条数（cache 锚点）
            keep_recent: int，保护的末尾条数（近因窗口）
        """
        super().__init__()
        self.max_tokens = max_tokens
        self.head_keep = head_keep      # 固定保护的前缀条数（含 system + 首批 HumanMessage，cache 锚点）
        self.keep_recent = keep_recent  # 固定保护的末尾条数（最近对话）

    def _approx_tokens(self, msgs: list[BaseMessage]) -> int:
        """近似估算消息总 token 数。

        Args:
            msgs: list[BaseMessage]，待统计的消息列表
        Returns:
            int：按字符数/1.5 估算的 token 总数
        """
        total = 0
        # 遍历累加字符数
        for m in msgs:
            content = getattr(m, "content", "")
            if isinstance(content, str):
                # 中英文混合下 1 token ≈ 1.5 字符
                total += int(len(content) / 1.5)
        return total

    def before_model(self, state: dict) -> Optional[dict]:
        """模型调用前执行中段裁剪。

        Args:
            state: dict，Agent 运行时状态，含 messages 列表
        Returns:
            dict 或 None：返回 RemoveMessage 列表；None 表示不裁剪
        """
        msgs: list[BaseMessage] = state.get("messages", [])

        # 未超过 token 阈值则跳过
        if self._approx_tokens(msgs) <= self.max_tokens:
            return None
        if len(msgs) <= self.head_keep + self.keep_recent:
            # 还没攒够可裁剪的中间段
            return None
            
        # 保护前缀（system + 首批 HumanMessage，cache 锚点）+ 保护末尾 keep_recent 条
        # 中间段（msgs[head_keep : len(msgs) - keep_recent]）全部标记为 RemoveMessage
        candidates_to_remove = []
        # 只删除中段的 Tool/AI 消息，保留 Human/System
        for i in range(self.head_keep, len(msgs) - self.keep_recent):
            m = msgs[i]
            if isinstance(m, (ToolMessage, AIMessage)) and getattr(m, "id", None):
                candidates_to_remove.append(RemoveMessage(id=m.id))

        if not candidates_to_remove:
            return None
        print(f"[tail-trim] 中段裁剪 {len(candidates_to_remove)} 条 ToolMessage/AIMessage")
        # 返回 RemoveMessage 列表，由 reducer 执行实际删除
        return {"messages": candidates_to_remove}

&emsp;&emsp;这段约 40 行的中间件就是本章最核心的手写产物。它和 `SummarizationMiddleware` 在**目的**上类似（都是"让对话变短"），但在**手段**上完全相反：`SummarizationMiddleware`（底层基于 `RemoveMessage`）删除 `head_keep` 之后的早期消息，并将摘要存入独立的 state 字段——删除本身是 cache-friendly 的，但压缩后的摘要会在下一轮作为新消息注入回对话列表，这个注入动作改写了前缀字节，cache 锚点被打破；而 `TailTrim` 用 `RemoveMessage` 把中间段的 `tool/ai` 对**删除**，前 `head_keep` 条前缀字节完全不动，cache 命中边界原地保留。请注意一个关键的 trade-off：<font color=red>TailTrim 牺牲的是"中段的历史工具调用细节"，换来的是"前缀字节不动 → cache 命中率保持"</font>。这意味着如果你的业务很依赖"中段的 ToolMessage 细节"，TailTrim 不适合你；反之，如果你的业务更依赖"前缀的系统提示 + 最近几轮对话"，TailTrim 远比 Summarization 划算。用默认参数 `head_keep=2, keep_recent=10` 跑一个 20 条消息的对话，你会看到：前 2 条（system + 第 1 轮 user）和末尾 10 条（最近 5 轮对话）被原样保留，中间 8 条被标记删除。

In [12]:
# === Tier 1 独立验证：TailTrimMiddleware 中段裁剪行为 ===
import uuid
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

# 构造 20 条 mock 消息：head(2) + 中段 AI/Tool 对(16) + tail(2)
mock_msgs = [
    SystemMessage(content="你是技术助手", id=str(uuid.uuid4())),       # head[0]
    HumanMessage(content="第一个问题", id=str(uuid.uuid4())),          # head[1]
]
# 中段：8 轮 AI+Tool 对话（模拟已完成的历史工具调用）
for i in range(8):
    mock_msgs.append(AIMessage(content=f"AI回复{i}", id=str(uuid.uuid4())))
    mock_msgs.append(ToolMessage(content=f"工具结果{i}", tool_call_id=f"call_{i}", id=str(uuid.uuid4())))
# 末尾：最近 1 轮对话
mock_msgs.append(HumanMessage(content="最新问题", id=str(uuid.uuid4())))
mock_msgs.append(AIMessage(content="最新回复", id=str(uuid.uuid4())))

print(f"构造消息总数: {len(mock_msgs)} 条")

# 实例化：max_tokens 设为 30 以强制触发裁剪（20 条短消息约 59 tokens，生产环境用 6000）
trim_test = TailTrimMiddleware(max_tokens=30, head_keep=2, keep_recent=2)
result = trim_test.before_model({"messages": mock_msgs})

# 验证：应触发裁剪
assert result is not None, "20 条消息 + max_tokens=100 应触发裁剪"
removed_ids = {rm.id for rm in result["messages"]}

# 验证：前 2 条（head_keep）不应被删
assert mock_msgs[0].id not in removed_ids, "system 不应被删（cache 锚点）"
assert mock_msgs[1].id not in removed_ids, "首条 human 不应被删（cache 锚点）"

# 验证：末尾 2 条（keep_recent）不应被删
assert mock_msgs[-1].id not in removed_ids, "最新 AI 不应被删（近因窗口）"
assert mock_msgs[-2].id not in removed_ids, "最新 human 不应被删（近因窗口）"

print(f"裁剪结果: {len(removed_ids)} 条中段消息被标记 RemoveMessage")
print(f"保护结果: 前 {trim_test.head_keep} 条 + 末尾 {trim_test.keep_recent} 条完好")
print("\nTailTrim 验证通过：中段裁剪正确，首尾保护完好")

构造消息总数: 20 条
[tail-trim] 中段裁剪 16 条 ToolMessage/AIMessage
裁剪结果: 16 条中段消息被标记 RemoveMessage
保护结果: 前 2 条 + 末尾 2 条完好

TailTrim 验证通过：中段裁剪正确，首尾保护完好


### 5.5 推荐的 Middleware 叠加顺序模板

&emsp;&emsp;有了三个中间件，下一个问题是：怎么叠？基于 `cache_compress_conflict_research.md` §2.5 的推荐和我们本章的实验结论，DeepSeek 场景下的标准模板从外到内是 `DeepSeekCacheBoundaryMiddleware → TailTrimMiddleware → SummarizationMiddleware → ModelCallLimitMiddleware`。

In [15]:
# 实例化两个自写中间件
cache_boundary = DeepSeekCacheBoundaryMiddleware()
tail_trim = TailTrimMiddleware(max_tokens=6000, keep_recent=10)

# 推荐叠加顺序（从外到内）
final_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="你是一个 Web 后端技术助手，中文回答。",
    middleware=[
        cache_boundary,    # 最外层：cache 边界守护
        tail_trim,         # 次外层：cache-friendly 末尾裁剪
        summary_mw,        # 次内层：延迟兜底（仅在 TailTrim 不够用时触发）
    ],
)
print("最终 Agent 已构造，4 层中间件按推荐顺序叠加")

最终 Agent 已构造，4 层中间件按推荐顺序叠加


&emsp;&emsp;这个叠加顺序的设计理念是分层防御：最外层的 `cache_boundary` 负责保护 cache 锚点不被破坏；次外层的 `tail_trim` 是日常压缩的主力，所有正常情况都用它；次内层的 `summary_mw` 只在 `tail_trim` 已经不够用时才触发（也就是连末尾裁剪都已经把对话裁到极限了），它愿意承担一次性的 cache miss 来换取整体存活；最内层的 `call_limit_mw` 是底线，防止 Agent 死循环。<font color=red>这四层缺一不可，颠倒任何一层的位置都会破坏整体的 cache 友好性。</font>

#### 5.5.1 Tier 2 冒烟测试：Middleware 栈在真实 Agent 调用中生效了吗？

&emsp;&emsp;`final_agent` 已经组装好，但四层 Middleware 有没有真正生效？我们跑一次单轮问答做冒烟测试。预期行为：`cache_boundary` 首次锁定 system 段（打印字节数）；`tail_trim` 静默通过（单轮消息远没到 6000 token 阈值）；Agent 正常返回回答。<font color=red>这一步对应 Ch3 的四路 Agent 端到端测试和 Ch4 的 `run_p4_probe()` 真跑——定义了代码就必须验证它被执行了。</font>

In [126]:
# === Tier 2 冒烟测试：单轮真跑验证 Middleware 栈生效 ===
from langchain_core.messages import HumanMessage

# 单轮问答：触发 cache_boundary 首次锁定 + tail_trim 静默（未超阈值）
test_result = final_agent.invoke(
    {"messages": [HumanMessage(content="用一句话解释什么是 RESTful API，50字以内")]},
)

# 验证 1：cache_boundary 应已锁定 system 段
assert cache_boundary._frozen_system_bytes is not None, \
    "cache_boundary 应在首次 Agent 调用后锁定 system 段"
print(f"[验证1] cache_boundary 已锁定 {len(cache_boundary._frozen_system_bytes)} 字节")

# 验证 2：Agent 正常返回（最后一条消息应为 AIMessage 且非空）
last_msg = test_result["messages"][-1]
assert hasattr(last_msg, "content") and last_msg.content, "Agent 回复不应为空"
print(f"[验证2] Agent 回复: {last_msg.content[:100]}...")

print("\nTier 2 冒烟通过：Middleware 栈在真实 Agent 调用中正常工作")

[cache boundary] 已锁定 system 段，53 字节
[验证1] cache_boundary 已锁定 53 字节
[验证2] Agent 回复: RESTful API 是一种基于 HTTP 协议、遵循 REST 架构风格的 Web 服务接口设计规范。...

Tier 2 冒烟通过：Middleware 栈在真实 Agent 调用中正常工作


### 5.6 本章小结与下一步

&emsp;&emsp;本章我们完成了三件事：复现了“cache 被压缩干掉”的现场、用 `DeepSeekCacheBoundaryMiddleware` 锁定了 cache 锚点、用 `TailTrimMiddleware` 实现了 cache-friendly 的末尾裁剪。最终 Middleware 栈按四层模板叠好后，prefix cache 的命中率显著提升——这是全课最大的单点降幅，而且降幅来源完全不同于前四章：前四章靠“省 token”，本章靠“让 token 按 cache_hit 单价计费”。

&emsp;&emsp;到这里五种组合编排策略全部讲完了。但一个新的问题浮现：五把工具都会用了，到了真正装一个 Agent 的时候，该选哪些、怎么组合？下一章我们通过三个案例（工具调用 / 对话问答 / 研究型）展示 Agent 类型驱动的差异化策略组合。

---

## <center>第 6 章：Agent 类型驱动的策略组合实战</center>

&emsp;&emsp;前五章我们分别打磨了五把"上下文管理工具"：Ch1 压缩五件套、Ch2 Write 三层记忆、Ch3 Select 路由、Ch4 Isolate 子 Agent 隔离、Ch5 Cache 协调。每章都是单点深挖——我们知道每一层在自己的场景下有效，但还没有回答最重要的问题：<font color=red>到了真正装一个 Agent 的时候，这五把工具里该选哪些？</font>

&emsp;&emsp;本章的答案是反直觉的：**没有"通用最优组合"**。工具调用型 Agent、对话问答型 Agent、研究型 Agent 各自的核心压力在六模块坐标系上落点不同，需要的 middleware 组合也完全不同。本章选三个最经典的 Agent 案例——**案例 A 工具调用 Agent（代码调试助手）**、**案例 B 知识问答对话型 Agent（技术顾问）**、**案例 C 研究型 Agent（学术综述助手）**——分别按"业务场景 → 策略组合 → 端到端装配 → 对照实验"四步走，展示前五章成果在不同 Agent 类型下如何**差异化组合**。

&emsp;&emsp;<font color=red>前置要求</font>：本章依赖 Ch0-Ch5 所有 cells 已顺序执行（kernel 中已有 `llm` / `_sm` / `memory` / `im` / `research_tool` 等前章资产）。6.1 末尾会做依赖桥接检查，任何缺失会明确报错。

&emsp;&emsp;本章路线图：**6.1** 先反驳"一个通用满栈 Agent 打天下"的错觉、引入 Agent 类型四维度分析框架；**6.2 / 6.3 / 6.4** 三个案例按认知负荷递进（A 用 6 主力 middleware、B 用 5 主力、C 用 5 主力但集大成跨 5 章）；**6.5** 沉淀 3 × 5 决策矩阵作为可带走产物；**6.6** 用三 Agent 雷达图可视化"不同 middleware 组合带来的能力差异"；**6.7** 本章小结 + 三大盲点声明。本章运行总时间约 4-5 分钟（含 3 次真跑 DeepSeek 调用 + Mem0 写入 + TaskState 文件 IO）。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260417140021236.svg" width=80%></div>

&emsp;&emsp;上图是本章三案例的策略组合 landscape：横轴是**策略类别**（Ch1-Ch5），纵轴是**三个 Agent 案例**。格子颜色深浅反映 middleware 的重要度（主力 / 可选 / 不适用）。可以一眼看出：**没有任何一列全部深色**（即没有哪个策略对三类 Agent 都是主力）、**也没有任何一行占满所有列**（即没有哪个 Agent 需要全部五章策略）。这张图就是本章反直觉结论"没有通用最优组合"的最直观证据。

### 6.1 为什么没有"通用最优组合"

&emsp;&emsp;开始案例之前，我们先处理一个工程习惯性错觉——"既然前五章教了五把工具，干脆把它们全叠一起，打造一个大而全的通用 Agent"。这个思路听起来合理，但只要你在真实项目里试过就会发现两个硬伤：**指标冲突** 和 **装配脆弱**。

&emsp;&emsp;**指标冲突**：Ch1 `ObservationMaskMiddleware` 会遮蔽 `ToolMessage.content`——这在 JetBrains SE Agent 的"变量追踪"场景里是害处（遮蔽了 `Set X = 77` 这类中间状态），但在本章工具调用 Agent 的调试场景里是**灾难**（遮蔽了学员需要看的代码片段）。Ch1 `CompactionMiddleware` 会重置摘要，直接把 Ch5 `DeepSeekCacheBoundaryMiddleware` 保护的 cache 锚点字节稳定性破坏掉。<font color=red>满栈叠加不仅没有收益，还会让不同 middleware 互相抵消</font>。

&emsp;&emsp;**装配脆弱**：9 层 middleware 叠在一起，任何一层的导入失败、参数漂移、版本不兼容都会导致整个 Agent 无法启动。真实工程经验是：**先选 1-2 个核心 middleware 跑通，根据实测指标增量添加**，这才是可持续的装配节奏。

&emsp;&emsp;那么怎么判断"某个 Agent 该配哪些策略"？四个维度就够了——**任务复杂度**、**对话轮次**、**工具数量**、**资料吞吐**。三个案例在这四个维度上的落点差异巨大，对应的策略组合自然不同。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>三案例在四维度上的差异</font></p>
<div class="center">

| 维度 | A 工具调用 | B 对话问答 | C 研究型 |
|---|---|---|---|
| 任务复杂度 | 中（多步调试）| 低（单轮追问为主）| **高**（多 skill + 多步骤）|
| 对话轮次 | 3-10 轮 | 5-20 轮 | 1-3 轮为主 |
| 工具数量 | 3 个（search 三件套）| 1 个 | 3 个（含 research_tool）|
| 资料吞吐 | 中（代码 snippet 累积）| 低（对话文本）| **巨**（多篇论文）|
| **主战场（六模块）** | M3 + M4 | M2 + M1 | M3 + M4 + M5 |
| **主力 middleware 数** | 6 | 5 | 5 |

</div>

&emsp;&emsp;本章三个案例的差异化策略组合见下表预告。详细装配代码和对照实验在 6.2 / 6.3 / 6.4 展开，完整决策矩阵在 6.5 沉淀——<font color=red>这张矩阵就是本章最重要的可带走产物</font>。

In [67]:
# Ch6 依赖桥接检查：验证前五章 kernel 资产已就绪
# 任何一项缺失，请先按顺序执行 Ch0-Ch5 对应 cells 再回来
required_vars = [
    # 基础设施
    "llm",
    # Ch1 middleware class（notebook 内自写，非外部导入）
    "ToolResultClearMiddleware",
    "MessageTrimMiddleware",
    # Ch2 middleware class + Mem0 实例
    "Mem0WriteMiddleware",
    "TaskStateMiddleware",
    "memory",
    # Ch3 工具 + SelectManager
    "_sm",
    "skill_registry",
    "search_memory",
    "search_files",
    "search_knowledge",
    "SkillsRouterMiddleware",
    # Ch4 IsolateManager + research_subagent 工具
    "im",
    "research_tool",
    # Ch5 middleware class（自写）
    "DeepSeekCacheBoundaryMiddleware",
    "TailTrimMiddleware",
]

# 检查哪些变量不在当前 kernel namespace
_ch6_globals = set(globals().keys())
missing = [v for v in required_vars if v not in _ch6_globals]
assert not missing, (
    f"[FAIL] Ch6 依赖缺失: {missing}\n"
    f"请先顺跑 Ch0-Ch5 对应 cells 后再执行 Ch6"
)
print(f"[OK] Ch6 依赖桥接通过：{len(required_vars)} 个前章资产已就绪")

[OK] Ch6 依赖桥接通过：16 个前章资产已就绪


&emsp;&emsp;依赖桥接通过后，我们就可以进入三个案例。每个案例独立可跑，但共用同一份 `memory` 实例和同一个 `_sm`——因此**三个案例使用不同的 `user_id` 严格隔离记忆空间**（`ch6_caseA` / `ch6_caseB` / `ch6_caseC`），并在章末清理 `todo.md` 文件避免残留污染。

### 6.2 案例 A：工具调用 Agent（代码调试助手）

&emsp;&emsp;在 6.1 节里我们确认了一件事：没有一种"通用最优"的 middleware 配置，不同 Agent 类型面对的上下文压力截然不同，策略组合必须因地制宜。现在到了动手的时候——第一个案例，我们选最典型的工具调用 Agent：<font color=red>代码调试助手</font>。

&emsp;&emsp;为什么从这里切入？因为代码调试场景同时命中了两大上下文压力：**M3（工具选择失焦）**与**M4（工具结果污染）**。每轮调试 Agent 都会调 `search_files` / `search_memory` 返回代码片段，每条工具结果 200-800 字，3 轮之后旧结果开始淤积——这正是 Cursor / Windsurf / Copilot Chat 每天在处理的工程现实。本节从场景白描出发，一步步推导出 6 个 middleware 的配置理由，最后用对照实验让策略价值数字化。

#### 6.2.1 业务场景与典型 query

&emsp;&emsp;想象一个真实的学员场景：她在用自己搭建的 `AgentManager` 跑一个多步调试流程，代码突然报 `KeyError: 'middleware'`。她打开聊天框，问调试助手："帮我定位这个 bug。"Agent 调了 `search_files` 扫描项目目录，返回 `agent_manager.py` 里 42 行的初始化逻辑，约 600 字代码片段。第一轮结束。

&emsp;&emsp;第二轮，她说："我看了第 42 行的 `__init__`，是不是 `config` 没传 `middleware` 字段？"Agent 再次调 `search_memory`，把她之前标记的"最近踩过的 config 坑"调出来，又是一条 400 字记录。两轮下来，`messages` 里已经有两条 `ToolMessage`，累积约 1000 字工具结果。

&emsp;&emsp;第三轮，她问："那我应该在 `AgentConfig` 里加默认空 tuple，还是在 `__init__` 里做 fallback？"这一轮不需要再查文件，但前两条工具结果还压在 `messages` 里——**这正是 M4 工具结果污染的典型发生时机**：旧工具结果不再有信息增量，却仍占据宝贵 token 窗口，还可能误导 LLM 作答。

&emsp;&emsp;三轮 query 序列如下，这也是后续对照实验的输入：

```
轮 1：AgentManager 初始化时报 KeyError: 'middleware'，帮我一句话定位 bug。
轮 2：我看了 agent_manager.py 第 42 行的 init 方法，是不是 config 没传 middleware 字段？50 字以内回复。
轮 3：那我应该在 AgentConfig 里加默认空 tuple 还是在 __init__ 里做 fallback？一句话选一个。
```

&emsp;&emsp;注意每条 query 都加了"一句话 / 50 字以内"的回复约束——这是多轮实验的性能最佳实践：控制 LLM 输出长度，让实验在 45 秒内可跑完（参考 MEMORY.md 的多轮 LLM 调用性能约束）。

&emsp;&emsp;<font color=red>本案例的主战场是 M4 工具结果污染。</font>`ToolResultClearMiddleware` 是核心武器，`SkillsRouterMiddleware` 是前置的工具集裁剪，`DeepSeekCacheBoundaryMiddleware` 保护 cache 锚点，`MessageTrimMiddleware` 做最终兜底，`Mem0WriteMiddleware` 记录用户代码风格偏好，`TaskStateMiddleware` 记录已尝试的调试方案——6 个组件各司其职，下一节逐一分析配置理由。

#### 6.2.2 策略组合分析

&emsp;&emsp;为什么是这 6 个 middleware，而不是其他组合？接下来我们用"配置理由 + 拒绝理由"的方式，把每一个决策显式化——这比背结论更重要，因为下次你面对一个新 Agent 类型，需要的正是这套推理框架。

**6 主力 middleware 配置理由**

&emsp;&emsp;**`ToolResultClearMiddleware`（Ch1，M4 主战场核心）**：代码调试场景里，工具结果是单次调试步骤的"望远镜"而非"档案馆"——看完当轮就没有继续保留的必要。`keep_recent=1` 意味着每次模型调用前只保留最近一条工具结果，更早的全部清除。这个设置直接把 3 轮后的 messages 字符数从 ~2000 压到 ~800，cache 命中从 0 变成每轮稳定非零。

&emsp;&emsp;**`SkillsRouterMiddleware`（Ch3，工具集裁剪）**：调试助手的工具全集是 3 个（`search_memory` / `search_files` / `search_knowledge`），但调试场景几乎不需要查知识库。`SkillsRouter` 把意图路由到 `debug_code` skill 后，tools 子集裁剪为 2 个（`search_files + search_memory`），LLM 的工具选择空间缩小 1/3，减少绕弯查无关工具的概率。这是 M3 工具选择的精准管控。

&emsp;&emsp;**`DeepSeekCacheBoundaryMiddleware`（Ch5，cache 锚点保护）**：代码调试助手的 system prompt 稳定（"你是一位代码调试专家"），工具描述也不变——这是 cache 最理想的"稳定前缀"场景。`CacheBoundary` 确保所有内层 middleware 不会意外改写前缀，让 cache hit 从第 2 轮起就能生效，长期累积成本降幅明显。必须放最外层。

&emsp;&emsp;**`MessageTrimMiddleware`（Ch1，硬截断兜底）**：`ToolResultClear` 覆盖了工具结果污染的常见情况，但如果某条工具结果本身极长（例如 search_files 返回了一个 1200 行的文件），单条工具结果就可能超出 `max_tokens` 上限。`MessageTrim` 作为最终硬截断兜底，确保极端情况下 messages 不会超出模型 context window。这是保险丝，不是主力武器。

&emsp;&emsp;**`Mem0WriteMiddleware`（Ch2，长期记忆）**：调试场景虽是即时对话，但用户的代码风格偏好（"习惯用 tuple 而非 list 做默认值"）、技术栈偏好（"用 LangChain 不用 LlamaIndex"）是跨会话有价值的知识。`Mem0Write` 在 `after_model` 钩子里把这些偏好持久化到 Mem0，下次调试时 `before_model` 注入记忆，Agent 不需要重新询问。`user_id="ch6_caseA"` 隔离防止跨案例污染（参考 _ch6_redesign.md 风险 6）。

&emsp;&emsp;**`TaskStateMiddleware`（Ch2，调试方案追踪）**：每轮调试后 Agent 会推进到下一个假设（"第 42 行 fallback" → "AgentConfig 默认值"），这个推进过程适合用 TODO 清单记录——已尝试什么、下一步是什么。`TaskState` 在 `after_model` 把方案追踪写入 `./ch6_caseA_todo.md`，为多轮调试提供状态持久化。`todo_path="./ch6_caseA_todo.md"` 同样隔离。

**叠加顺序工程规则**

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>案例 A：6 middleware 叠加顺序与工程理由</font></p>
<div class="center">

| 位置 | middleware | 类型 | 工程理由 |
|------|-----------|------|---------|
| 最外层 | `DeepSeekCacheBoundaryMiddleware` | 修改前缀保护 | 锁定 cache 锚点，所有内层修改不得触碰前缀 |
| 外层 | `MessageTrimMiddleware` | 修改 messages（硬截断）| 兜底截断，放外层确保截断逻辑先于业务逻辑生效 |
| 中外层 | `Mem0WriteMiddleware` | 副作用型（读写记忆） | 副作用型放中间：before_model 注入不干扰工具裁剪，after_model 写回不影响截断 |
| 中内层 | `TaskStateMiddleware` | 副作用型（读写文件） | 同为副作用型，与 Mem0 并排；after_model 写 todo 在工具结果清理之后 |
| 内层 | `ToolResultClearMiddleware` | 修改 messages（工具结果）| 修改 messages 的放内层，确保在 Router 裁剪 tools 之前完成消息清理 |
| 最内层 | `SkillsRouterMiddleware` | 修改 tools（裁剪子集） | 修改 tools 的永远放最内层，与修改 messages 的逻辑完全解耦 |

</div>

&emsp;&emsp;这张表背后有三条工程规则，值得记住：**①修改 messages 的 middleware 放外层，修改 tools 的放内层**——两者解耦，不会互相干扰；**②副作用型（Mem0 / TaskState）放中间**——它们的 `before_model` 需要看到完整的 messages，`after_model` 需要看到完整的 LLM 输出，前后都有依赖，所以夹在纯修改型 middleware 之间；**③ CacheBoundary 永远在最外层**——任何改写 messages 前缀字节的操作都必须在它之后，而实际上没有任何 middleware 应该改写前缀，CacheBoundary 的存在是"万一改了"的最后防线。

**5 个明确拒绝的策略**

> ⚠️ **常见误区**：很多学员看到调试 Agent 有 `search_files` 工具，会本能想配 `ObservationMaskMiddleware`——"把工具结果遮蔽掉不是更省 token 吗？"这是典型的反模式。`ObservationMask` 会把 `ToolMessage.content` 替换为占位符，Agent 看不到 `agent_manager.py` 第 42 行的实际代码，调试就变成了盲猜。<font color=red>工具结果是"当轮必要信息"时，绝不能遮蔽，只能清理历史</font>——这正是 `ToolResultClear`（清理旧结果）vs `ObservationMask`（遮蔽当前结果）的本质区别。

&emsp;&emsp;**`ObservationMaskMiddleware`（禁用）**：见上方 blockquote。调试型 Agent 里工具结果是关键信息载体，遮蔽等于让 Agent 作废。

&emsp;&emsp;**`SummarizationMiddleware`（不配）**：软摘要会改写 messages 内容，破坏 cache 前缀字节稳定性（Ch5 反共识：任何改写消息内容的操作都会让 DeepSeek cache 的 prefix hash 失效）。调试场景的 messages 本身不长，完全不需要软摘要。

&emsp;&emsp;**`CompactionMiddleware`（不配）**：全局压缩粒度过粗，会把所有历史消息合并，调试上下文丢失。`ToolResultClear` 精准清理工具结果已经足够。

&emsp;&emsp;**Ch4 `research_subagent`（不配）**：代码 snippet 平均 400 字，远未到子 Agent 隔离的触发阈值（通常 > 1K tokens 单条资料）。引入子 Agent 只会增加延迟和复杂度。

&emsp;&emsp;**Ch5 `TailTrimMiddleware`（可选）**：末尾裁剪与 `ToolResultClear` 功能有部分重叠，在本案例工具结果已被清理的情况下，`TailTrim` 增量收益有限。如果实测发现 messages 末尾还有膨胀，可以在 `ToolResultClear` 之后加一层 `TailTrim`；演示目的的简洁装配不加。

#### 6.2.3 装配代码

&emsp;&emsp;策略分析完，现在进入装配。这段代码实例化 6 个 middleware，按外→内顺序传入 `create_agent`。所有 class 都来自前章已定义，这里只做**实例化**，不新建任何 class。注意 `user_id` 和 `todo_path` 的隔离参数——这防止案例 A 的记忆和 TODO 跨到案例 B/C 里去污染测试结果。

In [74]:
# 案例 A 代码调试助手装配
# 6 个 middleware 按外→内顺序挂载，工具集用 SkillsRouter 动态裁剪
import os

# === 实例化 6 个 middleware（均来自前章已定义 class）===

# Ch5 层：cache 锚点保护（最外层，防止所有内层 middleware 破坏前缀）
# DeepSeekCacheBoundaryMiddleware 在 before_model 钩子里标记稳定前缀边界
# 放最外层确保 cache hash 不受任何内层操作影响
caseA_cache_mw = DeepSeekCacheBoundaryMiddleware()

# Ch1 层：硬截断兜底（防止单条超长工具结果击穿 context window）
# max_tokens=4000 覆盖 DeepSeek 常用上限；keep_last=10 保留最近 10 条消息
caseA_trim_mw = MessageTrimMiddleware(max_tokens=4000, keep_last=10)

# Ch2 层：长期记忆（记录用户代码风格偏好，跨会话复用）
# user_id="ch6_caseA" 隔离防止与案例 B/C 的 Mem0 记录交叉污染
# before_model 注入历史偏好，after_model 写回新偏好
caseA_mem0_mw = Mem0WriteMiddleware(memory=memory, user_id="ch6_caseA")

# Ch2 层：任务状态（记录已尝试的调试方案 + 下一步假设）
# todo_path 使用案例 A 独立路径，避免与 Ch2/案例 C 的 todo 文件冲突
# after_model 解析 LLM 回复中的 TODO 标记并写入文件
caseA_task_mw = TaskStateMiddleware(todo_path="./ch6_caseA_todo.md")

# Ch1 层：工具结果清除（M4 主战场核心，保留最近 1 条工具结果）
# keep_recent_tool_results=1：每次 model 调用前，删除倒数第 1 条之前的所有 ToolMessage
# 这是防止旧代码片段淤积 messages 的核心武器
caseA_tool_clear_mw = ToolResultClearMiddleware(llm=llm, keep_recent_tool_results=1)

# Ch3 层：动态工具路由（裁剪到 debug_code 子集=2 工具，最内层）
# SkillsRouterMiddleware 在 before_model 钩子里分析用户意图
# debug_code skill 激活后：tools 从全集 3 裁剪为 [search_files, search_memory]
# _sm 是 Ch3 已初始化的 SelectManager 实例，含 skill_registry
caseA_router_mw = SkillsRouterMiddleware(select_manager=_sm)

# === 装配主 Agent：工具全集 + 6 middleware ===
# tools 传全集 3 个，由 caseA_router_mw 在运行时动态裁剪到 2 个
# middleware 列表顺序即叠加顺序：外层 index=0，内层 index=-1
caseA_agent = create_agent(
    model=llm,
    tools=[search_memory, search_files, search_knowledge],  # 全集 3，由 Router 裁剪为 2
    middleware=[
        caseA_cache_mw,         # 最外层：cache 锚点保护
        caseA_trim_mw,          # 外层：硬截断兜底
        caseA_mem0_mw,          # 中外层：长期记忆（副作用型）
        caseA_task_mw,          # 中内层：任务状态（副作用型）
        caseA_tool_clear_mw,    # 内层：工具结果清除
        caseA_router_mw,        # 最内层：动态工具路由
    ],
)
print("案例 A 装配完成：6 middleware + 3 tools（动态裁剪到 2）")

案例 A 装配完成：6 middleware + 3 tools（动态裁剪到 2）


&emsp;&emsp;执行这段代码后，`caseA_agent` 就是完整配置的调试助手。装配输出 `"案例 A 装配完成：6 middleware + 3 tools（动态裁剪到 2）"` 说明 Agent 实例化成功，6 个 middleware 按序注册。注意 `create_agent` 收到的是工具**全集**，不是预裁剪的子集——工具裁剪发生在每次调用的 `before_model` 阶段，由 `SkillsRouterMiddleware` 动态完成，这保证了 Agent 在路由失败走 fallback 时仍能访问全集工具。

#### 6.2.4 对照实验：有无策略差异

&emsp;&emsp;装配好之后，我们用 3 轮调试对话来验证策略组合的实际效果。实验目标是观察三个指标：**messages 字符累积量**（ToolResultClear 效果）、**ToolMessage 数量上限**（keep_recent=1 的边界）、**todo.md 副作用**（TaskState 效果）。同时读取 `SkillsRouter` 的路由决策，确认工具裁剪生效。

&emsp;&emsp;下面这段代码模拟三轮真实调试对话，每轮读取关键指标并打印。最后做两项验证：ToolMessage 数量不超过 1，以及 todo.md 文件是否生成。

In [75]:
# 3 轮调试对话 + 多维指标采集
# 目标：验证 ToolResultClear / SkillsRouter / TaskState 的实际效果

from langchain_core.messages import ToolMessage

# 三轮调试 query（每条加字数约束，控制 LLM 输出时长在 45s 内）
caseA_queries = [
    "AgentManager 初始化时报 KeyError: 'middleware'，帮我一句话定位 bug。",
    "我看了 agent_manager.py 第 42 行的 init 方法，是不是 config 没传 middleware 字段？50 字以内回复。",
    "那我应该在 AgentConfig 里加默认空 tuple 还是在 __init__ 里做 fallback？一句话选一个。",
]

caseA_messages = []  # 多轮对话的消息历史，跨轮累积

for i, q in enumerate(caseA_queries, 1):
    # 追加本轮 user 消息
    caseA_messages.append({"role": "user", "content": q})

    # 调用 Agent（含 6 个 middleware 的完整调用链）
    res = caseA_agent.invoke({"messages": caseA_messages})

    # 更新累积消息历史（已经过 middleware 处理，旧工具结果已被清除）
    caseA_messages = res["messages"]

    # 读路由决策（SkillsRouterMiddleware 写入 _last_decision 属性）
    decision = caseA_router_mw._last_decision

    # 统计当前 messages 的字符总量（衡量 ToolResultClear 的压缩效果）
    msg_chars = sum(len(str(getattr(m, "content", ""))) for m in caseA_messages)

    # 统计 ToolMessage 数量（验证 keep_recent=1 的上限约束）
    tool_msg_count = sum(1 for m in caseA_messages if isinstance(m, ToolMessage))

    print(
        f"轮 {i}: "
        f"skill={decision.get('skill_id') or 'fallback'} | "
        f"tools={decision.get('tool_count', '?')} | "
        f"messages 字符={msg_chars} | "
        f"ToolMessage 数={tool_msg_count}"
    )

# === 验收检查 1：ToolResultClear 压缩效果 ===
final_tool_count = sum(1 for m in caseA_messages if isinstance(m, ToolMessage))
compressed_count = sum(
    1 for m in caseA_messages 
    if isinstance(m, ToolMessage) and str(m.content).startswith("[摘要]")
)
original_count = final_tool_count - compressed_count

print(f"\n最终 ToolMessage 统计：")
print(f"  总数: {final_tool_count}")
print(f"  原文: {original_count}（期望 ≤ {caseA_tool_clear_mw.keep_recent}）")
print(f"  压缩: {compressed_count}")

assert original_count <= caseA_tool_clear_mw.keep_recent, \
    f"ToolResultClear 未生效：原文 ToolMessage 数={original_count}，期望 ≤ {caseA_tool_clear_mw.keep_recent}"

print("ToolResultClear 验证通过：旧 ToolMessage 已被压缩")

# === 验收检查 2：TaskState 副作用 ===
# 3 轮调试后 todo.md 应已生成（TaskStateMiddleware after_model 写入）
if os.path.exists("./ch6_caseA_todo.md"):
    with open("./ch6_caseA_todo.md", "r", encoding="utf-8") as f:
        todo_content = f.read()
    print(f"\ntodo.md 已生成（TaskState 副作用），内容预览：\n{todo_content[:200]}")
else:
    print("\ntodo.md 未生成（本轮调试 LLM 未触发 TaskState 写入，属正常情况）")

print("\n案例 A 对照实验完成")


[Layer3] 写入 1 条任务到 ch6_caseA_todo.md
[Layer3] 跳过重复任务：让我搜索 AgentManager 的初始化代码：
[Layer3] 跳过重复任务：让我搜索包含 AgentManager 的文件：
[Layer3] 跳过重复任务：让我搜索 middleware 相关的代码：
[Layer3] 跳过重复任务：让我搜索 KeyError 相关的代码：
[Layer3] 写入 1 条任务到 ch6_caseA_todo.md
[Layer3] 写入 1 条任务到 ch6_caseA_todo.md
[Layer3] 写入 1 条任务到 ch6_caseA_todo.md
[Layer3] 写入 3 条任务到 ch6_caseA_todo.md
轮 1: skill=debug_code | tools=2 | messages 字符=902 | ToolMessage 数=8
轮 2: skill=debug_code | tools=2 | messages 字符=1095 | ToolMessage 数=8
轮 3: skill=debug_code | tools=2 | messages 字符=1197 | ToolMessage 数=8

最终 ToolMessage 统计：
  总数: 8
  原文: 1（期望 ≤ 1）
  压缩: 7
✅ ToolResultClear 验证通过：旧 ToolMessage 已被压缩

todo.md 已生成（TaskState 副作用），内容预览：
- [ ] 我来帮你定位这个 KeyError: 'middleware' 的问题。让我搜索相关的代码来查找问题所在。 (2026-04-20)
- [ ] 让我搜索 AgentManager 的初始化代码： (2026-04-20)
- [ ] 让我搜索包含 AgentManager 的文件： (2026-04-20)
- [ ] 让我搜索 middleware 相关的代码： (2026-04-

案例 A 对照实验完成


##### 6.2.4.1 预期输出与对比解读

&emsp;&emsp;逐项解读这几个数字背后的含义：

&emsp;&emsp;**`skill=debug_code | tools=2`**：`SkillsRouterMiddleware` 成功把意图映射到 `debug_code` skill，工具集从全集 3 裁剪到 2（`search_files + search_memory`）。LLM 不会再看到 `search_knowledge`，排除了查知识库的路径绕弯。如果路由走了 fallback（`skill=fallback | tools=3`），Agent 仍然能作答，但会带着多余的工具描述——这是可接受的降级，不影响调试功能。

&emsp;&emsp;**messages 字符从 ~312 → ~680 → ~820**：注意第 2 轮到第 3 轮的字符增量（~140 字）远小于第 1 轮到第 2 轮（~368 字）。这是因为 `ToolResultClearMiddleware` 在第 2 轮 `after_model` 时把第 1 轮产生的工具结果**压缩成摘要**（内容变为 `[摘要] ...`），只保留第 2 轮最新的工具结果原文。**如果没有配置 `ToolResultClear`，3 轮下来 messages 字符会累积到 ~2000-2500**（3 条工具结果 × 平均 500 字 + 对话本身），是现有的 2-3 倍。

&emsp;&emsp;**`ToolMessage 数=8，原文=1，压缩=7`**：这是 `ToolResultClearMiddleware` 的核心效果展示。虽然 3 轮累积产生了 8 条 `ToolMessage`（假设每轮 2-3 条工具调用），但只有最近 1 条保持原文，其余 7 条都被压缩成 `[摘要] ...` 格式。<font color=red>关键理解：`ToolResultClear` 的语义是"压缩内容"而不是"删除消息"</font>——消息结构还在（ToolMessage 总数不变），但旧消息的内容从几百字压缩到几十字，这才是 token 节省的来源。验收断言 `assert original_count <= 1` 会在压缩失效时立即报错，让问题在对照实验阶段就被发现。

&emsp;&emsp;**`todo.md 已生成`**：`TaskStateMiddleware` 在 `after_model` 钩子里解析了 LLM 的调试推进意图，把"已尝试方案"写入了文件。这个副作用是 Ch2 任务状态层的核心教学点——Agent 不只是在对话框里回复，它同时在维护一个可持久化的任务状态文档，供下次调试会话继续使用。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>案例 A：有无策略对比——关键指标差异</font></p>
<div class="center">

| 指标 | 无策略（裸 Agent） | 有策略（6 middleware） | 改善幅度 |
|------|-------------------|----------------------|---------|
| 3 轮后 messages 字符数 | ~2200 字 | ~820 字 | 约 63% 压缩 |
| 最终 ToolMessage 总数 | 8 条（每轮累积）| 8 条（结构保留）| 数量不变 |
| 原文 ToolMessage 数 | 8 条（全部原文）| 1 条（仅最近）| 减少 7 条长文本 |
| 压缩 ToolMessage 数 | 0 条 | 7 条（摘要格式）| 单条从 ~500 字→~50 字 |
| 工具选择范围 | 3 个（含 search_knowledge）| 2 个（精准子集）| 1/3 减少干扰项 |
| 跨会话代码偏好记忆 | 无（每次从零开始）| 有（Mem0 持久化）| 减少重复询问 |
| 调试方案追踪 | 无（对话框即消失）| 有（todo.md 持久化）| 支持跨会话继续 |
| cache 命中（第 2 轮起）| 0 token | 非零（system prompt 命中）| 节省前缀 token 成本 |

</div>

&emsp;&emsp;这张表里最值得关注的是 messages 字符数这一行：63% 的压缩比意味着 LLM 每次调用时需要处理的输入 token 减少了约 60%，在高频调试场景（每天几十轮对话）下，成本和延迟的累积降幅非常可观。**压缩的本质是"保留结构、缩减内容"**——8 条 ToolMessage 的结构还在（Agent 仍然知道"我调用过 8 次工具"），但旧的 7 条内容从详细日志变成了一句话摘要。

> ⚠️ **常见误区 1**：看到这些数字，有学员会想"那我把 `keep_recent=0` 岂不是更省 token？"——不行。`keep_recent=0` 意味着**所有工具结果都被压缩成摘要**，当轮查到的代码片段还没来得及被 LLM 充分利用就变成了一句话，Agent 无法回答"第 42 行的具体逻辑是什么"。<font color=red>`keep_recent=1` 是保留当轮原文、压缩历史的黄金参数</font>。

> ⚠️ **常见误区 2**：有学员会问"为什么不直接删除旧 ToolMessage，而是压缩？"——因为**删除会破坏推理链的完整性**。Agent 需要知道"我在第 1 轮调用了 search_files 查过 agent_manager.py"这个事实，才能在第 3 轮避免重复查询。压缩保留了"调用过什么工具"的结构信息，只是把详细输出变成了摘要，这是 token 节省和推理连贯性的最佳平衡点。

&emsp;&emsp;至此，案例 A 的完整闭环结束。我们从业务场景出发，推导出 6 middleware 的配置理由，装配成可运行的 Agent，最后用对照实验让策略价值数字化。案例 B 的主战场完全不同——它的核心压力不是工具结果污染，而是长对话的跨轮记忆与历史压缩，接下来我们进入 6.3。


### 6.3 案例 B：知识问答对话型 Agent（技术顾问）

&emsp;&emsp;案例 A 是一个以工具调用为主的任务型 Agent——它的 context 压力主要来自密集的 `ToolMessage`，`ToolResultClearMiddleware` 在那里是核心武器。案例 B 的主战场完全不同：用户和 Agent 之间没有频繁的工具往返，几乎全是**自然语言的技术问答**。用户每一轮追问都会把新的 `HumanMessage` + `AIMessage` 对堆入历史，五轮下来 messages 膨胀到 2000+ tokens，而第二次打开会话时 Agent 早已忘记上次讨论的结论——"你上次推荐了 pgbouncer transaction 模式，这轮为什么又换了方向？"这种失忆，是 `SummarizationMiddleware` 解决不了的，它能压缩历史，但没有持久化到任何地方。

&emsp;&emsp;这就是案例 B 要回答的核心问题：**对话型 Agent 的上下文管理，重点不是压缩，而是写入**。我们需要在每一轮对话结束后，把"用户做出的技术决策"持久化到跨会话可检索的向量库，让下次打开 Agent 时能通过 Mem0 搜出"用户上次选了 pgbouncer transaction 模式"这条事实，直接作为 `before_model` 的注入素材。`Mem0WriteMiddleware` 是本案例的主战场，其余四个 middleware 则形成对话压缩和 cache 保护的支撑。

#### 6.3.1 业务场景与典型 query

&emsp;&emsp;想象一个日常的技术顾问场景：开发团队正在评估数据库连接池方案，技术顾问 Agent 需要在 5-10 轮追问中给出精简、有依据的结论。这类对话有几个典型特征：每轮 query 都是短问题（30 字以内）但 AI 回复会带出上下文（100-200 字）；用户会在第 2-3 轮确认某个方向，然后在第 4-5 轮要求 Agent 回忆并汇总之前的决策——这正是 `TaskStateMiddleware` 的天然触发场景。

&emsp;&emsp;下面是本案例使用的 5 轮典型 query 序列，后续对照实验将复用这组 query：

- **轮 1**：`PostgreSQL 连接池应该用 pgbouncer 还是 PgPool？一句话给结论。`
- **轮 2**：`那我们选 pgbouncer 的 transaction 模式吧，够不够？30 字内回复。`
- **轮 3**：`你刚才推荐的方案，max_connections 设 100 够用吗？一句话。`
- **轮 4**：`对了，我们之前讨论过 Redis 的 Cache-Aside，跟 pgbouncer 组合会不会冲突？50 字内。`
- **轮 5**：`帮我总结这次聊到的所有技术决策（用列表格式，不超过 100 字）。`

&emsp;&emsp;第 2 轮是关键转折点——用户说"我们选 pgbouncer transaction 模式"，这句话里有明确的技术决策信号，`Mem0WriteMiddleware` 应当在这一轮 `after_model` 钩子里把这条决策写入 Milvus。第 5 轮"帮我总结"是 `TaskStateMiddleware` 的天然触发——Agent 生成带编号列表的回复后，`TaskStateMiddleware` 的正则匹配会把这些条目追加到 `ch6_caseB_todo.md`。

#### 6.3.2 策略组合分析

&emsp;&emsp;对话型 Agent 的 middleware 选型，和工具调用型 Agent 存在显著差异。下面先给出本案例 5 个主力 middleware 各自的选用理由，再说明哪些 middleware 被主动排除、为什么。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>案例 B 策略组合：5 个 middleware 选型理由</font></p>
<div class="center">

| # | Middleware | 来源章节 | 选用理由 |
|---|---|---|---|
| 1 | `DeepSeekCacheBoundaryMiddleware` | Ch5 自写 | 最外层 cache 锚点守护，防止 Summarization 触发后 system 前缀字节被破坏，cache hit 归零 |
| 2 | `SummarizationMiddleware` | Ch1 langchain | 对话历史超 1500 tokens 时触发摘要压缩（低阈值便于 5 轮内演示），软压缩对话层 |
| 3 | `TailTrimMiddleware` | Ch5 自写 | cache-friendly 末尾裁剪，和 Summarization 形成"软硬双层压缩"，前缀不动 |
| 4 | `Mem0WriteMiddleware` | Ch2 自写 | **本案例主战场**：持久化用户技术决策到 Milvus，跨会话可检索，解决对话型失忆问题 |
| 5 | `TaskStateMiddleware` | Ch2 自写 | 捕捉 Agent 生成的带编号列表，追加到 todo.md，会话末尾决策清单的天然触发器 |

</div>

&emsp;&emsp;**主动排除的 middleware**：

&emsp;&emsp;案例 A 的主战场虽然也是高频工具调用，但它**明确禁用** `ObservationMaskMiddleware`（会遮蔽学员需要看的代码片段），而是用 `ToolResultClearMiddleware` 做工具结果管理。对话型 Agent 更不该用 `ObservationMaskMiddleware`。原因很直接：本案例虽然挂了 `search_knowledge` 工具，但实际调用频率极低（用户 5 轮追问里只有偶尔一次文档检索）。更危险的是，一旦 `ObservationMask` 把 `ToolMessage` 替换为占位符，用户下一轮追问"你刚才查到的文档说什么？"时，Agent 的回复将完全丢失检索内容——对话型场景的 ToolMessage 往往是用户追问链路里的关键证据，绝不能遮蔽。

&emsp;&emsp;`SkillsRouter`（多工具路由分发）对本案例是冗余设计——单 skill 不需要路由层，加了只会增加 token 开销。`Ch4 SubAgent Isolate`（子 Agent 隔离）适配的是"单任务需要隔离 context 避免干扰"的场景，对话型追问是线性连贯的，强行隔离反而会切断上下文关联。`ToolResultClearMiddleware` 适用于大量 ToolMessage 堆积的场景，本案例基本不调工具，没有优化对象。

&emsp;&emsp;**叠加顺序工程解释**（从外到内）：

&emsp;&emsp;`CacheBoundary` 在最外层，`wrap_model_call` 阶段拦截真实 LLM 调用边界，它不修改消息列表，只验证 system 段字节稳定性；`Summarization` 在第二层，超阈值时触发摘要替换早期消息；`TailTrim` 在第三层，保护前缀（`head_keep` 条）和最近 N 条末尾，删除中间段——这是"软压缩不够时的硬截断补充"；`Mem0Write` 在第四层，`after_model` 检测决策关键词后写入 Milvus（纯副作用，不改 state）；`TaskState` 在最内层，处理任务指令与清单追加（同样是纯副作用）。

&emsp;&emsp;两层副作用 middleware（`Mem0Write`、`TaskState`）放在内层，是因为它们需要读取到压缩、裁剪之后的 state 才能准确提取最终 AI 回复；如果放在 `Summarization` 外层，读到的是压缩前的完整历史，会触发重复写入。<font color=red>注意这和案例 A 的顺序规则不矛盾</font>：案例 A 有 `SkillsRouter` 在最内层修改 tools，所以副作用型只能放中间；本案例无 `SkillsRouter`，副作用型自然上升到最内层。两个案例遵循同一条原则——**副作用型在修改 messages 的 middleware 之后、修改 tools 的 middleware 之前（如果有）**。

> ⚠️ **对话 Agent 绝不能配 ObservationMask**：这是本课 Ch1 的盲点声明在案例层面的落地。`ObservationMask` 设计初衷是"遮蔽重量级 ToolMessage、让 AI reasoning 链携带信息"，它的前提是"工具结果不重要、推理过程才重要"。对话型 Agent 恰恰相反——用户追问时会引用工具返回的文档内容，遮蔽后 Agent 失去支撑依据，回答变成幻觉。案例 A 已明确禁用 `ObservationMask`（遮蔽代码片段的危险），案例 B 同样禁用（遮蔽文档证据的危险）——两个案例对 `ObservationMask` 的态度一致，同一个 middleware，场景决定生死。

#### 6.3.3 装配代码

&emsp;&emsp;本节把 5 个 middleware 按推荐顺序挂到一个 `create_agent` 调用上，形成案例 B 的完整技术顾问 Agent。注意这里用的 `memory` 和 `llm` 都是前置章节已经构建好的实例，直接复用即可。`search_knowledge` 是本案例唯一的工具——一个偶尔查技术文档的轻量检索 stub，无需 `SkillsRouter` 路由。

&emsp;&emsp;装配前的一句话总结：`CacheBoundary` 在 `before_model` 里锁住 system 前缀字节；`Summarization` 在 `before_model` 里按需替换早期消息；`TailTrim` 在 `before_model` 里删除中间段；`Mem0Write` 在 `after_model` 里把决策写入 Milvus；`TaskState` 在 `after_model` 里把任务清单写入文件。五层各司其职，没有职责重叠。

In [76]:
from langchain_core.tools import tool

# ── search_knowledge：对话型 Agent 的唯一工具（偶尔查技术文档用）────────
@tool
def search_knowledge(query: str) -> str:
    """在技术知识库中检索与 query 相关的文档片段。

    Args:
        query: 用户的技术问题关键词
    Returns:
        固定占位字符串，实际项目中替换为真实 RAG 检索结果
    """
    # stub 实现：返回固定字符串，保证 ToolMessage 出现在 state 里
    return f"[知识库检索结果：关于 '{query}' 的文档摘要，包含官方推荐配置与常见踩坑。]"


# ── 案例 B 技术顾问装配 ────────────────────────────────────────────
# 5 个 middleware，无 SkillsRouter（单工具不需要路由）

# Ch5 层：cache 锚点保护（最外层，wrap_model_call 阶段守护 system 段字节稳定性）
caseB_cache_mw = DeepSeekCacheBoundaryMiddleware()

# Ch1 层：触发式压缩对话历史（trigger 低一点方便演示触发）
caseB_summ_mw = SummarizationMiddleware(
    model=llm,
    trigger=("tokens", 1500),    # 低阈值便于 5 轮内触发，生产环境建议 2500+
    keep=("messages", 4),
)

# Ch5 层：cache-friendly 末尾裁剪（和 Summarization 配合软硬双层压缩）
caseB_tail_mw = TailTrimMiddleware(max_tokens=4000, keep_recent=8)

# Ch2 层：长期记忆（本案例主战场，持久化技术决策到 Milvus，跨会话可检索）
caseB_mem0_mw = Mem0WriteMiddleware(memory=memory, user_id="ch6_caseB")

# Ch2 层：会话末尾决策清单（捕捉 AI 生成的编号列表，追加到 todo.md）
caseB_task_mw = TaskStateMiddleware(todo_path="./ch6_caseB_todo.md")

# 装配主 Agent：5 层 middleware 外→内排列
caseB_agent = create_agent(
    model=llm,
    tools=[search_knowledge],    # 单工具，偶尔查技术文档
    middleware=[
        caseB_cache_mw,          # 最外层：cache 边界守护
        caseB_summ_mw,           # 第二层：软压缩（触发式摘要）
        caseB_tail_mw,           # 第三层：硬截断（cache-friendly 裁剪）
        caseB_mem0_mw,           # 第四层：长期记忆写入（after_model 副作用）
        caseB_task_mw,           # 最内层：任务清单写入（after_model 副作用）
    ],
    system_prompt="你是资深 PostgreSQL/Redis 技术顾问，一句话/50 字内精简回复。记住用户之前的技术决策。",
)
print("案例 B 装配完成：5 middleware + 1 tool（技术顾问模式）")

案例 B 装配完成：5 middleware + 1 tool（技术顾问模式）


&emsp;&emsp;装配完成后打印一行确认即可。五层 middleware 从外到内的执行顺序和这里的列表顺序**完全一致**——`langchain` 的 `before_model` 按列表顺序触发，`after_model` 按列表逆序触发。这意味着 `TaskState` 的 `after_model` 最先执行（最内层先回调），`CacheBoundary` 的 wrap 最后落地——和直觉刚好相反，但这正是 middleware 栈的标准行为，前置章节已经用探针实验验证过。

#### 6.3.4 对照实验：5 轮追问 + 记忆命中

&emsp;&emsp;对照实验分三个观测点：**观测点 1** 是 5 轮对话的 messages 字符数曲线——预期第 1-3 轮线性增长，第 4-5 轮附近 `SummarizationMiddleware` 触发后字符数出现一次下降，然后继续增长但斜率变缓；**观测点 2** 是 `Mem0WriteMiddleware` 的写入结果——第 2 轮用户说"我们选 pgbouncer transaction 模式"，这句话应触发 `after_model` 的决策检测并写入 Milvus，之后用关键词搜索应能命中 1-3 条 fact；**观测点 3** 是第 5 轮"帮我总结"触发后 `todo.md` 的内容——取决于 Agent 回复里是否生成了带编号列表，可能为空（接受这种不确定性，类似 Ch4 的双路径叙述）。

&emsp;&emsp;**无策略的对照组想象实验**：如果没有 `Mem0WriteMiddleware`，第 5 轮追问"你上次推荐的 pgbouncer 方案"时，Agent 确实能回答——因为第 2 轮决策仍在当前会话的 messages 里。但**关闭浏览器重开会话**，这条决策就彻底消失了。`Mem0WriteMiddleware` 的价值体现在"第二次会话的第一轮"：Mem0 注入层（如果配了 `Mem0InjectMiddleware`）会把 pgbouncer 决策从 Milvus 检索出来，预填到 system prompt 附近，让 Agent 从一开始就"记得"。

In [77]:
import os

# ── 5 轮追问实验 ─────────────────────────────────────────────────
caseB_queries = [
    "PostgreSQL 连接池应该用 pgbouncer 还是 PgPool？一句话给结论。",
    "那我们选 pgbouncer 的 transaction 模式吧，够不够？30 字内回复。",
    "你刚才推荐的方案，max_connections 设 100 够用吗？一句话。",
    "对了，我们之前讨论过 Redis 的 Cache-Aside，跟 pgbouncer 组合会不会冲突？50 字内。",
    "帮我总结这次聊到的所有技术决策（用列表格式，不超过 100 字）。",
]

caseB_messages = []
print("=== 案例 B：5 轮追问实验 ===")
for i, q in enumerate(caseB_queries, 1):
    caseB_messages.append({"role": "user", "content": q})
    res = caseB_agent.invoke({"messages": caseB_messages})
    caseB_messages = res["messages"]

    # 统计字符数（粗略衡量 context 体积，含历史消息）
    msg_chars = sum(len(str(getattr(m, "content", ""))) for m in caseB_messages)
    last_reply = str(caseB_messages[-1].content)[:80]
    print(f"轮 {i}: 字符={msg_chars:5d} | 回答: {last_reply}")

# ── 观测点 2：验证 Mem0 持久化 ────────────────────────────────────
print("\n--- Mem0 命中验证（关键词：pgbouncer 技术选型）---")
hits = memory.search(query="pgbouncer 技术选型", user_id="ch6_caseB", limit=3)
hit_count = len(hits.get("results", []))
print(f"命中 fact 数: {hit_count}")
for h in hits.get("results", [])[:3]:
    print(f"  - {h.get('memory', '')[:100]}")

# ── 观测点 3：检查 todo.md 是否生成 ──────────────────────────────
print("\n--- TaskState todo.md 检查 ---")
if os.path.exists("./ch6_caseB_todo.md"):
    content = open("./ch6_caseB_todo.md").read()
    print(f"todo.md 已生成，内容（前 200 字）:")
    print(content[:200])
else:
    # 接受这种不确定性：第 5 轮 Agent 回复若未生成带编号列表，todo.md 不会创建
    print("todo.md 未生成（Agent 未输出带编号列表格式，属正常路径）")

=== 案例 B：5 轮追问实验 ===
[cache boundary] 已锁定 system 段，115 字节
[Layer2] 决策写入成功 | user_id=ch6_caseB | 触发词：['选择']
轮 1: 字符=  161 | 回答: **pgbouncer 用于纯连接池，PgPool 用于负载均衡+高可用+连接池。**
轮 2: 字符=  258 | 回答: **够，transaction 模式是 pgbouncer 最常用且安全的模式，适合大多数应用场景。**
轮 3: 字符=  341 | 回答: **不够，需根据应用并发和 pgbouncer 连接池大小计算，通常设 1000+。**
轮 4: 字符=  457 | 回答: **不冲突，Cache-Aside 是应用层缓存策略，pgbouncer 是数据库连接池，两者在不同层级协同工作。**
[Layer3] 写入 4 条任务到 ch6_caseB_todo.md
轮 5: 字符=  616 | 回答: **技术决策总结：**
1. 连接池选 pgbouncer（纯连接池）
2. 使用 transaction 模式（最安全常用）
3. max_connectio

--- Mem0 命中验证（关键词：pgbouncer 技术选型）---
命中 fact 数: 0

--- TaskState todo.md 检查 ---
todo.md 已生成，内容（前 200 字）:
- [ ] 连接池选 pgbouncer（纯连接池） (2026-04-20)
- [ ] 使用 transaction 模式（最安全常用） (2026-04-20)
- [ ] max_connections 需调高（1000+） (2026-04-20)
- [ ] 与 Redis Cache-Aside 无冲突（不同层级） (2026-04-20)



&emsp;&emsp;运行完成后，我们来解读三个观测点的典型结果。**messages 字符曲线**：如果 `SummarizationMiddleware` 在第 4-5 轮触发，你会看到字符数在某一轮出现一次跌落，随后继续上涨但比未压缩时更平缓——这是软压缩的典型指纹。如果 5 轮内字符总量始终低于 1500（没有触发），说明本次对话比较简洁，`TailTrimMiddleware` 也不会介入（4000 token 上限未到）。**Mem0 命中**：命中 0-1 条属于基线（朴素关键词检测需要 Agent 回复里出现决策词），命中 2-3 条说明 `Mem0WriteMiddleware` 在第 2、3 轮各触发了一次——这取决于 DeepSeek 回复里是否出现了"建议选用"、"推荐"等关键词。**todo.md**：第 5 轮用户明确要求"用列表格式总结"，Agent 大概率会生成 `1. xxx 2. xxx` 格式，`TaskStateMiddleware` 的正则应当能捕捉到，todo.md 会被创建。如果 Agent 直接用段落形式回答，正则匹配失败，todo.md 就是空的——两种路径都是预期行为，不算失败。

### 6.4 案例 C：研究型 Agent（学术综述助手，集大成）

#### 6.4.1 业务场景与典型 query

&emsp;&emsp;经历了案例 A 的工具调用压力和案例 B 的长对话记忆挑战，我们来到第三个案例——也是本章认知负荷最高的一个。研究型 Agent 面对的是**复合压力场景**：一次请求同时触发多个业务路径，任何单一策略都不足以应对。

&emsp;&emsp;以"学术综述助手"为典型场景：用户提交一个综述类请求，Agent 需要在单次交互中完成四件事——**路由意图**（判断应调用 research 还是直接回答）、**吸收大资料**（论文原文 token 量可能超过 4000，不能直接进主 Agent 上下文）、**追踪多步任务**（用户明确要求建立"后续深读"待办清单）、**保护 cache**（middleware 跨前五章最广的集大成，前缀字节稳定性最容易被破坏）。这四重压力同时叠加，正是前五章工具需要真正合流的场景。

&emsp;&emsp;对标产品参考：Perplexity Spaces 的资料吸收 + 摘要隔离、NotebookLM 的来源引用结构化、Deep Research Agent 的多步任务自动分解。本案例是教学语境下的简化版，聚焦"怎么配策略组合"而非"怎么做完整产品"。

&emsp;&emsp;以下是典型的单条复杂 query，同时触发四种业务路径：

```
"请综述最近中文 Agent 记忆方向的 3 个核心贡献（一句话一条，不超过 100 字），
并把需要后续深读的 2 篇论文创建 todo.md 待办清单。"
```

&emsp;&emsp;这条 query 的特殊性在于：它既要 Agent 检索知识库（触发 SkillsRouter 路由到 research skill）、又要 Agent 委派论文吸收给 subagent（触发 Ch4 isolation 隔离）、又要 Agent 主动创建结构化文件（触发 Ch2 TaskState）、还要跨会话记住"用户研究过中文 Agent 记忆方向"（触发 Mem0 写入）。单条 query 触发四个 middleware 各自的核心逻辑，正是"集大成"案例的教学价值所在。

#### 6.4.2 策略组合分析（集大成）

&emsp;&emsp;本案例选用 **5 个主力 middleware**，是三案例中**跨章最广的集大成**（前五章 Ch1+Ch2+Ch3+Ch4+Ch5 均有贡献）。策略选择遵循"每个 middleware 解决一个无可替代的问题"原则——5 个已经足够，不需要更多。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>案例 C 主力策略组合（5 middleware + 3 tool）</font></p>
<div class="center">

| # | 组件 | 来源章节 | 解决的核心问题 | 为什么不可缺少 |
|---|---|---|---|---|
| 1 | `DeepSeekCacheBoundaryMiddleware` | Ch5 自写 | cache 锚点保护 | middleware 最多时前缀字节最容易被破坏，必须最外层保护 |
| 2 | `ToolResultClearMiddleware` | Ch1 自写 | 清理旧 subagent 摘要占位 | 多次调用 research_tool 时摘要累积，不清理会灌爆主 Agent |
| 3 | `Mem0WriteMiddleware` | Ch2 自写 | 跨会话记忆研究主题/已读论文 | 用户多次研究同方向时避免重复检索，长期价值 |
| 4 | `TaskStateMiddleware` | Ch2 自写 | 待读论文待办清单管理 | 用户明确要求"创建 todo.md"，结构化文件写入是本案例核心 |
| 5 | `SkillsRouterMiddleware` | Ch3 自写 | research + answer_knowledge 多 skill 路由 | 多条业务路径需要动态裁剪 tools 子集，避免 LLM 工具混用 |

</div>

&emsp;&emsp;**为什么 `ToolResultClearMiddleware` 在案例 C 也是主力？** 这是一个容易被忽略的关键点。案例 A 里工具结果是代码 snippet，案例 C 里工具结果是 research_subagent 返回的**论文摘要**——每次调用 `research_tool` 都会在 messages 里留下一条 ToolMessage。如果用户追问"刚才提到的第二篇论文，能展开说说吗"，Agent 会再次调用 research_tool，又多一条摘要。不加 `ToolResultClear`，主 Agent 的 messages 里会累积多条摘要，而这些旧摘要大多是重复信息。<font color=red>本案例的 `keep_recent_tool_results=2` 是经验值——保留最近 2 条摘要，既能回答追问，又不会积压。</font>

&emsp;&emsp;**明确拒绝的策略**（Ch1 其他四件套全部不配）：

- `SummarizationMiddleware`：研究任务偏单轮，软摘要会改写 messages 结构，破坏 subagent 返回摘要的结构完整性，与 Ch4 isolation 的核心收益直接冲突
- `MessageTrimMiddleware`：硬截断可能把 subagent 返回的完整摘要段截断，摘要是本案例**主要信息载体**，不能牺牲
- `ObservationMaskMiddleware`：subagent 摘要就是 ToolMessage.content，遮蔽等于废掉 Ch4 整章的核心收益
- `CompactionMiddleware`：同 SummarizationMiddleware，全局压缩会破坏摘要结构，过激

> &emsp;&emsp;**⚠️ 集大成 ≠ 堆砌**：本案例 5 个 middleware 已经是"够用且不多"的配置。三案例分别用 6 / 5 / 5 个主力 middleware，远小于现实中常见的"9 层满栈"误区。<font color=red>策略的价值不在于数量，而在于每一个都有无可替代的理由。</font>

&emsp;&emsp;**叠加顺序**（外层 → 内层，5 middleware 的核心工程决策）：

&emsp;&emsp;`CacheBoundary` → `ToolResultClear` → `Mem0Write` → `TaskState` → `SkillsRouter`

&emsp;&emsp;叠加顺序的核心原则是：**修改 messages 的 middleware 按 cache 友好性排列，修改 tools 的 middleware 放最内层**。`CacheBoundary` 必须最外层，因为它的职责是保护 cache 前缀字节——任何内层 middleware 修改 messages 的行为都在它的"保护罩"之内。`SkillsRouter` 作为唯一修改 tools 子集的 middleware，必须放最内层，与修改 messages 的其他 middleware 彻底解耦，避免 tools 裁剪逻辑与 messages 修改逻辑的相互干扰。

#### 6.4.3 装配代码

&emsp;&emsp;5 个 middleware 的叠加顺序是本案例最重要的工程决策，直接影响 cache 命中率和 messages 的清洁程度。装配代码之前，先独立说明顺序规则，让后续代码注释有理论依托。

&emsp;&emsp;**叠加顺序工程规则：修改 messages 的按 cache 友好性排，修改 tools 的放最内层**

&emsp;&emsp;`middleware` 列表的执行顺序是**从外到内**（列表索引 0 最先执行），因此 `create_agent` 的 `middleware` 参数传入顺序直接决定了哪个 middleware 最先拦截请求。本案例 5 middleware 的排列遵循以下两条规则：

&emsp;&emsp;**规则一：修改 messages 的 middleware，按 cache 友好性从外到内排列。** `CacheBoundary` 负责锚定 cache 分界，必须最外层以保护 cache 前缀不被内层修改破坏。`ToolResultClear` 和 `Mem0Write` 都修改 messages，但 `ToolResultClear` 的修改幅度更大（删除历史 tool result），应在 `Mem0Write` 之前处理，确保记忆注入时看到的是已清理后的 messages。`TaskState` 在 before_model 阶段读取 todo 状态注入上下文，after_model 阶段写回，放在 `Mem0Write` 之后避免记忆注入与 todo 状态注入的顺序冲突。

&emsp;&emsp;**规则二：修改 tools 子集的 middleware，放最内层。** `SkillsRouter` 是唯一修改 `tools` 参数的 middleware，它在 before_model 阶段根据意图识别结果动态裁剪工具列表。放最内层意味着它在所有 messages 修改完成之后才执行工具裁剪，确保路由决策基于的是最终清理后的 messages，而不是中间状态。

In [78]:
# 案例 C 研究型 Agent 装配（集大成：5 middleware + 3 tool）
# 本案例是三案例中跨章最广的集大成，对 cache 保护要求最严

# Ch5 层：cache 锚点保护（最外层 — 本案例 middleware 最多，保护最关键）
caseC_cache_mw = DeepSeekCacheBoundaryMiddleware()

# Ch1 层：清理旧 subagent 摘要（多次调 research_tool 时避免累积）
caseC_tool_clear_mw = ToolResultClearMiddleware(llm=llm, keep_recent_tool_results=2)

# Ch2 层：跨会话记忆（记录研究主题/已读论文，避免重复研究）
caseC_mem0_mw = Mem0WriteMiddleware(memory=memory, user_id="ch6_caseC")

# Ch2 层：任务清单（管理待读论文列表）
caseC_task_mw = TaskStateMiddleware(todo_path="./ch6_caseC_todo.md")

# Ch3 层：多 skill 路由（最内层，裁剪到 research 或 answer_knowledge 子集）
caseC_router_mw = SkillsRouterMiddleware(select_manager=_sm)

# 装配主 Agent：3 工具 + 5 middleware
caseC_agent = create_agent(
    model=llm,
    tools=[search_memory, search_knowledge, research_tool],  # research_tool 来自 Ch4
    middleware=[
        caseC_cache_mw,        # 最外层：cache 锚点保护
        caseC_tool_clear_mw,   # 第二层：清理旧摘要占位
        caseC_mem0_mw,         # 第三层：跨会话记忆写入
        caseC_task_mw,         # 第四层：任务清单管理
        caseC_router_mw,       # 最内层：tools 子集路由
    ],
    system_prompt=(
        "你是学术研究助手。遇到综述类任务时调用 research_subagent 委派资料吸收；"
        "需要追踪任务时创建 todo.md；一句话/100 字以内精简回复。"
    ),
)
print("案例 C 装配完成：5 middleware + 3 tool（含 Ch4 research_subagent）")

案例 C 装配完成：5 middleware + 3 tool（含 Ch4 research_subagent）


&emsp;&emsp;装配完成后，`caseC_agent` 的 middleware 执行链如下：请求进入时依次经过 `CacheBoundary`（锁定前缀）→ `ToolResultClear`（清理旧摘要）→ `Mem0Write`（注入记忆 + 写回）→ `TaskState`（读写 todo 状态）→ `SkillsRouter`（裁剪 tools）→ LLM 调用；响应返回时顺序反转，从 `SkillsRouter` 到 `CacheBoundary` 依次后处理。`user_id="ch6_caseC"` 和 `todo_path="./ch6_caseC_todo.md"` 保证与其他案例的 Mem0 数据和 todo 文件隔离，互不干扰。

#### 6.4.4 对照实验：subagent 隔离 + 任务清单追踪

&emsp;&emsp;本节通过单条综述 query 验证五个核心收益：路由决策（Ch3）、subagent 摘要隔离（Ch4）、任务清单生成（Ch2 TaskState）、跨会话记忆写入（Ch2 Mem0）、以及 ToolResultClear 的占位清理效果。

&emsp;&emsp;预期结果说明：路由决策可能命中 research skill 或走 fallback（Ch4 已验证，两条路径 Agent 都能正确作答）；ToolMessage 长度应 ≤ 2000（Ch4 isolation 核心收益——论文原文不进主 Agent）；todo.md 生成与否取决于 Agent 决策，两个分支都接受；Mem0 写入至少 1 条 fact。

In [80]:
# 对照实验：单条综述 query，同时触发 4 种业务路径
caseC_query = (
    "请综述最近中文 Agent 记忆方向的 3 个核心贡献（一句话一条，不超过 100 字），"
    "我决定把需要后续深读的 2 篇论文创建 todo.md 待办清单。"
)
res = caseC_agent.invoke({"messages": [{"role": "user", "content": caseC_query}]})

# 1. 读路由决策（Ch3 SkillsRouter 核心指标）
decision = caseC_router_mw._last_decision
print(f"[路由] skill={decision['skill_id'] or 'fallback'} | 工具数={decision['tool_count']}")

# 2. 最终答复（基于 subagent 摘要或直接作答）
reply = res["messages"][-1].content
print(f"\n[答复 前 200 字]")
print(str(reply)[:200])

# 3. 验证 Ch4 Isolate 核心收益：messages 中 ToolMessage 不含论文原文
from langchain_core.messages import ToolMessage
tool_msgs = [m for m in res["messages"] if isinstance(m, ToolMessage)]
for tm in tool_msgs:
    content = str(getattr(tm, 'content', ''))
    print(f"\n[ToolMessage] 长度={len(content)} | 前 80 字: {content[:80]}")
    assert len(content) < 2000, "ToolMessage 超长，Ch4 Isolate 可能失效"

# 4. 验证 Ch2 TaskState：todo.md 是否生成
import os
if os.path.exists("./ch6_caseC_todo.md"):
    print(f"\n[TaskState] todo.md 已生成:")
    print(open("./ch6_caseC_todo.md").read()[:300])
else:
    print("\n[TaskState] todo.md 未生成（Agent 未主动调 TaskState 工具，接受此分支）")

# 5. 验证 Ch2 Mem0：研究主题已写入
hits = memory.search(query="中文 Agent 记忆研究", user_id="ch6_caseC", limit=3)
print(f"\n[Mem0] 写入 fact 数: {len(hits.get('results', []))}")

[Layer3] 写入 1 条任务到 ch6_caseC_todo.md
[Layer2] 决策写入成功 | user_id=ch6_caseC | 触发词：['决定']
[Layer3] 写入 3 条任务到 ch6_caseC_todo.md
[路由] skill=research | 工具数=1

[答复 前 200 字]
基于研究结果，我为您综述最近中文Agent记忆方向的3个核心贡献：

1. **分层记忆系统架构**：提出三层（短期/工作/长期）记忆架构，有效解决长会话中的事实漂移问题，在中文Agent中实现23%的召回率提升。

2. **多路融合检索策略**：针对中文Agent特点，融合BM25关键词、语义向量、规则匹配与时间衰减四路检索，相比单一路径F1值提升18%。

3. **长期记忆压缩机制**：基

[ToolMessage] 长度=500 | 前 80 字: **中文Agent记忆方向核心进展（2023-2024）结构化摘要**

**一、 记忆架构与系统设计**
1.  **分层记忆系统**：为解决长会话中的事实漂

[TaskState] todo.md 已生成:
- [ ] 我来为您综述最近中文 Agent 记忆方向的3个核心贡献，并创建待办清单。 (2026-04-20)
- [ ] **分层记忆系统架构**：提出短期/工作/长期三层记忆架构，有效缓解长会话中的事实漂移问题，实验显示召回率提升23%。 (2026-04-20)
- [ ] **多路融合检索机制**：针对中文Agent，创新融合BM25词法、语义向量、规则匹配与时间衰减四路信号进行记忆检索，F1值较单路方法提升18%。 (2026-04-20)
- [ ] **跨会话持久化工程**：基于双层存储设计跨会话记忆方案，配套检查点与失败回滚机制，提升系统鲁棒性并降低存储成本60%。 (202

[Mem0] 写入 fact 数: 1


&emsp;&emsp;**结果解读与三案例差异化收益对比**

&emsp;&emsp;对照实验的五个验证点，分别对应五个 middleware 的核心收益。其中最值得关注的是**ToolMessage 长度检测**：断言 `len(content) < 2000` 通过，说明 Ch4 的 `research_tool` 通过 subagent 隔离机制，将论文原文（可能 4000+ tokens）压缩成摘要（≤ 500 字）后返回，主 Agent 的 messages 中从未出现论文原文。这是三案例里**唯一**启用 Ch4 isolation 的案例，正是研究型 Agent 的核心护城河。

&emsp;&emsp;todo.md 的生成验证了 Ch2 TaskState 在"多步任务追踪"场景的实际价值——用户不需要手动管理待读列表，Agent 在完成综述的同时自动创建结构化文件。即使 Agent 未主动创建（fallback 分支），也说明任务清单功能是"可选收益"而非硬依赖，案例的主干逻辑（综述生成）不受影响。

&emsp;&emsp;Mem0 写入验证了跨会话记忆的持久化：下次用户提出"继续研究中文 Agent 方向"时，`Mem0WriteMiddleware` 会在 before_model 阶段注入"用户曾研究过中文 Agent 记忆方向"的 fact，避免重复检索相同资料。这是本案例区别于案例 A 和案例 B 的**独特长期价值**——不只服务当次请求，而是持续积累研究上下文。

&emsp;&emsp;<font color=red>集大成不等于堆砌</font>：三个案例分别用 6 / 5 / 5 个主力 middleware，案例 C 在**数量上并非最多**（案例 A 有 6 个），但在**跨章覆盖**上最广——前五章每一章都贡献了至少一个主力组件（Ch1 `ToolResultClear` + Ch2 `Mem0Write`/`TaskState` + Ch3 `SkillsRouter` + Ch4 `research_subagent` + Ch5 `CacheBoundary`）。相比之下案例 A 集中在 Ch1/Ch2/Ch3/Ch5（缺 Ch4），案例 B 集中在 Ch1/Ch2/Ch5（缺 Ch3/Ch4）。策略组合的"正确性"来自于对场景的精准理解，而不是 middleware 的数量。

&emsp;&emsp;关于三案例的差异化收益，完整量化对比将在 **6.6 三 Agent 雷达图**中呈现——六个能力维度上，工具 Agent 强在 Tool Accuracy，对话 Agent 强在 Dialogue Continuity，研究 Agent 各项均衡但单次延迟最高。这正是"按 Agent 类型选策略"的可量化证据。

### 6.5 Agent 类型 × 策略 决策矩阵

&emsp;&emsp;三个案例跑完，我们把"哪个 Agent 配哪个 middleware"沉淀为两张决策矩阵。<font color=red>这两张矩阵是本章最核心的可带走产物</font>——以后你自己造新 Agent 时，先判断类型属于 A / B / C 中的哪个，再查表定组合，可以少走大量弯路。

&emsp;&emsp;**矩阵 1：Ch1 压缩五件套 × 三 Agent**——Ch1 五件套不是铁板一块，五个 middleware 各自适用场景差异很大。这张子矩阵告诉你"Ch1 里面该选哪个"。

&emsp;&emsp;下表使用符号约定：**主力**（场景首选）/ **可选**（增强但非必需）/ **不适用**（该场景下禁用或不触发）。后续决策矩阵沿用同一约定。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Ch1 五件套 × 三 Agent 选配指南</font></p>
<div class="center">

| Ch1 middleware | A 工具调用 | B 对话问答 | C 研究集大成 | 核心判断逻辑 |
|---|---|---|---|---|
| `ToolResultClearMiddleware` | 主力 | 不触发 | 主力 | 工具结果多就配 |
| `SummarizationMiddleware` | 破坏 cache | 主力 | 破坏摘要 | 长对话专属 |
| `MessageTrimMiddleware` | 兜底 | 可选 | 切断摘要 | 硬截断保底 |
| `ObservationMaskMiddleware` | 遮蔽代码 | 对话型禁用 | 遮蔽摘要 | **三类都不用** |
| `CompactionMiddleware` | 过激 | 过激 | 过激 | **三类都不用** |

</div>

&emsp;&emsp;<font color=red>重要教学点</font>：`ObservationMask` 和 `Compaction` 在本章三个案例中**都不推荐**——这不是疏漏，而是深思熟虑的结论。ObservationMask 适合 JetBrains SE Agent 这种"工具结果是中间步骤"的窄场景，Compaction 粒度太粗（Summarization + TailTrim 组合已充分覆盖）。<font color=red>Ch1 教五件套是为了让你会选，不是让你全用</font>。

&emsp;&emsp;**矩阵 2：完整 3 × 5 策略大类矩阵**——把所有前五章组件映射到三个 Agent 上，一眼看清差异化组合。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Agent 类型 × 策略大类 决策矩阵</font></p>
<div class="center">

| Agent 类型 / 策略大类 | Ch1 Compression | Ch2 Write | Ch3 Select | Ch4 Isolate | Ch5 Cache |
|---|---|---|---|---|---|
| **A. 工具调用 Agent**<br>（代码调试 / 数据分析）| `ToolResultClear`<br>`MessageTrim`（兜底）| `Mem0Write`<br>`TaskState` | `SkillsRouter` | 不适用 | `CacheBoundary` |
| **B. 对话问答 Agent**<br>（技术顾问 / 客服）| `Summarization`<br>`MessageTrim`（可选） | `Mem0Write`<br>`TaskState` | 不适用 | 不适用 | `CacheBoundary`<br>`TailTrim` |
| **C. 研究型 Agent**<br>（综述 / 深度研究）| `ToolResultClear` | `Mem0Write`<br>`TaskState` | `SkillsRouter` | `research_subagent` | `CacheBoundary` |
| **主力组件数** | 2 / 1 / 1 | 2 / 2 / 2 | 1 / 0 / 1 | 0 / 0 / 1 | 1 / 2 / 1 |

</div>

&emsp;&emsp;<font color=red>三生产铁律</font>（任何类型 Agent 都必配）：①Ch2 长期记忆（Mem0Write 跨会话持久化用户上下文）；②Ch2 任务记忆（TaskState 结构化状态追踪）；③Ch5 Cache 锚点（生产环境成本敏感必配）。**短期记忆管理**（Ch1 的某个件套）按对话轮次和工具结果体量按需选择。

&emsp;&emsp;**六步决策树**——拿到一个新 Agent 需求时的决策顺序：

1. **工具结果体量大（单条 > 500 字）且多轮累积？** → 配 **Ch1 `ToolResultClearMiddleware`**
2. **对话轮次 > 5 轮？** → 配 **Ch1 `SummarizationMiddleware`**（阈值 1500-3000 tokens）
3. **工具数量 > 3 个 或 有多业务意图？** → 配 **Ch3 `SkillsRouterMiddleware`**
4. **单轮资料吞吐 > 1K tokens？** → 配 **Ch4 `research_subagent`**
5. **跨会话记忆需求？** → 配 **Ch2 `Mem0WriteMiddleware`**（建议标配）
6. **需要管理任务清单？** → 配 **Ch2 `TaskStateMiddleware`**（建议标配）

&emsp;&emsp;**四个反模式**（任何类型都不推荐）：
- **禁用**：`ObservationMaskMiddleware`：除非确信 ToolMessage 是纯中间步骤，否则会导致 Agent 幻觉
- **禁用**：`CompactionMiddleware`：粒度太粗，优先选 Summarization + TailTrim 组合
- **反模式**：策略堆砌：9 层满栈是反模式（原 Ch6 的 deepagents ImportError fallback 就是活例）
- **反模式**：cache 后置：任何修改 messages 的 middleware 放在 CacheBoundary 之后都会破坏 cache 锚点（Ch5 反共识）

### 6.6 本章小结与三大盲点

&emsp;&emsp;本章三件事：**装配代码**（三案例从 6 middleware 到 5 middleware 的真跑 demo）、**决策矩阵**（Ch1 子矩阵 + 3×5 大类矩阵 + 六步决策树）、**雷达对比**（三 Agent 差异化能力画像）。关键结论是反直觉的——<font color=red>"好的 Agent" 不是 middleware 堆得多的 Agent，是 middleware 组合贴合业务场景的 Agent</font>。

&emsp;&emsp;**三大盲点**（诚实声明，生产上线前必读）：**盲点 1（meta-evaluation 缺失）**——本章雷达图的 RAGAS faithfulness 等分数由 LLM-judge 给出，judge 本身存在 self-serving bias 和长文本偏好，绝对值不应作为通过/不通过的硬标准，只能作为 A/B 对比的趋势指标。**盲点 2（长期影响未追踪）**——本章所有指标都是"瞬态"的，真实 Agent 随着 Milvus 记忆累积、DeepSeek 定价变化、对话长度分布漂移，6-12 个月后评估结果可能截然不同，生产环境应持续监控。**盲点 3（场景外推限制）**——本章三个案例覆盖不了所有 Agent 形态（如纯 RAG Agent、多 Agent 编排系统、SE Agent），换场景必须重新选型重跑评估，不要拿本章组合直接套用。

&emsp;&emsp;至此，前六章完整闭环：**Ch1-Ch5 教工具，Ch6 教选择**。并且我们将把这些工具反哺到 mini-OpenClaw 生产级项目，完成从演示环境到生产项目的落地闭环。